# nb_mdf_00_setup_framework - create schemas and load the control tables

Run once per workspace (and again whenever you change the configuration).

**Prerequisite:** a lakehouse created with **"Lakehouse schemas" enabled**, attached as this notebook's default lakehouse.

What it does
1. Creates the schemas `meta`, `bronze`, `silver`, `gold`, `quarantine`.
2. Loads the chosen **industry pack** into the control tables. Packs: `energy`, `telco`, `fsi`, `manufacturing`,
   `healthcare`, `ites`, `government`, `retail`, `digital`.

| Control table | Purpose |
|---|---|
| `meta.source_system` | The 3 source systems + which Fabric ingestion options apply (copy / mirror / shortcut / stream) |
| `meta.ingestion_object` | One row per source object: format, load type, keys, watermark, target tables, write mode, owner, SLA |
| `meta.column_map` | Source->Silver rename, type, derivation, **PII treatment** and sensitivity per column |
| `meta.dq_rule` | Technical + business data-quality rules with severity and owner |
| `meta.gold_transform` | Gold tables as SQL, with run order and business owner |
| `meta.setting` | Pack name, PII salt, the pack definition and the "onboard a new source" definition |
| `meta.watermark`, `meta.run_log`, `meta.dq_result` | Operational state, created empty |

Config source: `Files/config/<pack>.json` if you uploaded it, otherwise the copy embedded at the bottom of this notebook.


In [ ]:
industry = "energy"          # energy | telco | fsi | manufacturing | healthcare | ites | government | retail | digital
reset_all = False            # True = DROP every framework table (bronze/silver/gold/quarantine/meta) and start clean
pii_salt = "demo-salt-change-me"   # demo only - in production keep this in Key Vault


In [ ]:
import json, os
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, BooleanType, DoubleType

SCHEMAS = ["meta", "bronze", "silver", "gold", "quarantine"]

try:
    spark.sql("CREATE SCHEMA IF NOT EXISTS meta")
except Exception as e:
    raise Exception("Could not create schemas. Attach a lakehouse that was created with 'Lakehouse schemas' enabled.") from e

if reset_all:
    for s in SCHEMAS:
        try:
            for t in spark.sql(f"SHOW TABLES IN {s}").collect():
                spark.sql(f"DROP TABLE IF EXISTS {s}.{t['tableName']}")
                print("dropped", f"{s}.{t['tableName']}")
        except Exception as e:
            print(f"skip {s}: {e}")

for s in SCHEMAS:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {s}")
print("schemas ready:", SCHEMAS)

---
### Embedded configuration (generated from `config/*.json` by `tools/build_kit.py` - edit the JSON, not this cell)

In [ ]:
EMBEDDED_CONFIG = json.loads(r'''{"energy":{"pack":{"id":"energy","industry":"Energy & utilities","org":"Southern Grid Transmission","short":"SGT","prefix":"EN","tagline":"electricity transmission network operator in regional NSW","email_domain":"sgt.example.com.au","region_col":"region","region_label":"region","regions":["Riverina","Central West","New England","Hunter","Southern Tablelands","Far West","Sydney Metro"],"name_words":["Wagga","Orange","Armidale","Muswellbrook","Goulburn","Broken Hill","Marulan","Dubbo","Tamworth","Yass","Lismore","Parkes"],"sys":{"ops":{"code":"eam","name":"Enterprise Asset Management (on-prem SQL Server)","type":"Database","owner":"Asset systems","fabric":"Mirroring for SQL Server (near real-time, no ETL) vs Copy job incremental vs pipeline Copy"},"tel":{"code":"scada","name":"SCADA historian exports (15-min CSV) and BoM weather feed","type":"OT file drop","owner":"Network Operations","fabric":"Eventstream if the historian can publish; otherwise file drop to OneLake + shortcut"},"crm":{"code":"sfdc","name":"Salesforce (landholder engagement)","type":"SaaS CRM","owner":"Stakeholder Engagement","fabric":"Copy job with the Salesforce connector, or Dataverse/Salesforce mirroring where available"}},"entity":{"table":"asset","src":"assets","label":"asset","plural":"assets","id":"asset_id","code":"asset_tag","code_prefix":"AST","name_col":"asset_name","type_col":"asset_class","types":["SUBSTATION","TRANSFORMER","CIRCUIT_BREAKER","CAPACITOR_BANK","TRANSMISSION_LINE"],"status":["IN_SERVICE","STANDBY","OUT_OF_SERVICE","DECOMMISSIONED"],"tier_col":"criticality","tier_label":"criticality (1-5, 5 = most critical)","cap_col":"rating_mva","cap_label":"rating (MVA)","cap_range":[10,900],"start_col":"install_date","start_years":[1968,2024],"drift_col":"asset_owner_group","drift_values":["Northern Field Ops","Southern Field Ops","Major Projects"],"owner":"Asset Management","count":240},"event":{"table":"work_order","src":"work_orders","label":"work order","plural":"work orders","id":"wo_id","id_start":500001,"type_col":"wo_type","types":["PM","CM","INSP","EMERG"],"type_names":["Preventive maintenance","Corrective maintenance","Inspection","Emergency"],"urgent_type":"EMERG","status":["OPEN","IN_PROGRESS","COMPLETE","CANCELLED"],"open_ts":"raised_ts","close_ts":"completed_ts","open_verb":"raised","close_verb":"completed","amount_col":"cost_aud","amount_range":[300,85000],"qty_col":"labour_hours","qty_range":[1,120],"descriptions":["Thermal scan found hot joint","Oil sample analysis","Replace bushing","Protection relay test","Vegetation encroachment near span","Corona discharge inspection","Storm damage repair","Battery bank replacement"],"owner":"Asset Management"},"reading":{"table":"tx_telemetry","src":"transformer_telemetry","label":"transformer telemetry","fact":"tx_load","monitored":24,"monitored_types":["TRANSFORMER"],"metrics":[{"col":"oil_temp_c","label":"oil temperature","unit":"°C","base":62,"sd":6,"min":-10,"max":140,"threshold":95},{"col":"load_mw","label":"load","unit":"MW","base":120,"sd":35,"min":0,"max":1000},{"col":"ambient_temp_c","label":"ambient temperature","unit":"°C","base":24,"sd":6,"min":-10,"max":55}],"sentinel":999.9,"owner":"Network Operations"},"party":{"table":"landholder","src":"landholder_contacts","label":"landholder","plural":"landholders","id":"landholder_id","id_prefix":"LH","sf_type":"Contact","segment_col":"landholder_type","segments":["Farm","Rural residential","Commercial","Crown land"],"status":["Active","Inactive"],"sensitive_col":"property_lot_dp","sensitive_label":"property lot/DP reference","sensitive_treatment":"hash","owner":"Stakeholder Engagement","count":300},"extra":{"table":"weather_daily","src":"weather_observations","label":"BoM weather observations","sys":"tel","cols":[{"col":"max_temp_c","base":29,"sd":6,"min":-10,"max":50},{"col":"max_wind_gust_kmh","base":45,"sd":20,"min":0,"max":200},{"col":"rainfall_mm","base":2,"sd":6,"min":0,"max":300}],"cat_col":"fire_danger_rating","cat_values":["Moderate","High","Extreme","Catastrophic"],"owner":"Network Operations","story":"fire danger and wind gusts against emergency work orders by region"},"alert":{"stream":"breaker_trip","label":"breaker trip","codes":["PROTECTION_TRIP","AUTO_RECLOSE","LOCKOUT"],"storm":"storm cell"},"ref":{"title":"Work order response targets","owner":"Asset Management","rows":[["PM","Planned",720,"Maintenance Planning"],["CM","Reactive",168,"Field Services"],["INSP","Planned",336,"Asset Inspection"],["EMERG","Emergency",8,"Network Operations"]]},"ai":{"persona":"network operations and asset management","cust":"electricity transmission and distribution network operators","q_model":["Which region had the most unplanned work orders this financial year?","Show total work order cost by asset class and month.","Which critical assets (criticality 4 or 5) have open work orders?"],"q_agent":["Which 10 transformers have the highest risk score, and how many open work orders does each have?","Compare average hours to complete emergency work orders by region.","Which transformers breached the 95 °C oil temperature threshold in the last 3 days?","What did we spend on corrective maintenance in the Hunter region?","List landholders by type in New England and how many are active."],"q_onto":["Show me every asset in the Riverina that has an open emergency work order and a high risk band.","For the transformer with the highest oil temperature, what work orders have been raised against it and what did they cost?","Which landholders are linked to assets that are out of service?"]},"impact":{"col":"outage_cost_at_risk_aud","label":"outage cost at risk","open_cost":6000,"breach_cost":2500,"cap_rate":20,"rule":"$6,000 per open work order + $2,500 per 15-min window above the oil-temperature threshold + rating (MVA) x criticality x $20"}},"source_system":[{"source_system":"eam","industry":"energy","display_name":"Enterprise Asset Management (on-prem SQL Server)","system_type":"Database","connection_pattern":"On-premises data gateway -> pipeline Copy with watermark query","technical_owner":"Asset systems","fabric_options":"Mirroring for SQL Server (near real-time, no ETL) vs Copy job incremental vs pipeline Copy"},{"source_system":"scada","industry":"energy","display_name":"SCADA historian exports (15-min CSV) and BoM weather feed","system_type":"OT file drop","connection_pattern":"Files dropped to OneLake (Files/landing) every 15 minutes","technical_owner":"Network Operations","fabric_options":"Eventstream if the historian can publish; otherwise file drop to OneLake + shortcut"},{"source_system":"sfdc","industry":"energy","display_name":"Salesforce (landholder engagement)","system_type":"SaaS CRM","connection_pattern":"REST/OData API -> JSON landed per day","technical_owner":"Stakeholder Engagement","fabric_options":"Copy job with the Salesforce connector, or Dataverse/Salesforce mirroring where available"}],"ingestion_object":[{"object_id":"EN_EAM_ASSETS","industry":"energy","source_system":"eam","source_object":"assets","source_type":"database","ingest_method":"copy_db","landing_format":"csv","landing_path":"landing/eam/assets","read_options":"{\"header\": true, \"delimiter\": \",\"}","source_table":"src.assets","load_type":"incremental","watermark_column":"last_modified","primary_keys":"asset_id","dedupe_order_by":"last_modified DESC, _ingested_at DESC","bronze_table":"eam_assets","silver_table":"asset","silver_write_mode":"merge_scd2","load_group":"reference","priority":10,"sla_hours":26,"data_owner":"Asset Management","sensitivity":"Internal","active":true,"description":"Asset register. SCD2 keeps status, criticality and rating_mva history.","fabric_alternative":"Strong mirroring candidate - small, slowly changing, needed in near real time by operations"},{"object_id":"EN_EAM_WORK_ORDERS","industry":"energy","source_system":"eam","source_object":"work_orders","source_type":"database","ingest_method":"copy_db","landing_format":"csv","landing_path":"landing/eam/work_orders","read_options":"{\"header\": true, \"delimiter\": \",\"}","source_table":"src.work_orders","load_type":"incremental","watermark_column":"last_modified","primary_keys":"wo_id","dedupe_order_by":"last_modified DESC, _ingested_at DESC","bronze_table":"eam_work_orders","silver_table":"work_order","silver_write_mode":"merge_scd1","load_group":"transactional","priority":20,"sla_hours":26,"data_owner":"Asset Management","sensitivity":"Internal","active":true,"description":"Work orders - incremental by last_modified; can arrive before their asset (late-arriving).","fabric_alternative":"Copy job incremental or mirroring; keep the watermark query if the source is on-prem"},{"object_id":"EN_SCADA_TRANSFORMER_TELEMETRY","industry":"energy","source_system":"scada","source_object":"transformer_telemetry","source_type":"file","ingest_method":"landed_file","landing_format":"csv","landing_path":"landing/scada/transformer_telemetry","read_options":"{\"header\": true, \"delimiter\": \",\"}","source_table":null,"load_type":"incremental","watermark_column":null,"primary_keys":"asset_id,reading_ts","dedupe_order_by":"_ingested_at DESC","bronze_table":"scada_transformer_telemetry","silver_table":"tx_telemetry","silver_write_mode":"insert_only","load_group":"telemetry","priority":30,"sla_hours":2,"data_owner":"Network Operations","sensitivity":"Internal","active":true,"description":"Transformer telemetry every 15 minutes, four micro-batch files a day. Insert-only and idempotent.","fabric_alternative":"Eventstream into Eventhouse for real time; keep this batch path for history and reconciliation"},{"object_id":"EN_SFDC_LANDHOLDER_CONTACTS","industry":"energy","source_system":"sfdc","source_object":"landholder_contacts","source_type":"api","ingest_method":"landed_file","landing_format":"json","landing_path":"landing/sfdc/landholder_contacts","read_options":"{\"multiline\": true, \"json_root\": \"records\"}","source_table":null,"load_type":"full","watermark_column":null,"primary_keys":"landholder_id","dedupe_order_by":"last_modified_ts DESC, _ingested_at DESC","bronze_table":"sfdc_landholder_contacts","silver_table":"landholder","silver_write_mode":"merge_scd1","load_group":"reference","priority":15,"sla_hours":26,"data_owner":"Stakeholder Engagement","sensitivity":"Confidential","active":true,"description":"Landholders - full daily snapshot from the CRM API (contains PII). Missing rows are soft-deleted.","fabric_alternative":"Copy job with the CRM connector; Dataverse-based CRMs can use Link to Fabric instead of copying"}],"column_map":[{"object_id":"EN_EAM_ASSETS","column_order":0,"source_column":"asset_id","target_column":"asset_id","data_type":"int","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"EN_EAM_ASSETS","column_order":1,"source_column":"asset_tag","target_column":"asset_tag","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"EN_EAM_ASSETS","column_order":2,"source_column":"asset_name","target_column":"asset_name","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"EN_EAM_ASSETS","column_order":3,"source_column":"asset_class","target_column":"asset_class","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"EN_EAM_ASSETS","column_order":4,"source_column":"region","target_column":"region","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"EN_EAM_ASSETS","column_order":5,"source_column":"status","target_column":"status","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"EN_EAM_ASSETS","column_order":6,"source_column":"criticality","target_column":"criticality","data_type":"int","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"EN_EAM_ASSETS","column_order":7,"source_column":"rating_mva","target_column":"rating_mva","data_type":"double","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"EN_EAM_ASSETS","column_order":8,"source_column":"install_date","target_column":"install_date","data_type":"date","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"EN_EAM_ASSETS","column_order":9,"source_column":"last_modified","target_column":"last_modified","data_type":"timestamp","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"EN_EAM_WORK_ORDERS","column_order":0,"source_column":"wo_id","target_column":"wo_id","data_type":"int","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"EN_EAM_WORK_ORDERS","column_order":1,"source_column":"asset_id","target_column":"asset_id","data_type":"int","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"EN_EAM_WORK_ORDERS","column_order":2,"source_column":"wo_type","target_column":"wo_type","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"EN_EAM_WORK_ORDERS","column_order":3,"source_column":"priority","target_column":"priority","data_type":"int","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"EN_EAM_WORK_ORDERS","column_order":4,"source_column":"status","target_column":"status","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"EN_EAM_WORK_ORDERS","column_order":5,"source_column":"raised_ts","target_column":"raised_ts","data_type":"timestamp","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"EN_EAM_WORK_ORDERS","column_order":6,"source_column":"completed_ts","target_column":"completed_ts","data_type":"timestamp","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"EN_EAM_WORK_ORDERS","column_order":7,"source_column":"cost_aud","target_column":"cost_aud","data_type":"decimal(12,2)","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"EN_EAM_WORK_ORDERS","column_order":8,"source_column":"labour_hours","target_column":"labour_hours","data_type":"double","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"EN_EAM_WORK_ORDERS","column_order":9,"source_column":"description","target_column":"description","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"EN_EAM_WORK_ORDERS","column_order":10,"source_column":"last_modified","target_column":"last_modified","data_type":"timestamp","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"EN_SCADA_TRANSFORMER_TELEMETRY","column_order":0,"source_column":"asset_id","target_column":"asset_id","data_type":"int","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"EN_SCADA_TRANSFORMER_TELEMETRY","column_order":1,"source_column":"reading_time","target_column":"reading_ts","data_type":"timestamp","transform_expr":"to_timestamp(reading_time, 'dd/MM/yyyy HH:mm')","pii_treatment":"none","sensitivity":"none","description":""},{"object_id":"EN_SCADA_TRANSFORMER_TELEMETRY","column_order":2,"source_column":"oil_temp_c","target_column":"oil_temp_c","data_type":"double","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"EN_SCADA_TRANSFORMER_TELEMETRY","column_order":3,"source_column":"load_mw","target_column":"load_mw","data_type":"double","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"EN_SCADA_TRANSFORMER_TELEMETRY","column_order":4,"source_column":"ambient_temp_c","target_column":"ambient_temp_c","data_type":"double","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"EN_SFDC_LANDHOLDER_CONTACTS","column_order":0,"source_column":"Id","target_column":"landholder_id","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"EN_SFDC_LANDHOLDER_CONTACTS","column_order":1,"source_column":"FirstName","target_column":"first_name","data_type":"string","transform_expr":null,"pii_treatment":"drop","sensitivity":"Confidential","description":""},{"object_id":"EN_SFDC_LANDHOLDER_CONTACTS","column_order":2,"source_column":"LastName","target_column":"last_name","data_type":"string","transform_expr":null,"pii_treatment":"drop","sensitivity":"Confidential","description":""},{"object_id":"EN_SFDC_LANDHOLDER_CONTACTS","column_order":3,"source_column":"Email","target_column":"email_masked","data_type":"string","transform_expr":null,"pii_treatment":"mask_email","sensitivity":"Confidential","description":""},{"object_id":"EN_SFDC_LANDHOLDER_CONTACTS","column_order":4,"source_column":"MobilePhone","target_column":"mobile_masked","data_type":"string","transform_expr":null,"pii_treatment":"mask_phone","sensitivity":"Confidential","description":""},{"object_id":"EN_SFDC_LANDHOLDER_CONTACTS","column_order":5,"source_column":"Birthdate","target_column":"birth_year","data_type":"int","transform_expr":null,"pii_treatment":"year_only","sensitivity":"Confidential","description":""},{"object_id":"EN_SFDC_LANDHOLDER_CONTACTS","column_order":6,"source_column":"MailingPostalCode","target_column":"postcode","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"EN_SFDC_LANDHOLDER_CONTACTS","column_order":7,"source_column":"Region__c","target_column":"region","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"EN_SFDC_LANDHOLDER_CONTACTS","column_order":8,"source_column":"LandholderType__c","target_column":"landholder_type","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"EN_SFDC_LANDHOLDER_CONTACTS","column_order":9,"source_column":"AssetId__c","target_column":"asset_id","data_type":"int","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"EN_SFDC_LANDHOLDER_CONTACTS","column_order":10,"source_column":"Status__c","target_column":"status","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"EN_SFDC_LANDHOLDER_CONTACTS","column_order":11,"source_column":"PropertyLotDp__c","target_column":"property_lot_dp_hash","data_type":"string","transform_expr":null,"pii_treatment":"hash","sensitivity":"Restricted","description":""},{"object_id":"EN_SFDC_LANDHOLDER_CONTACTS","column_order":12,"source_column":"LastModifiedDate","target_column":"last_modified_ts","data_type":"timestamp","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""}],"dq_rule":[{"rule_id":"EN-ENT-01","object_id":"EN_EAM_ASSETS","rule_name":"Asset id present","rule_category":"technical","check_type":"not_null","column_name":"asset_id","params":"{}","severity":"fail","fail_threshold_pct":0,"rule_owner":"Data engineering","description":"Primary key must never be null","active":true},{"rule_id":"EN-ENT-02","object_id":"EN_EAM_ASSETS","rule_name":"Known asset_class","rule_category":"technical","check_type":"allowed_values","column_name":"asset_class","params":"{\"values\": [\"SUBSTATION\", \"TRANSFORMER\", \"CIRCUIT_BREAKER\", \"CAPACITOR_BANK\", \"TRANSMISSION_LINE\"]}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Data engineering","description":"Unknown classes are kept but flagged for the data owner","active":true},{"rule_id":"EN-ENT-03","object_id":"EN_EAM_ASSETS","rule_name":"criticality between 1 and 5","rule_category":"technical","check_type":"range","column_name":"criticality","params":"{\"min\": 1, \"max\": 5}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Data engineering","description":"Out-of-range ratings usually mean a keying error in the source","active":true},{"rule_id":"EN-ENT-04","object_id":"EN_EAM_ASSETS","rule_name":"Known status","rule_category":"business","check_type":"allowed_values","column_name":"status","params":"{\"values\": [\"IN_SERVICE\", \"STANDBY\", \"OUT_OF_SERVICE\", \"DECOMMISSIONED\"]}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Asset Management","description":"Status list agreed with Asset Management","active":true},{"rule_id":"EN-EVT-01","object_id":"EN_EAM_WORK_ORDERS","rule_name":"Work order id present","rule_category":"technical","check_type":"not_null","column_name":"wo_id","params":"{}","severity":"fail","fail_threshold_pct":0,"rule_owner":"Data engineering","description":"Primary key must never be null","active":true},{"rule_id":"EN-EVT-02","object_id":"EN_EAM_WORK_ORDERS","rule_name":"Asset exists","rule_category":"technical","check_type":"referential","column_name":"asset_id","params":"{\"ref_table\": \"silver.asset\", \"ref_column\": \"asset_id\"}","severity":"quarantine","fail_threshold_pct":null,"rule_owner":"Data engineering","description":"Held until the asset arrives (late-arriving reference data), then merged automatically","active":true},{"rule_id":"EN-EVT-03","object_id":"EN_EAM_WORK_ORDERS","rule_name":"Completed after raised","rule_category":"business","check_type":"expression","column_name":null,"params":"{\"expr\": \"completed_ts IS NULL OR completed_ts >= raised_ts\"}","severity":"quarantine","fail_threshold_pct":null,"rule_owner":"Asset Management","description":"A work order can't be completed before it was raised","active":true},{"rule_id":"EN-EVT-04","object_id":"EN_EAM_WORK_ORDERS","rule_name":"cost_aud not negative","rule_category":"technical","check_type":"range","column_name":"cost_aud","params":"{\"min\": 0}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Data engineering","description":"Negative values are sign errors in the source extract","active":true},{"rule_id":"EN-EVT-05","object_id":"EN_EAM_WORK_ORDERS","rule_name":"EMERG is priority 1","rule_category":"business","check_type":"expression","column_name":null,"params":"{\"expr\": \"wo_type <> 'EMERG' OR priority = 1\"}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Asset Management","description":"Emergency work orders must be raised as priority 1","active":true},{"rule_id":"EN-EVT-06","object_id":"EN_EAM_WORK_ORDERS","rule_name":"raised_ts parses","rule_category":"technical","check_type":"not_null","column_name":"raised_ts","params":"{}","severity":"fail","fail_threshold_pct":5,"rule_owner":"Data engineering","description":"If more than 5% of timestamps fail to parse, the extract format has changed - stop the load","active":true},{"rule_id":"EN-RDG-01","object_id":"EN_SCADA_TRANSFORMER_TELEMETRY","rule_name":"Oil temperature plausible","rule_category":"technical","check_type":"range","column_name":"oil_temp_c","params":"{\"min\": -10, \"max\": 140}","severity":"quarantine","fail_threshold_pct":null,"rule_owner":"Data engineering","description":"Sensor fault values (e.g. 999.9) are quarantined, never averaged into Gold","active":true},{"rule_id":"EN-RDG-02","object_id":"EN_SCADA_TRANSFORMER_TELEMETRY","rule_name":"Asset known","rule_category":"technical","check_type":"referential","column_name":"asset_id","params":"{\"ref_table\": \"silver.asset\", \"ref_column\": \"asset_id\"}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Data engineering","description":"Readings for unknown tags are flagged","active":true},{"rule_id":"EN-RDG-03","object_id":"EN_SCADA_TRANSFORMER_TELEMETRY","rule_name":"Reading time parses","rule_category":"technical","check_type":"not_null","column_name":"reading_ts","params":"{}","severity":"fail","fail_threshold_pct":5,"rule_owner":"Data engineering","description":"Timestamp format change stops the load","active":true},{"rule_id":"EN-PTY-01","object_id":"EN_SFDC_LANDHOLDER_CONTACTS","rule_name":"Landholder id present","rule_category":"technical","check_type":"not_null","column_name":"landholder_id","params":"{}","severity":"fail","fail_threshold_pct":0,"rule_owner":"Data engineering","description":"Primary key must never be null","active":true},{"rule_id":"EN-PTY-02","object_id":"EN_SFDC_LANDHOLDER_CONTACTS","rule_name":"Email looks valid","rule_category":"technical","check_type":"regex","column_name":"email_masked","params":"{\"pattern\": \"^[^@\\\\s]+@[^@\\\\s]+\\\\.[^@\\\\s]+$\"}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Data engineering","description":"Checked on the raw value before masking","active":true},{"rule_id":"EN-PTY-03","object_id":"EN_SFDC_LANDHOLDER_CONTACTS","rule_name":"Australian postcode","rule_category":"technical","check_type":"regex","column_name":"postcode","params":"{\"pattern\": \"^\\\\d{4}$\"}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Data engineering","description":"Four digits","active":true},{"rule_id":"EN-PTY-04","object_id":"EN_SFDC_LANDHOLDER_CONTACTS","rule_name":"Known landholder_type","rule_category":"business","check_type":"allowed_values","column_name":"landholder_type","params":"{\"values\": [\"Farm\", \"Rural residential\", \"Commercial\", \"Crown land\"]}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Stakeholder Engagement","description":"Segments agreed with Stakeholder Engagement","active":true}],"gold_transform":[{"transform_id":"EN-G01","industry":"energy","gold_table":"dim_date","run_order":100,"sql_text":"SELECT CAST(date_format(d, 'yyyyMMdd') AS INT) AS date_key, d AS calendar_date, year(d) AS calendar_year, quarter(d) AS calendar_quarter, month(d) AS month_number, date_format(d, 'MMM') AS month_name, date_format(d, 'yyyy-MM') AS year_month, dayofmonth(d) AS day_of_month, date_format(d, 'EEE') AS day_name, CASE WHEN month(d) >= 7 THEN year(d) + 1 ELSE year(d) END AS fy_year, concat('FY', substring(CAST(CASE WHEN month(d) >= 7 THEN year(d) + 1 ELSE year(d) END AS STRING), 3, 2)) AS fy_label, CASE WHEN dayofweek(d) IN (1, 7) THEN true ELSE false END AS is_weekend FROM (SELECT explode(sequence(to_date('2020-01-01'), to_date('2028-12-31'), interval 1 day)) AS d) dates","description":"Calendar with Australian financial year","business_owner":"Data engineering","consumers":"Power BI / Direct Lake, data agent","active":true},{"transform_id":"EN-G02","industry":"energy","gold_table":"dim_asset","run_order":110,"sql_text":"SELECT asset_id, asset_tag, asset_name, asset_class, region, status, criticality, rating_mva, install_date, CAST(floor(months_between(current_date(), install_date) / 12) AS INT) AS age_years, _valid_from AS version_valid_from FROM silver.asset WHERE _is_current = true","description":"Current view of the asset register (history stays in Silver SCD2)","business_owner":"Asset Management","consumers":"Power BI / Direct Lake, data agent","active":true},{"transform_id":"EN-G03","industry":"energy","gold_table":"fact_work_order","run_order":120,"sql_text":"SELECT v.wo_id, v.asset_id, v.wo_type, v.priority, v.status, v.raised_ts, v.completed_ts, to_date(v.raised_ts) AS opened_date, CAST(date_format(v.raised_ts, 'yyyyMMdd') AS INT) AS opened_date_key, ROUND((unix_timestamp(v.completed_ts) - unix_timestamp(v.raised_ts)) / 3600.0, 2) AS hours_to_close, CASE WHEN v.status IN ('OPEN', 'IN_PROGRESS') THEN 1 ELSE 0 END AS is_open, v.cost_aud, v.labour_hours, d.region, d.asset_class, d.criticality FROM silver.work_order v LEFT JOIN silver.asset d ON d.asset_id = v.asset_id AND d._is_current = true WHERE v._is_deleted = false","description":"Work orders with duration and asset attributes","business_owner":"Asset Management","consumers":"Power BI / Direct Lake, data agent","active":true},{"transform_id":"EN-G04","industry":"energy","gold_table":"fact_tx_load_hourly","run_order":130,"sql_text":"SELECT asset_id, date_trunc('HOUR', reading_ts) AS reading_hour, to_date(reading_ts) AS reading_date, CAST(date_format(reading_ts, 'yyyyMMdd') AS INT) AS reading_date_key, COUNT(*) AS readings, ROUND(AVG(oil_temp_c), 2) AS avg_oil_temp_c, ROUND(MAX(oil_temp_c), 2) AS max_oil_temp_c, ROUND(AVG(load_mw), 2) AS avg_load_mw, ROUND(AVG(ambient_temp_c), 2) AS avg_ambient_temp_c, SUM(CASE WHEN oil_temp_c > 95 THEN 1 ELSE 0 END) AS breach_readings FROM silver.tx_telemetry GROUP BY asset_id, date_trunc('HOUR', reading_ts), to_date(reading_ts), CAST(date_format(reading_ts, 'yyyyMMdd') AS INT)","description":"Hourly transformer telemetry with threshold breaches","business_owner":"Network Operations","consumers":"Power BI / Direct Lake, data agent","active":true},{"transform_id":"EN-G05","industry":"energy","gold_table":"asset_health_scorecard","run_order":200,"sql_text":"WITH asat AS (SELECT MAX(to_date(raised_ts)) AS as_at FROM silver.work_order), ev AS (SELECT v.asset_id, COUNT(*) AS work_orders_12m, SUM(CASE WHEN v.status IN ('OPEN', 'IN_PROGRESS') THEN 1 ELSE 0 END) AS open_work_orders, SUM(CASE WHEN v.wo_type = 'EMERG' THEN 1 ELSE 0 END) AS urgent_work_orders_12m, ROUND(SUM(v.cost_aud), 2) AS cost_aud_12m FROM silver.work_order v CROSS JOIN asat WHERE v._is_deleted = false AND to_date(v.raised_ts) > add_months(asat.as_at, -12) GROUP BY v.asset_id), rd AS (SELECT asset_id, MAX(oil_temp_c) AS max_oil_temp_c, SUM(CASE WHEN oil_temp_c > 95 THEN 1 ELSE 0 END) AS breach_readings FROM silver.tx_telemetry GROUP BY asset_id), s AS (SELECT d.asset_id, d.asset_tag, d.asset_name, d.asset_class, d.region, d.status, d.criticality, COALESCE(ev.work_orders_12m, 0) AS work_orders_12m, COALESCE(ev.open_work_orders, 0) AS open_work_orders, COALESCE(ev.urgent_work_orders_12m, 0) AS urgent_work_orders_12m, COALESCE(ev.cost_aud_12m, 0) AS cost_aud_12m, rd.max_oil_temp_c, COALESCE(rd.breach_readings, 0) AS breach_readings, d.rating_mva FROM silver.asset d LEFT JOIN ev ON ev.asset_id = d.asset_id LEFT JOIN rd ON rd.asset_id = d.asset_id WHERE d._is_current = true) SELECT s.*, ROUND(LEAST(100, COALESCE(s.criticality, 1) * 8 + s.open_work_orders * 6 + s.urgent_work_orders_12m * 4 + LEAST(s.breach_readings, 20) * 1.5), 1) AS risk_score, CASE WHEN LEAST(100, COALESCE(s.criticality, 1) * 8 + s.open_work_orders * 6 + s.urgent_work_orders_12m * 4 + LEAST(s.breach_readings, 20) * 1.5) >= 60 THEN 'High' WHEN LEAST(100, COALESCE(s.criticality, 1) * 8 + s.open_work_orders * 6 + s.urgent_work_orders_12m * 4 + LEAST(s.breach_readings, 20) * 1.5) >= 35 THEN 'Medium' ELSE 'Low' END AS risk_band, ROUND(s.open_work_orders * 6000 + LEAST(s.breach_readings, 20) * 2500 + COALESCE(s.rating_mva, 0) * COALESCE(s.criticality, 1) * 20, 0) AS outage_cost_at_risk_aud FROM s","description":"Asset risk score - work orders, urgent work and oil temperature breaches, plus outage cost at risk in AUD (weights and dollar rates are illustrative: $6,000 per open work order + $2,500 per 15-min window above the oil-temperature threshold + rating (MVA) x criticality x $20)","business_owner":"Asset Management","consumers":"Power BI / Direct Lake, data agent","active":true},{"transform_id":"EN-G06","industry":"energy","gold_table":"landholder_summary","run_order":210,"sql_text":"SELECT region, landholder_type, COUNT(*) AS landholders, SUM(CASE WHEN status = 'Active' THEN 1 ELSE 0 END) AS active_landholders, ROUND(AVG(year(current_date()) - birth_year), 1) AS avg_age, COUNT(DISTINCT asset_id) AS linked_assets FROM silver.landholder WHERE _is_deleted = false GROUP BY region, landholder_type","description":"Landholders by region and landholder_type","business_owner":"Stakeholder Engagement","consumers":"Power BI / Direct Lake, data agent","active":true}],"onboard":{"object":{"object_id":"EN_SCADA_WEATHER_OBSERVATIONS","industry":"energy","source_system":"scada","source_object":"weather_observations","source_type":"file","ingest_method":"landed_file","landing_format":"txt","landing_path":"landing/scada/weather_observations","read_options":"{\"header\": true, \"delimiter\": \"|\"}","source_table":null,"load_type":"incremental","watermark_column":null,"primary_keys":"region,obs_date","dedupe_order_by":"_ingested_at DESC","bronze_table":"scada_weather_observations","silver_table":"weather_daily","silver_write_mode":"merge_scd1","load_group":"reference","priority":25,"sla_hours":26,"data_owner":"Network Operations","sensitivity":"Public","active":true,"description":"Daily BoM weather observations by region (pipe-delimited, dd/MM/yyyy) - onboarded live in the demo","fabric_alternative":"Public or partner feed - could be a REST copy, or a shortcut if the provider shares a folder"},"columns":[["region","region","string",null],["obs_date","obs_date","date","to_date(obs_date, 'dd/MM/yyyy')"],["max_temp_c","max_temp_c","double",null],["max_wind_gust_kmh","max_wind_gust_kmh","double",null],["rainfall_mm","rainfall_mm","double",null],["fire_danger_rating","fire_danger_rating","string",null]],"rules":[["EN-EXT-01","max_temp_c plausible","technical","range","max_temp_c","{\"min\": -10, \"max\": 50}","warn"],["EN-EXT-02","Known fire_danger_rating","business","allowed_values","fire_danger_rating","{\"values\": [\"Moderate\", \"High\", \"Extreme\", \"Catastrophic\"]}","warn"]],"gold":{"transform_id":"EN-G07","industry":"energy","gold_table":"work_order_vs_weather_daily","run_order":220,"sql_text":"WITH ev AS (SELECT d.region, to_date(v.raised_ts) AS obs_date, COUNT(*) AS work_orders, SUM(CASE WHEN v.wo_type = 'EMERG' THEN 1 ELSE 0 END) AS urgent_work_orders FROM silver.work_order v JOIN silver.asset d ON d.asset_id = v.asset_id AND d._is_current = true WHERE v._is_deleted = false GROUP BY d.region, to_date(v.raised_ts)) SELECT x.region, x.obs_date, x.max_temp_c, x.max_wind_gust_kmh, x.rainfall_mm, x.fire_danger_rating, COALESCE(ev.work_orders, 0) AS work_orders, COALESCE(ev.urgent_work_orders, 0) AS urgent_work_orders FROM silver.weather_daily x LEFT JOIN ev ON ev.region = x.region AND ev.obs_date = x.obs_date WHERE x._is_deleted = false","description":"Fire danger and wind gusts against emergency work orders by region","business_owner":"Network Operations","consumers":"Power BI / Direct Lake","active":true}}},"telco":{"pack":{"id":"telco","industry":"Telecommunications","org":"Southern Cross Telecom","short":"SCT","prefix":"TC","tagline":"mobile, fixed and NBN operator","email_domain":"sct.example.com.au","region_col":"region","region_label":"region","regions":["Sydney Metro","Melbourne Metro","Brisbane","Perth","Adelaide","Regional NSW","Regional VIC"],"name_words":["Parramatta","Docklands","Fortitude Valley","Joondalup","Glenelg","Bathurst","Geelong","Penrith","Box Hill","Ipswich"],"sys":{"ops":{"code":"nims","name":"Network Inventory Management System (on-prem Oracle)","type":"Database","owner":"Network IT","fabric":"Mirroring for Oracle (preview) vs Copy job with CDC vs pipeline Copy through the gateway"},"tel":{"code":"oss","name":"OSS performance counters (15-min CSV) and events calendar","type":"OSS file drop","owner":"Network Performance","fabric":"Eventstream from Kafka/Event Hubs for streaming KPIs; files to OneLake for batch"},"crm":{"code":"dyn","name":"Dynamics 365 Customer Service","type":"SaaS CRM","owner":"Customer Operations","fabric":"Link to Microsoft Fabric (Dataverse) - no copy - or Copy job"}},"entity":{"table":"network_element","src":"network_elements","label":"network element","plural":"network elements","id":"element_id","code":"element_code","code_prefix":"NE","name_col":"element_name","type_col":"element_type","types":["RAN_4G","RAN_5G","CORE_ROUTER","OLT","MICROWAVE"],"status":["ACTIVE","MAINTENANCE","FAULTY","DECOMMISSIONED"],"tier_col":"criticality","tier_label":"criticality (1-5, 5 = most critical)","cap_col":"capacity_users","cap_label":"capacity (users)","cap_range":[200,20000],"start_col":"install_date","start_years":[2008,2025],"drift_col":"firmware_channel","drift_values":["stable","early-adopter","lts"],"owner":"Network Engineering","count":240},"event":{"table":"fault_ticket","src":"fault_tickets","label":"fault ticket","plural":"fault tickets","id":"ticket_id","id_start":700001,"type_col":"fault_category","types":["OUTAGE","DEGRADATION","HARDWARE","CONFIG"],"type_names":["Service outage","Service degradation","Hardware fault","Configuration error"],"urgent_type":"OUTAGE","status":["NEW","IN_PROGRESS","RESOLVED","CANCELLED"],"open_ts":"opened_ts","close_ts":"resolved_ts","open_verb":"opened","close_verb":"resolved","amount_col":"field_cost_aud","amount_range":[0,25000],"qty_col":"engineer_hours","qty_range":[0.5,48],"descriptions":["Cell site down - power failure","High packet loss on backhaul","Card failure in OLT shelf","Wrong VLAN after change","Fibre cut by civil works","Interference reported by customers","Battery backup alarm","Software crash after upgrade"],"owner":"Service Assurance"},"reading":{"table":"cell_kpi","src":"cell_kpi","label":"cell KPI counters","fact":"cell_kpi","monitored":24,"monitored_types":["RAN_4G","RAN_5G"],"metrics":[{"col":"prb_util_pct","label":"PRB utilisation","unit":"%","base":55,"sd":15,"min":0,"max":100,"threshold":90},{"col":"active_users","label":"active users","unit":"users","base":420,"sd":150,"min":0,"max":20000},{"col":"drop_rate_pct","label":"call drop rate","unit":"%","base":0.6,"sd":0.3,"min":0,"max":100}],"sentinel":999.9,"owner":"Network Performance"},"party":{"table":"customer","src":"customers","label":"customer","plural":"customers","id":"customer_id","id_prefix":"CU","sf_type":"Account","segment_col":"segment","segments":["Consumer","Small Business","Enterprise","Government"],"status":["Active","Ported out"],"sensitive_col":"id_document_no","sensitive_label":"ID document number","sensitive_treatment":"drop","owner":"Customer Operations","count":300},"extra":{"table":"major_event","src":"major_events","label":"major events calendar","sys":"tel","cols":[{"col":"expected_crowd","base":25000,"sd":15000,"min":0,"max":150000},{"col":"event_hours","base":4,"sd":2,"min":0,"max":24},{"col":"temp_sites","base":2,"sd":1,"min":0,"max":20}],"cat_col":"event_kind","cat_values":["Sport","Concert","Festival","Conference"],"owner":"Network Performance","story":"crowd events against network faults by region"},"alert":{"stream":"network_alarm","label":"network alarm","codes":["LINK_DOWN","HIGH_TEMP","POWER_FAIL"],"storm":"stadium surge"},"ref":{"title":"Fault restoration targets","owner":"Service Assurance","rows":[["OUTAGE","Critical",4,"NOC"],["DEGRADATION","Major",12,"Network Performance"],["HARDWARE","Major",24,"Field Services"],["CONFIG","Minor",48,"Network Engineering"]]},"ai":{"persona":"network operations and service assurance","cust":"mobile, fixed and wholesale network operators","q_model":["Which region had the most outage tickets this month?","Show average hours to resolve by fault category.","Which critical network elements have open fault tickets?"],"q_agent":["Which 10 cells have the highest PRB utilisation and how many open faults do they have?","Compare mean time to resolve outage tickets between Sydney Metro and Melbourne Metro.","Which network elements breached 90% PRB utilisation in the last 3 days?","What did field visits cost us for hardware faults by region?","How many enterprise customers are active in Brisbane?"],"q_onto":["Show 5G elements in Sydney Metro with an open outage ticket and a high risk band.","For the most congested cell, list its fault tickets and the customers linked to it.","Which customers are linked to elements that are currently faulty?"]},"impact":{"col":"revenue_at_risk_aud","label":"revenue at risk","open_cost":4000,"breach_cost":600,"cap_rate":0.5,"rule":"$4,000 per open fault + $600 per congested 15-min window + capacity (users) x criticality x $0.50"}},"source_system":[{"source_system":"nims","industry":"telco","display_name":"Network Inventory Management System (on-prem Oracle)","system_type":"Database","connection_pattern":"On-premises data gateway -> pipeline Copy with watermark query","technical_owner":"Network IT","fabric_options":"Mirroring for Oracle (preview) vs Copy job with CDC vs pipeline Copy through the gateway"},{"source_system":"oss","industry":"telco","display_name":"OSS performance counters (15-min CSV) and events calendar","system_type":"OSS file drop","connection_pattern":"Files dropped to OneLake (Files/landing) every 15 minutes","technical_owner":"Network Performance","fabric_options":"Eventstream from Kafka/Event Hubs for streaming KPIs; files to OneLake for batch"},{"source_system":"dyn","industry":"telco","display_name":"Dynamics 365 Customer Service","system_type":"SaaS CRM","connection_pattern":"REST/OData API -> JSON landed per day","technical_owner":"Customer Operations","fabric_options":"Link to Microsoft Fabric (Dataverse) - no copy - or Copy job"}],"ingestion_object":[{"object_id":"TC_NIMS_NETWORK_ELEMENTS","industry":"telco","source_system":"nims","source_object":"network_elements","source_type":"database","ingest_method":"copy_db","landing_format":"csv","landing_path":"landing/nims/network_elements","read_options":"{\"header\": true, \"delimiter\": \",\"}","source_table":"src.network_elements","load_type":"incremental","watermark_column":"last_modified","primary_keys":"element_id","dedupe_order_by":"last_modified DESC, _ingested_at DESC","bronze_table":"nims_network_elements","silver_table":"network_element","silver_write_mode":"merge_scd2","load_group":"reference","priority":10,"sla_hours":26,"data_owner":"Network Engineering","sensitivity":"Internal","active":true,"description":"Network element register. SCD2 keeps status, criticality and capacity_users history.","fabric_alternative":"Strong mirroring candidate - small, slowly changing, needed in near real time by operations"},{"object_id":"TC_NIMS_FAULT_TICKETS","industry":"telco","source_system":"nims","source_object":"fault_tickets","source_type":"database","ingest_method":"copy_db","landing_format":"csv","landing_path":"landing/nims/fault_tickets","read_options":"{\"header\": true, \"delimiter\": \",\"}","source_table":"src.fault_tickets","load_type":"incremental","watermark_column":"last_modified","primary_keys":"ticket_id","dedupe_order_by":"last_modified DESC, _ingested_at DESC","bronze_table":"nims_fault_tickets","silver_table":"fault_ticket","silver_write_mode":"merge_scd1","load_group":"transactional","priority":20,"sla_hours":26,"data_owner":"Service Assurance","sensitivity":"Internal","active":true,"description":"Fault tickets - incremental by last_modified; can arrive before their network element (late-arriving).","fabric_alternative":"Copy job incremental or mirroring; keep the watermark query if the source is on-prem"},{"object_id":"TC_OSS_CELL_KPI","industry":"telco","source_system":"oss","source_object":"cell_kpi","source_type":"file","ingest_method":"landed_file","landing_format":"csv","landing_path":"landing/oss/cell_kpi","read_options":"{\"header\": true, \"delimiter\": \",\"}","source_table":null,"load_type":"incremental","watermark_column":null,"primary_keys":"element_id,reading_ts","dedupe_order_by":"_ingested_at DESC","bronze_table":"oss_cell_kpi","silver_table":"cell_kpi","silver_write_mode":"insert_only","load_group":"telemetry","priority":30,"sla_hours":2,"data_owner":"Network Performance","sensitivity":"Internal","active":true,"description":"Cell kpi counters every 15 minutes, four micro-batch files a day. Insert-only and idempotent.","fabric_alternative":"Eventstream into Eventhouse for real time; keep this batch path for history and reconciliation"},{"object_id":"TC_DYN_CUSTOMERS","industry":"telco","source_system":"dyn","source_object":"customers","source_type":"api","ingest_method":"landed_file","landing_format":"json","landing_path":"landing/dyn/customers","read_options":"{\"multiline\": true, \"json_root\": \"records\"}","source_table":null,"load_type":"full","watermark_column":null,"primary_keys":"customer_id","dedupe_order_by":"last_modified_ts DESC, _ingested_at DESC","bronze_table":"dyn_customers","silver_table":"customer","silver_write_mode":"merge_scd1","load_group":"reference","priority":15,"sla_hours":26,"data_owner":"Customer Operations","sensitivity":"Confidential","active":true,"description":"Customers - full daily snapshot from the CRM API (contains PII). Missing rows are soft-deleted.","fabric_alternative":"Copy job with the CRM connector; Dataverse-based CRMs can use Link to Fabric instead of copying"}],"column_map":[{"object_id":"TC_NIMS_NETWORK_ELEMENTS","column_order":0,"source_column":"element_id","target_column":"element_id","data_type":"int","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"TC_NIMS_NETWORK_ELEMENTS","column_order":1,"source_column":"element_code","target_column":"element_code","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"TC_NIMS_NETWORK_ELEMENTS","column_order":2,"source_column":"element_name","target_column":"element_name","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"TC_NIMS_NETWORK_ELEMENTS","column_order":3,"source_column":"element_type","target_column":"element_type","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"TC_NIMS_NETWORK_ELEMENTS","column_order":4,"source_column":"region","target_column":"region","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"TC_NIMS_NETWORK_ELEMENTS","column_order":5,"source_column":"status","target_column":"status","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"TC_NIMS_NETWORK_ELEMENTS","column_order":6,"source_column":"criticality","target_column":"criticality","data_type":"int","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"TC_NIMS_NETWORK_ELEMENTS","column_order":7,"source_column":"capacity_users","target_column":"capacity_users","data_type":"double","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"TC_NIMS_NETWORK_ELEMENTS","column_order":8,"source_column":"install_date","target_column":"install_date","data_type":"date","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"TC_NIMS_NETWORK_ELEMENTS","column_order":9,"source_column":"last_modified","target_column":"last_modified","data_type":"timestamp","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"TC_NIMS_FAULT_TICKETS","column_order":0,"source_column":"ticket_id","target_column":"ticket_id","data_type":"int","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"TC_NIMS_FAULT_TICKETS","column_order":1,"source_column":"element_id","target_column":"element_id","data_type":"int","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"TC_NIMS_FAULT_TICKETS","column_order":2,"source_column":"fault_category","target_column":"fault_category","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"TC_NIMS_FAULT_TICKETS","column_order":3,"source_column":"priority","target_column":"priority","data_type":"int","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"TC_NIMS_FAULT_TICKETS","column_order":4,"source_column":"status","target_column":"status","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"TC_NIMS_FAULT_TICKETS","column_order":5,"source_column":"opened_ts","target_column":"opened_ts","data_type":"timestamp","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"TC_NIMS_FAULT_TICKETS","column_order":6,"source_column":"resolved_ts","target_column":"resolved_ts","data_type":"timestamp","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"TC_NIMS_FAULT_TICKETS","column_order":7,"source_column":"field_cost_aud","target_column":"field_cost_aud","data_type":"decimal(12,2)","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"TC_NIMS_FAULT_TICKETS","column_order":8,"source_column":"engineer_hours","target_column":"engineer_hours","data_type":"double","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"TC_NIMS_FAULT_TICKETS","column_order":9,"source_column":"description","target_column":"description","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"TC_NIMS_FAULT_TICKETS","column_order":10,"source_column":"last_modified","target_column":"last_modified","data_type":"timestamp","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"TC_OSS_CELL_KPI","column_order":0,"source_column":"element_id","target_column":"element_id","data_type":"int","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"TC_OSS_CELL_KPI","column_order":1,"source_column":"reading_time","target_column":"reading_ts","data_type":"timestamp","transform_expr":"to_timestamp(reading_time, 'dd/MM/yyyy HH:mm')","pii_treatment":"none","sensitivity":"none","description":""},{"object_id":"TC_OSS_CELL_KPI","column_order":2,"source_column":"prb_util_pct","target_column":"prb_util_pct","data_type":"double","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"TC_OSS_CELL_KPI","column_order":3,"source_column":"active_users","target_column":"active_users","data_type":"double","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"TC_OSS_CELL_KPI","column_order":4,"source_column":"drop_rate_pct","target_column":"drop_rate_pct","data_type":"double","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"TC_DYN_CUSTOMERS","column_order":0,"source_column":"Id","target_column":"customer_id","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"TC_DYN_CUSTOMERS","column_order":1,"source_column":"FirstName","target_column":"first_name","data_type":"string","transform_expr":null,"pii_treatment":"drop","sensitivity":"Confidential","description":""},{"object_id":"TC_DYN_CUSTOMERS","column_order":2,"source_column":"LastName","target_column":"last_name","data_type":"string","transform_expr":null,"pii_treatment":"drop","sensitivity":"Confidential","description":""},{"object_id":"TC_DYN_CUSTOMERS","column_order":3,"source_column":"Email","target_column":"email_masked","data_type":"string","transform_expr":null,"pii_treatment":"mask_email","sensitivity":"Confidential","description":""},{"object_id":"TC_DYN_CUSTOMERS","column_order":4,"source_column":"MobilePhone","target_column":"mobile_masked","data_type":"string","transform_expr":null,"pii_treatment":"mask_phone","sensitivity":"Confidential","description":""},{"object_id":"TC_DYN_CUSTOMERS","column_order":5,"source_column":"Birthdate","target_column":"birth_year","data_type":"int","transform_expr":null,"pii_treatment":"year_only","sensitivity":"Confidential","description":""},{"object_id":"TC_DYN_CUSTOMERS","column_order":6,"source_column":"MailingPostalCode","target_column":"postcode","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"TC_DYN_CUSTOMERS","column_order":7,"source_column":"Region__c","target_column":"region","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"TC_DYN_CUSTOMERS","column_order":8,"source_column":"Segment__c","target_column":"segment","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"TC_DYN_CUSTOMERS","column_order":9,"source_column":"ElementId__c","target_column":"element_id","data_type":"int","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"TC_DYN_CUSTOMERS","column_order":10,"source_column":"Status__c","target_column":"status","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"TC_DYN_CUSTOMERS","column_order":11,"source_column":"IdDocumentNo__c","target_column":"id_document_no","data_type":"string","transform_expr":null,"pii_treatment":"drop","sensitivity":"Restricted","description":""},{"object_id":"TC_DYN_CUSTOMERS","column_order":12,"source_column":"LastModifiedDate","target_column":"last_modified_ts","data_type":"timestamp","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""}],"dq_rule":[{"rule_id":"TC-ENT-01","object_id":"TC_NIMS_NETWORK_ELEMENTS","rule_name":"Network element id present","rule_category":"technical","check_type":"not_null","column_name":"element_id","params":"{}","severity":"fail","fail_threshold_pct":0,"rule_owner":"Data engineering","description":"Primary key must never be null","active":true},{"rule_id":"TC-ENT-02","object_id":"TC_NIMS_NETWORK_ELEMENTS","rule_name":"Known element_type","rule_category":"technical","check_type":"allowed_values","column_name":"element_type","params":"{\"values\": [\"RAN_4G\", \"RAN_5G\", \"CORE_ROUTER\", \"OLT\", \"MICROWAVE\"]}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Data engineering","description":"Unknown classes are kept but flagged for the data owner","active":true},{"rule_id":"TC-ENT-03","object_id":"TC_NIMS_NETWORK_ELEMENTS","rule_name":"criticality between 1 and 5","rule_category":"technical","check_type":"range","column_name":"criticality","params":"{\"min\": 1, \"max\": 5}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Data engineering","description":"Out-of-range ratings usually mean a keying error in the source","active":true},{"rule_id":"TC-ENT-04","object_id":"TC_NIMS_NETWORK_ELEMENTS","rule_name":"Known status","rule_category":"business","check_type":"allowed_values","column_name":"status","params":"{\"values\": [\"ACTIVE\", \"MAINTENANCE\", \"FAULTY\", \"DECOMMISSIONED\"]}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Network Engineering","description":"Status list agreed with Network Engineering","active":true},{"rule_id":"TC-EVT-01","object_id":"TC_NIMS_FAULT_TICKETS","rule_name":"Fault ticket id present","rule_category":"technical","check_type":"not_null","column_name":"ticket_id","params":"{}","severity":"fail","fail_threshold_pct":0,"rule_owner":"Data engineering","description":"Primary key must never be null","active":true},{"rule_id":"TC-EVT-02","object_id":"TC_NIMS_FAULT_TICKETS","rule_name":"Network element exists","rule_category":"technical","check_type":"referential","column_name":"element_id","params":"{\"ref_table\": \"silver.network_element\", \"ref_column\": \"element_id\"}","severity":"quarantine","fail_threshold_pct":null,"rule_owner":"Data engineering","description":"Held until the network element arrives (late-arriving reference data), then merged automatically","active":true},{"rule_id":"TC-EVT-03","object_id":"TC_NIMS_FAULT_TICKETS","rule_name":"Resolved after opened","rule_category":"business","check_type":"expression","column_name":null,"params":"{\"expr\": \"resolved_ts IS NULL OR resolved_ts >= opened_ts\"}","severity":"quarantine","fail_threshold_pct":null,"rule_owner":"Service Assurance","description":"A fault ticket can't be resolved before it was opened","active":true},{"rule_id":"TC-EVT-04","object_id":"TC_NIMS_FAULT_TICKETS","rule_name":"field_cost_aud not negative","rule_category":"technical","check_type":"range","column_name":"field_cost_aud","params":"{\"min\": 0}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Data engineering","description":"Negative values are sign errors in the source extract","active":true},{"rule_id":"TC-EVT-05","object_id":"TC_NIMS_FAULT_TICKETS","rule_name":"OUTAGE is priority 1","rule_category":"business","check_type":"expression","column_name":null,"params":"{\"expr\": \"fault_category <> 'OUTAGE' OR priority = 1\"}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Service Assurance","description":"Service outage fault tickets must be raised as priority 1","active":true},{"rule_id":"TC-EVT-06","object_id":"TC_NIMS_FAULT_TICKETS","rule_name":"opened_ts parses","rule_category":"technical","check_type":"not_null","column_name":"opened_ts","params":"{}","severity":"fail","fail_threshold_pct":5,"rule_owner":"Data engineering","description":"If more than 5% of timestamps fail to parse, the extract format has changed - stop the load","active":true},{"rule_id":"TC-RDG-01","object_id":"TC_OSS_CELL_KPI","rule_name":"Prb utilisation plausible","rule_category":"technical","check_type":"range","column_name":"prb_util_pct","params":"{\"min\": 0, \"max\": 100}","severity":"quarantine","fail_threshold_pct":null,"rule_owner":"Data engineering","description":"Sensor fault values (e.g. 999.9) are quarantined, never averaged into Gold","active":true},{"rule_id":"TC-RDG-02","object_id":"TC_OSS_CELL_KPI","rule_name":"Network element known","rule_category":"technical","check_type":"referential","column_name":"element_id","params":"{\"ref_table\": \"silver.network_element\", \"ref_column\": \"element_id\"}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Data engineering","description":"Readings for unknown tags are flagged","active":true},{"rule_id":"TC-RDG-03","object_id":"TC_OSS_CELL_KPI","rule_name":"Reading time parses","rule_category":"technical","check_type":"not_null","column_name":"reading_ts","params":"{}","severity":"fail","fail_threshold_pct":5,"rule_owner":"Data engineering","description":"Timestamp format change stops the load","active":true},{"rule_id":"TC-PTY-01","object_id":"TC_DYN_CUSTOMERS","rule_name":"Customer id present","rule_category":"technical","check_type":"not_null","column_name":"customer_id","params":"{}","severity":"fail","fail_threshold_pct":0,"rule_owner":"Data engineering","description":"Primary key must never be null","active":true},{"rule_id":"TC-PTY-02","object_id":"TC_DYN_CUSTOMERS","rule_name":"Email looks valid","rule_category":"technical","check_type":"regex","column_name":"email_masked","params":"{\"pattern\": \"^[^@\\\\s]+@[^@\\\\s]+\\\\.[^@\\\\s]+$\"}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Data engineering","description":"Checked on the raw value before masking","active":true},{"rule_id":"TC-PTY-03","object_id":"TC_DYN_CUSTOMERS","rule_name":"Australian postcode","rule_category":"technical","check_type":"regex","column_name":"postcode","params":"{\"pattern\": \"^\\\\d{4}$\"}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Data engineering","description":"Four digits","active":true},{"rule_id":"TC-PTY-04","object_id":"TC_DYN_CUSTOMERS","rule_name":"Known segment","rule_category":"business","check_type":"allowed_values","column_name":"segment","params":"{\"values\": [\"Consumer\", \"Small Business\", \"Enterprise\", \"Government\"]}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Customer Operations","description":"Segments agreed with Customer Operations","active":true}],"gold_transform":[{"transform_id":"TC-G01","industry":"telco","gold_table":"dim_date","run_order":100,"sql_text":"SELECT CAST(date_format(d, 'yyyyMMdd') AS INT) AS date_key, d AS calendar_date, year(d) AS calendar_year, quarter(d) AS calendar_quarter, month(d) AS month_number, date_format(d, 'MMM') AS month_name, date_format(d, 'yyyy-MM') AS year_month, dayofmonth(d) AS day_of_month, date_format(d, 'EEE') AS day_name, CASE WHEN month(d) >= 7 THEN year(d) + 1 ELSE year(d) END AS fy_year, concat('FY', substring(CAST(CASE WHEN month(d) >= 7 THEN year(d) + 1 ELSE year(d) END AS STRING), 3, 2)) AS fy_label, CASE WHEN dayofweek(d) IN (1, 7) THEN true ELSE false END AS is_weekend FROM (SELECT explode(sequence(to_date('2020-01-01'), to_date('2028-12-31'), interval 1 day)) AS d) dates","description":"Calendar with Australian financial year","business_owner":"Data engineering","consumers":"Power BI / Direct Lake, data agent","active":true},{"transform_id":"TC-G02","industry":"telco","gold_table":"dim_network_element","run_order":110,"sql_text":"SELECT element_id, element_code, element_name, element_type, region, status, criticality, capacity_users, install_date, CAST(floor(months_between(current_date(), install_date) / 12) AS INT) AS age_years, _valid_from AS version_valid_from FROM silver.network_element WHERE _is_current = true","description":"Current view of the network element register (history stays in Silver SCD2)","business_owner":"Network Engineering","consumers":"Power BI / Direct Lake, data agent","active":true},{"transform_id":"TC-G03","industry":"telco","gold_table":"fact_fault_ticket","run_order":120,"sql_text":"SELECT v.ticket_id, v.element_id, v.fault_category, v.priority, v.status, v.opened_ts, v.resolved_ts, to_date(v.opened_ts) AS opened_date, CAST(date_format(v.opened_ts, 'yyyyMMdd') AS INT) AS opened_date_key, ROUND((unix_timestamp(v.resolved_ts) - unix_timestamp(v.opened_ts)) / 3600.0, 2) AS hours_to_close, CASE WHEN v.status IN ('NEW', 'IN_PROGRESS') THEN 1 ELSE 0 END AS is_open, v.field_cost_aud, v.engineer_hours, d.region, d.element_type, d.criticality FROM silver.fault_ticket v LEFT JOIN silver.network_element d ON d.element_id = v.element_id AND d._is_current = true WHERE v._is_deleted = false","description":"Fault tickets with duration and network element attributes","business_owner":"Service Assurance","consumers":"Power BI / Direct Lake, data agent","active":true},{"transform_id":"TC-G04","industry":"telco","gold_table":"fact_cell_kpi_hourly","run_order":130,"sql_text":"SELECT element_id, date_trunc('HOUR', reading_ts) AS reading_hour, to_date(reading_ts) AS reading_date, CAST(date_format(reading_ts, 'yyyyMMdd') AS INT) AS reading_date_key, COUNT(*) AS readings, ROUND(AVG(prb_util_pct), 2) AS avg_prb_util_pct, ROUND(MAX(prb_util_pct), 2) AS max_prb_util_pct, ROUND(AVG(active_users), 2) AS avg_active_users, ROUND(AVG(drop_rate_pct), 2) AS avg_drop_rate_pct, SUM(CASE WHEN prb_util_pct > 90 THEN 1 ELSE 0 END) AS breach_readings FROM silver.cell_kpi GROUP BY element_id, date_trunc('HOUR', reading_ts), to_date(reading_ts), CAST(date_format(reading_ts, 'yyyyMMdd') AS INT)","description":"Hourly cell KPI counters with threshold breaches","business_owner":"Network Performance","consumers":"Power BI / Direct Lake, data agent","active":true},{"transform_id":"TC-G05","industry":"telco","gold_table":"network_element_health_scorecard","run_order":200,"sql_text":"WITH asat AS (SELECT MAX(to_date(opened_ts)) AS as_at FROM silver.fault_ticket), ev AS (SELECT v.element_id, COUNT(*) AS fault_tickets_12m, SUM(CASE WHEN v.status IN ('NEW', 'IN_PROGRESS') THEN 1 ELSE 0 END) AS open_fault_tickets, SUM(CASE WHEN v.fault_category = 'OUTAGE' THEN 1 ELSE 0 END) AS urgent_fault_tickets_12m, ROUND(SUM(v.field_cost_aud), 2) AS field_cost_aud_12m FROM silver.fault_ticket v CROSS JOIN asat WHERE v._is_deleted = false AND to_date(v.opened_ts) > add_months(asat.as_at, -12) GROUP BY v.element_id), rd AS (SELECT element_id, MAX(prb_util_pct) AS max_prb_util_pct, SUM(CASE WHEN prb_util_pct > 90 THEN 1 ELSE 0 END) AS breach_readings FROM silver.cell_kpi GROUP BY element_id), s AS (SELECT d.element_id, d.element_code, d.element_name, d.element_type, d.region, d.status, d.criticality, COALESCE(ev.fault_tickets_12m, 0) AS fault_tickets_12m, COALESCE(ev.open_fault_tickets, 0) AS open_fault_tickets, COALESCE(ev.urgent_fault_tickets_12m, 0) AS urgent_fault_tickets_12m, COALESCE(ev.field_cost_aud_12m, 0) AS field_cost_aud_12m, rd.max_prb_util_pct, COALESCE(rd.breach_readings, 0) AS breach_readings, d.capacity_users FROM silver.network_element d LEFT JOIN ev ON ev.element_id = d.element_id LEFT JOIN rd ON rd.element_id = d.element_id WHERE d._is_current = true) SELECT s.*, ROUND(LEAST(100, COALESCE(s.criticality, 1) * 8 + s.open_fault_tickets * 6 + s.urgent_fault_tickets_12m * 4 + LEAST(s.breach_readings, 20) * 1.5), 1) AS risk_score, CASE WHEN LEAST(100, COALESCE(s.criticality, 1) * 8 + s.open_fault_tickets * 6 + s.urgent_fault_tickets_12m * 4 + LEAST(s.breach_readings, 20) * 1.5) >= 60 THEN 'High' WHEN LEAST(100, COALESCE(s.criticality, 1) * 8 + s.open_fault_tickets * 6 + s.urgent_fault_tickets_12m * 4 + LEAST(s.breach_readings, 20) * 1.5) >= 35 THEN 'Medium' ELSE 'Low' END AS risk_band, ROUND(s.open_fault_tickets * 4000 + LEAST(s.breach_readings, 20) * 600 + COALESCE(s.capacity_users, 0) * COALESCE(s.criticality, 1) * 0.5, 0) AS revenue_at_risk_aud FROM s","description":"Network element risk score - fault tickets, urgent work and PRB utilisation breaches, plus revenue at risk in AUD (weights and dollar rates are illustrative: $4,000 per open fault + $600 per congested 15-min window + capacity (users) x criticality x $0.50)","business_owner":"Network Engineering","consumers":"Power BI / Direct Lake, data agent","active":true},{"transform_id":"TC-G06","industry":"telco","gold_table":"customer_summary","run_order":210,"sql_text":"SELECT region, segment, COUNT(*) AS customers, SUM(CASE WHEN status = 'Active' THEN 1 ELSE 0 END) AS active_customers, ROUND(AVG(year(current_date()) - birth_year), 1) AS avg_age, COUNT(DISTINCT element_id) AS linked_network_elements FROM silver.customer WHERE _is_deleted = false GROUP BY region, segment","description":"Customers by region and segment","business_owner":"Customer Operations","consumers":"Power BI / Direct Lake, data agent","active":true}],"onboard":{"object":{"object_id":"TC_OSS_MAJOR_EVENTS","industry":"telco","source_system":"oss","source_object":"major_events","source_type":"file","ingest_method":"landed_file","landing_format":"txt","landing_path":"landing/oss/major_events","read_options":"{\"header\": true, \"delimiter\": \"|\"}","source_table":null,"load_type":"incremental","watermark_column":null,"primary_keys":"region,obs_date","dedupe_order_by":"_ingested_at DESC","bronze_table":"oss_major_events","silver_table":"major_event","silver_write_mode":"merge_scd1","load_group":"reference","priority":25,"sla_hours":26,"data_owner":"Network Performance","sensitivity":"Public","active":true,"description":"Daily major events calendar by region (pipe-delimited, dd/MM/yyyy) - onboarded live in the demo","fabric_alternative":"Public or partner feed - could be a REST copy, or a shortcut if the provider shares a folder"},"columns":[["region","region","string",null],["obs_date","obs_date","date","to_date(obs_date, 'dd/MM/yyyy')"],["expected_crowd","expected_crowd","double",null],["event_hours","event_hours","double",null],["temp_sites","temp_sites","double",null],["event_kind","event_kind","string",null]],"rules":[["TC-EXT-01","expected_crowd plausible","technical","range","expected_crowd","{\"min\": 0, \"max\": 150000}","warn"],["TC-EXT-02","Known event_kind","business","allowed_values","event_kind","{\"values\": [\"Sport\", \"Concert\", \"Festival\", \"Conference\"]}","warn"]],"gold":{"transform_id":"TC-G07","industry":"telco","gold_table":"fault_ticket_vs_major_event","run_order":220,"sql_text":"WITH ev AS (SELECT d.region, to_date(v.opened_ts) AS obs_date, COUNT(*) AS fault_tickets, SUM(CASE WHEN v.fault_category = 'OUTAGE' THEN 1 ELSE 0 END) AS urgent_fault_tickets FROM silver.fault_ticket v JOIN silver.network_element d ON d.element_id = v.element_id AND d._is_current = true WHERE v._is_deleted = false GROUP BY d.region, to_date(v.opened_ts)) SELECT x.region, x.obs_date, x.expected_crowd, x.event_hours, x.temp_sites, x.event_kind, COALESCE(ev.fault_tickets, 0) AS fault_tickets, COALESCE(ev.urgent_fault_tickets, 0) AS urgent_fault_tickets FROM silver.major_event x LEFT JOIN ev ON ev.region = x.region AND ev.obs_date = x.obs_date WHERE x._is_deleted = false","description":"Crowd events against network faults by region","business_owner":"Network Performance","consumers":"Power BI / Direct Lake","active":true}}},"fsi":{"pack":{"id":"fsi","industry":"Financial services","org":"Harbour Mutual Bank","short":"HMB","prefix":"FS","tagline":"retail and business bank","email_domain":"hmb.example.com.au","region_col":"state","region_label":"state","regions":["NSW","VIC","QLD","WA","SA","TAS","ACT"],"name_words":["Everyday","Bonus Saver","Offset","Business","Platinum","Kids","Term","Rewards","Farm","Smart"],"sys":{"ops":{"code":"core","name":"Core banking platform (on-prem SQL Server)","type":"Database","owner":"Core Banking","fabric":"Mirroring for SQL Server vs Copy job incremental; keep PII handling in mind before mirroring"},"tel":{"code":"pay","name":"Payments transaction-monitoring files (15-min windows) and Scamwatch feed","type":"File drop (SFTP)","owner":"Payments","fabric":"Eventstream for real-time payment events; files to OneLake for reconciliation"},"crm":{"code":"sfdc","name":"Salesforce Financial Services Cloud","type":"SaaS CRM","owner":"Customer Experience","fabric":"Copy job with the Salesforce connector"}},"entity":{"table":"account","src":"accounts","label":"account","plural":"accounts","id":"account_id","code":"account_number","code_prefix":"ACC","name_col":"account_name","type_col":"product_type","types":["TRANSACTION","SAVINGS","PERSONAL_LOAN","HOME_LOAN","CREDIT_CARD"],"status":["PERFORMING","DORMANT","ARREARS","CLOSED"],"tier_col":"risk_grade","tier_label":"credit risk grade (1-5, 5 = highest risk)","cap_col":"balance_aud","cap_label":"balance or limit (AUD)","cap_range":[1000,900000],"start_col":"open_date","start_years":[1995,2025],"drift_col":"hardship_flag","drift_values":["Y","N"],"owner":"Retail Credit Risk","count":240},"event":{"table":"risk_case","src":"risk_cases","label":"risk case","plural":"risk cases","id":"case_id","id_start":900001,"type_col":"case_type","types":["CARD_FRAUD","SCAM","ACCOUNT_TAKEOVER","HARDSHIP"],"type_names":["Card fraud","Scam payment","Account takeover","Hardship / arrears"],"urgent_type":"ACCOUNT_TAKEOVER","status":["OPEN","INVESTIGATING","CLOSED","WITHDRAWN"],"open_ts":"raised_ts","close_ts":"closed_ts","open_verb":"raised","close_verb":"closed","amount_col":"case_amount_aud","amount_range":[0,60000],"qty_col":"analyst_hours","qty_range":[0.25,40],"descriptions":["Card testing pattern detected","Customer reported romance scam","New device and password reset","Missed two home loan repayments","Hardship request after job loss","Remote access scam call","Mule account behaviour","Credit card over limit for 60 days"],"owner":"Fraud & Collections"},"reading":{"table":"txn_monitor","src":"txn_monitoring","label":"account transaction monitoring (15-min windows)","fact":"txn_monitor","monitored":24,"monitored_types":["TRANSACTION","CREDIT_CARD"],"metrics":[{"col":"decline_pct","label":"declined transaction rate","unit":"%","base":4,"sd":2.5,"min":0,"max":100,"threshold":30},{"col":"txn_value_aud","label":"transaction value","unit":"AUD","base":380,"sd":220,"min":0,"max":200000},{"col":"txn_count","label":"transactions","unit":"count","base":6,"sd":3,"min":0,"max":10000}],"sentinel":999.9,"owner":"Payments"},"party":{"table":"customer","src":"customers","label":"customer","plural":"customers","id":"customer_id","id_prefix":"C","sf_type":"Account","segment_col":"segment","segments":["Retail","Premium","Business","Agribusiness"],"status":["Active","Exited"],"sensitive_col":"tfn","sensitive_label":"tax file number","sensitive_treatment":"drop","owner":"Customer Experience","count":300},"extra":{"table":"scam_report","src":"scam_reports","label":"Scamwatch reports","sys":"tel","cols":[{"col":"reports","base":40,"sd":15,"min":0,"max":5000},{"col":"reported_loss_aud","base":90000,"sd":40000,"min":0,"max":10000000},{"col":"phishing_share_pct","base":35,"sd":10,"min":0,"max":100}],"cat_col":"top_scam_type","cat_values":["Phishing","Investment","Romance","Remote access"],"owner":"Financial Crime","story":"public scam reports against risk cases by state"},"alert":{"stream":"fraud_signal","label":"fraud signal","codes":["CARD_TESTING","IMPOSSIBLE_TRAVEL","HIGH_RISK_MCC"],"storm":"card testing burst"},"ref":{"title":"Risk case handling targets","owner":"Fraud & Collections","rows":[["CARD_FRAUD","Fraud",48,"Card Fraud Ops"],["SCAM","Fraud",24,"Scam Response"],["ACCOUNT_TAKEOVER","Critical",4,"Fraud Rapid Response"],["HARDSHIP","Credit",504,"Collections & Hardship"]]},"impact":{"col":"expected_loss_aud","label":"expected loss","open_cost":3500,"breach_cost":250,"cap_rate":0.002,"rule":"$3,500 per open case + $250 per suspicious 15-min window + balance x risk grade x 0.2%"},"ai":{"persona":"fraud, credit risk and collections","cust":"retail and business banks, mutuals and credit unions","q_model":["What was the total case amount by case type this financial year?","Which state has the most accounts in arrears?","Show total expected loss by product type."],"q_agent":["Which 10 accounts have the highest expected loss, and what is their risk band and number of open cases?","Compare average hours to close account takeover cases by state.","Which accounts had a declined transaction rate above 30% in the last 3 days?","How many home loan accounts are in arrears with an open hardship case, and what is their total balance?","How much have we lost to scams in NSW compared with VIC?"],"q_onto":["Show credit card accounts in NSW with an open account takeover case and a high risk band.","For the account with the highest expected loss, list its risk cases and the customer segment behind it.","Which customers own accounts that are in arrears and have an open fraud or hardship case?"]}},"source_system":[{"source_system":"core","industry":"fsi","display_name":"Core banking platform (on-prem SQL Server)","system_type":"Database","connection_pattern":"On-premises data gateway -> pipeline Copy with watermark query","technical_owner":"Core Banking","fabric_options":"Mirroring for SQL Server vs Copy job incremental; keep PII handling in mind before mirroring"},{"source_system":"pay","industry":"fsi","display_name":"Payments transaction-monitoring files (15-min windows) and Scamwatch feed","system_type":"File drop (SFTP)","connection_pattern":"Files dropped to OneLake (Files/landing) every 15 minutes","technical_owner":"Payments","fabric_options":"Eventstream for real-time payment events; files to OneLake for reconciliation"},{"source_system":"sfdc","industry":"fsi","display_name":"Salesforce Financial Services Cloud","system_type":"SaaS CRM","connection_pattern":"REST/OData API -> JSON landed per day","technical_owner":"Customer Experience","fabric_options":"Copy job with the Salesforce connector"}],"ingestion_object":[{"object_id":"FS_CORE_ACCOUNTS","industry":"fsi","source_system":"core","source_object":"accounts","source_type":"database","ingest_method":"copy_db","landing_format":"csv","landing_path":"landing/core/accounts","read_options":"{\"header\": true, \"delimiter\": \",\"}","source_table":"src.accounts","load_type":"incremental","watermark_column":"last_modified","primary_keys":"account_id","dedupe_order_by":"last_modified DESC, _ingested_at DESC","bronze_table":"core_accounts","silver_table":"account","silver_write_mode":"merge_scd2","load_group":"reference","priority":10,"sla_hours":26,"data_owner":"Retail Credit Risk","sensitivity":"Internal","active":true,"description":"Account register. SCD2 keeps status, risk_grade and balance_aud history.","fabric_alternative":"Strong mirroring candidate - small, slowly changing, needed in near real time by operations"},{"object_id":"FS_CORE_RISK_CASES","industry":"fsi","source_system":"core","source_object":"risk_cases","source_type":"database","ingest_method":"copy_db","landing_format":"csv","landing_path":"landing/core/risk_cases","read_options":"{\"header\": true, \"delimiter\": \",\"}","source_table":"src.risk_cases","load_type":"incremental","watermark_column":"last_modified","primary_keys":"case_id","dedupe_order_by":"last_modified DESC, _ingested_at DESC","bronze_table":"core_risk_cases","silver_table":"risk_case","silver_write_mode":"merge_scd1","load_group":"transactional","priority":20,"sla_hours":26,"data_owner":"Fraud & Collections","sensitivity":"Internal","active":true,"description":"Risk cases - incremental by last_modified; can arrive before their account (late-arriving).","fabric_alternative":"Copy job incremental or mirroring; keep the watermark query if the source is on-prem"},{"object_id":"FS_PAY_TXN_MONITORING","industry":"fsi","source_system":"pay","source_object":"txn_monitoring","source_type":"file","ingest_method":"landed_file","landing_format":"csv","landing_path":"landing/pay/txn_monitoring","read_options":"{\"header\": true, \"delimiter\": \",\"}","source_table":null,"load_type":"incremental","watermark_column":null,"primary_keys":"account_id,reading_ts","dedupe_order_by":"_ingested_at DESC","bronze_table":"pay_txn_monitoring","silver_table":"txn_monitor","silver_write_mode":"insert_only","load_group":"telemetry","priority":30,"sla_hours":2,"data_owner":"Payments","sensitivity":"Internal","active":true,"description":"Account transaction monitoring (15-min windows) every 15 minutes, four micro-batch files a day. Insert-only and idempotent.","fabric_alternative":"Eventstream into Eventhouse for real time; keep this batch path for history and reconciliation"},{"object_id":"FS_SFDC_CUSTOMERS","industry":"fsi","source_system":"sfdc","source_object":"customers","source_type":"api","ingest_method":"landed_file","landing_format":"json","landing_path":"landing/sfdc/customers","read_options":"{\"multiline\": true, \"json_root\": \"records\"}","source_table":null,"load_type":"full","watermark_column":null,"primary_keys":"customer_id","dedupe_order_by":"last_modified_ts DESC, _ingested_at DESC","bronze_table":"sfdc_customers","silver_table":"customer","silver_write_mode":"merge_scd1","load_group":"reference","priority":15,"sla_hours":26,"data_owner":"Customer Experience","sensitivity":"Confidential","active":true,"description":"Customers - full daily snapshot from the CRM API (contains PII). Missing rows are soft-deleted.","fabric_alternative":"Copy job with the CRM connector; Dataverse-based CRMs can use Link to Fabric instead of copying"}],"column_map":[{"object_id":"FS_CORE_ACCOUNTS","column_order":0,"source_column":"account_id","target_column":"account_id","data_type":"int","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"FS_CORE_ACCOUNTS","column_order":1,"source_column":"account_number","target_column":"account_number","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"FS_CORE_ACCOUNTS","column_order":2,"source_column":"account_name","target_column":"account_name","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"FS_CORE_ACCOUNTS","column_order":3,"source_column":"product_type","target_column":"product_type","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"FS_CORE_ACCOUNTS","column_order":4,"source_column":"state","target_column":"state","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"FS_CORE_ACCOUNTS","column_order":5,"source_column":"status","target_column":"status","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"FS_CORE_ACCOUNTS","column_order":6,"source_column":"risk_grade","target_column":"risk_grade","data_type":"int","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"FS_CORE_ACCOUNTS","column_order":7,"source_column":"balance_aud","target_column":"balance_aud","data_type":"double","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"FS_CORE_ACCOUNTS","column_order":8,"source_column":"open_date","target_column":"open_date","data_type":"date","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"FS_CORE_ACCOUNTS","column_order":9,"source_column":"last_modified","target_column":"last_modified","data_type":"timestamp","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"FS_CORE_RISK_CASES","column_order":0,"source_column":"case_id","target_column":"case_id","data_type":"int","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"FS_CORE_RISK_CASES","column_order":1,"source_column":"account_id","target_column":"account_id","data_type":"int","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"FS_CORE_RISK_CASES","column_order":2,"source_column":"case_type","target_column":"case_type","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"FS_CORE_RISK_CASES","column_order":3,"source_column":"priority","target_column":"priority","data_type":"int","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"FS_CORE_RISK_CASES","column_order":4,"source_column":"status","target_column":"status","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"FS_CORE_RISK_CASES","column_order":5,"source_column":"raised_ts","target_column":"raised_ts","data_type":"timestamp","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"FS_CORE_RISK_CASES","column_order":6,"source_column":"closed_ts","target_column":"closed_ts","data_type":"timestamp","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"FS_CORE_RISK_CASES","column_order":7,"source_column":"case_amount_aud","target_column":"case_amount_aud","data_type":"decimal(12,2)","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"FS_CORE_RISK_CASES","column_order":8,"source_column":"analyst_hours","target_column":"analyst_hours","data_type":"double","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"FS_CORE_RISK_CASES","column_order":9,"source_column":"description","target_column":"description","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"FS_CORE_RISK_CASES","column_order":10,"source_column":"last_modified","target_column":"last_modified","data_type":"timestamp","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"FS_PAY_TXN_MONITORING","column_order":0,"source_column":"account_id","target_column":"account_id","data_type":"int","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"FS_PAY_TXN_MONITORING","column_order":1,"source_column":"reading_time","target_column":"reading_ts","data_type":"timestamp","transform_expr":"to_timestamp(reading_time, 'dd/MM/yyyy HH:mm')","pii_treatment":"none","sensitivity":"none","description":""},{"object_id":"FS_PAY_TXN_MONITORING","column_order":2,"source_column":"decline_pct","target_column":"decline_pct","data_type":"double","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"FS_PAY_TXN_MONITORING","column_order":3,"source_column":"txn_value_aud","target_column":"txn_value_aud","data_type":"double","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"FS_PAY_TXN_MONITORING","column_order":4,"source_column":"txn_count","target_column":"txn_count","data_type":"double","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"FS_SFDC_CUSTOMERS","column_order":0,"source_column":"Id","target_column":"customer_id","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"FS_SFDC_CUSTOMERS","column_order":1,"source_column":"FirstName","target_column":"first_name","data_type":"string","transform_expr":null,"pii_treatment":"drop","sensitivity":"Confidential","description":""},{"object_id":"FS_SFDC_CUSTOMERS","column_order":2,"source_column":"LastName","target_column":"last_name","data_type":"string","transform_expr":null,"pii_treatment":"drop","sensitivity":"Confidential","description":""},{"object_id":"FS_SFDC_CUSTOMERS","column_order":3,"source_column":"Email","target_column":"email_masked","data_type":"string","transform_expr":null,"pii_treatment":"mask_email","sensitivity":"Confidential","description":""},{"object_id":"FS_SFDC_CUSTOMERS","column_order":4,"source_column":"MobilePhone","target_column":"mobile_masked","data_type":"string","transform_expr":null,"pii_treatment":"mask_phone","sensitivity":"Confidential","description":""},{"object_id":"FS_SFDC_CUSTOMERS","column_order":5,"source_column":"Birthdate","target_column":"birth_year","data_type":"int","transform_expr":null,"pii_treatment":"year_only","sensitivity":"Confidential","description":""},{"object_id":"FS_SFDC_CUSTOMERS","column_order":6,"source_column":"MailingPostalCode","target_column":"postcode","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"FS_SFDC_CUSTOMERS","column_order":7,"source_column":"State__c","target_column":"state","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"FS_SFDC_CUSTOMERS","column_order":8,"source_column":"Segment__c","target_column":"segment","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"FS_SFDC_CUSTOMERS","column_order":9,"source_column":"AccountId__c","target_column":"account_id","data_type":"int","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"FS_SFDC_CUSTOMERS","column_order":10,"source_column":"Status__c","target_column":"status","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"FS_SFDC_CUSTOMERS","column_order":11,"source_column":"Tfn__c","target_column":"tfn","data_type":"string","transform_expr":null,"pii_treatment":"drop","sensitivity":"Restricted","description":""},{"object_id":"FS_SFDC_CUSTOMERS","column_order":12,"source_column":"LastModifiedDate","target_column":"last_modified_ts","data_type":"timestamp","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""}],"dq_rule":[{"rule_id":"FS-ENT-01","object_id":"FS_CORE_ACCOUNTS","rule_name":"Account id present","rule_category":"technical","check_type":"not_null","column_name":"account_id","params":"{}","severity":"fail","fail_threshold_pct":0,"rule_owner":"Data engineering","description":"Primary key must never be null","active":true},{"rule_id":"FS-ENT-02","object_id":"FS_CORE_ACCOUNTS","rule_name":"Known product_type","rule_category":"technical","check_type":"allowed_values","column_name":"product_type","params":"{\"values\": [\"TRANSACTION\", \"SAVINGS\", \"PERSONAL_LOAN\", \"HOME_LOAN\", \"CREDIT_CARD\"]}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Data engineering","description":"Unknown classes are kept but flagged for the data owner","active":true},{"rule_id":"FS-ENT-03","object_id":"FS_CORE_ACCOUNTS","rule_name":"risk_grade between 1 and 5","rule_category":"technical","check_type":"range","column_name":"risk_grade","params":"{\"min\": 1, \"max\": 5}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Data engineering","description":"Out-of-range ratings usually mean a keying error in the source","active":true},{"rule_id":"FS-ENT-04","object_id":"FS_CORE_ACCOUNTS","rule_name":"Known status","rule_category":"business","check_type":"allowed_values","column_name":"status","params":"{\"values\": [\"PERFORMING\", \"DORMANT\", \"ARREARS\", \"CLOSED\"]}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Retail Credit Risk","description":"Status list agreed with Retail Credit Risk","active":true},{"rule_id":"FS-EVT-01","object_id":"FS_CORE_RISK_CASES","rule_name":"Risk case id present","rule_category":"technical","check_type":"not_null","column_name":"case_id","params":"{}","severity":"fail","fail_threshold_pct":0,"rule_owner":"Data engineering","description":"Primary key must never be null","active":true},{"rule_id":"FS-EVT-02","object_id":"FS_CORE_RISK_CASES","rule_name":"Account exists","rule_category":"technical","check_type":"referential","column_name":"account_id","params":"{\"ref_table\": \"silver.account\", \"ref_column\": \"account_id\"}","severity":"quarantine","fail_threshold_pct":null,"rule_owner":"Data engineering","description":"Held until the account arrives (late-arriving reference data), then merged automatically","active":true},{"rule_id":"FS-EVT-03","object_id":"FS_CORE_RISK_CASES","rule_name":"Closed after raised","rule_category":"business","check_type":"expression","column_name":null,"params":"{\"expr\": \"closed_ts IS NULL OR closed_ts >= raised_ts\"}","severity":"quarantine","fail_threshold_pct":null,"rule_owner":"Fraud & Collections","description":"A risk case can't be closed before it was raised","active":true},{"rule_id":"FS-EVT-04","object_id":"FS_CORE_RISK_CASES","rule_name":"case_amount_aud not negative","rule_category":"technical","check_type":"range","column_name":"case_amount_aud","params":"{\"min\": 0}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Data engineering","description":"Negative values are sign errors in the source extract","active":true},{"rule_id":"FS-EVT-05","object_id":"FS_CORE_RISK_CASES","rule_name":"ACCOUNT_TAKEOVER is priority 1","rule_category":"business","check_type":"expression","column_name":null,"params":"{\"expr\": \"case_type <> 'ACCOUNT_TAKEOVER' OR priority = 1\"}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Fraud & Collections","description":"Account takeover risk cases must be raised as priority 1","active":true},{"rule_id":"FS-EVT-06","object_id":"FS_CORE_RISK_CASES","rule_name":"raised_ts parses","rule_category":"technical","check_type":"not_null","column_name":"raised_ts","params":"{}","severity":"fail","fail_threshold_pct":5,"rule_owner":"Data engineering","description":"If more than 5% of timestamps fail to parse, the extract format has changed - stop the load","active":true},{"rule_id":"FS-RDG-01","object_id":"FS_PAY_TXN_MONITORING","rule_name":"Declined transaction rate plausible","rule_category":"technical","check_type":"range","column_name":"decline_pct","params":"{\"min\": 0, \"max\": 100}","severity":"quarantine","fail_threshold_pct":null,"rule_owner":"Data engineering","description":"Sensor fault values (e.g. 999.9) are quarantined, never averaged into Gold","active":true},{"rule_id":"FS-RDG-02","object_id":"FS_PAY_TXN_MONITORING","rule_name":"Account known","rule_category":"technical","check_type":"referential","column_name":"account_id","params":"{\"ref_table\": \"silver.account\", \"ref_column\": \"account_id\"}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Data engineering","description":"Readings for unknown tags are flagged","active":true},{"rule_id":"FS-RDG-03","object_id":"FS_PAY_TXN_MONITORING","rule_name":"Reading time parses","rule_category":"technical","check_type":"not_null","column_name":"reading_ts","params":"{}","severity":"fail","fail_threshold_pct":5,"rule_owner":"Data engineering","description":"Timestamp format change stops the load","active":true},{"rule_id":"FS-PTY-01","object_id":"FS_SFDC_CUSTOMERS","rule_name":"Customer id present","rule_category":"technical","check_type":"not_null","column_name":"customer_id","params":"{}","severity":"fail","fail_threshold_pct":0,"rule_owner":"Data engineering","description":"Primary key must never be null","active":true},{"rule_id":"FS-PTY-02","object_id":"FS_SFDC_CUSTOMERS","rule_name":"Email looks valid","rule_category":"technical","check_type":"regex","column_name":"email_masked","params":"{\"pattern\": \"^[^@\\\\s]+@[^@\\\\s]+\\\\.[^@\\\\s]+$\"}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Data engineering","description":"Checked on the raw value before masking","active":true},{"rule_id":"FS-PTY-03","object_id":"FS_SFDC_CUSTOMERS","rule_name":"Australian postcode","rule_category":"technical","check_type":"regex","column_name":"postcode","params":"{\"pattern\": \"^\\\\d{4}$\"}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Data engineering","description":"Four digits","active":true},{"rule_id":"FS-PTY-04","object_id":"FS_SFDC_CUSTOMERS","rule_name":"Known segment","rule_category":"business","check_type":"allowed_values","column_name":"segment","params":"{\"values\": [\"Retail\", \"Premium\", \"Business\", \"Agribusiness\"]}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Customer Experience","description":"Segments agreed with Customer Experience","active":true}],"gold_transform":[{"transform_id":"FS-G01","industry":"fsi","gold_table":"dim_date","run_order":100,"sql_text":"SELECT CAST(date_format(d, 'yyyyMMdd') AS INT) AS date_key, d AS calendar_date, year(d) AS calendar_year, quarter(d) AS calendar_quarter, month(d) AS month_number, date_format(d, 'MMM') AS month_name, date_format(d, 'yyyy-MM') AS year_month, dayofmonth(d) AS day_of_month, date_format(d, 'EEE') AS day_name, CASE WHEN month(d) >= 7 THEN year(d) + 1 ELSE year(d) END AS fy_year, concat('FY', substring(CAST(CASE WHEN month(d) >= 7 THEN year(d) + 1 ELSE year(d) END AS STRING), 3, 2)) AS fy_label, CASE WHEN dayofweek(d) IN (1, 7) THEN true ELSE false END AS is_weekend FROM (SELECT explode(sequence(to_date('2020-01-01'), to_date('2028-12-31'), interval 1 day)) AS d) dates","description":"Calendar with Australian financial year","business_owner":"Data engineering","consumers":"Power BI / Direct Lake, data agent","active":true},{"transform_id":"FS-G02","industry":"fsi","gold_table":"dim_account","run_order":110,"sql_text":"SELECT account_id, account_number, account_name, product_type, state, status, risk_grade, balance_aud, open_date, CAST(floor(months_between(current_date(), open_date) / 12) AS INT) AS age_years, _valid_from AS version_valid_from FROM silver.account WHERE _is_current = true","description":"Current view of the account register (history stays in Silver SCD2)","business_owner":"Retail Credit Risk","consumers":"Power BI / Direct Lake, data agent","active":true},{"transform_id":"FS-G03","industry":"fsi","gold_table":"fact_risk_case","run_order":120,"sql_text":"SELECT v.case_id, v.account_id, v.case_type, v.priority, v.status, v.raised_ts, v.closed_ts, to_date(v.raised_ts) AS opened_date, CAST(date_format(v.raised_ts, 'yyyyMMdd') AS INT) AS opened_date_key, ROUND((unix_timestamp(v.closed_ts) - unix_timestamp(v.raised_ts)) / 3600.0, 2) AS hours_to_close, CASE WHEN v.status IN ('OPEN', 'INVESTIGATING') THEN 1 ELSE 0 END AS is_open, v.case_amount_aud, v.analyst_hours, d.state, d.product_type, d.risk_grade FROM silver.risk_case v LEFT JOIN silver.account d ON d.account_id = v.account_id AND d._is_current = true WHERE v._is_deleted = false","description":"Risk cases with duration and account attributes","business_owner":"Fraud & Collections","consumers":"Power BI / Direct Lake, data agent","active":true},{"transform_id":"FS-G04","industry":"fsi","gold_table":"fact_txn_monitor_hourly","run_order":130,"sql_text":"SELECT account_id, date_trunc('HOUR', reading_ts) AS reading_hour, to_date(reading_ts) AS reading_date, CAST(date_format(reading_ts, 'yyyyMMdd') AS INT) AS reading_date_key, COUNT(*) AS readings, ROUND(AVG(decline_pct), 2) AS avg_decline_pct, ROUND(MAX(decline_pct), 2) AS max_decline_pct, ROUND(AVG(txn_value_aud), 2) AS avg_txn_value_aud, ROUND(AVG(txn_count), 2) AS avg_txn_count, SUM(CASE WHEN decline_pct > 30 THEN 1 ELSE 0 END) AS breach_readings FROM silver.txn_monitor GROUP BY account_id, date_trunc('HOUR', reading_ts), to_date(reading_ts), CAST(date_format(reading_ts, 'yyyyMMdd') AS INT)","description":"Hourly account transaction monitoring (15-min windows) with threshold breaches","business_owner":"Payments","consumers":"Power BI / Direct Lake, data agent","active":true},{"transform_id":"FS-G05","industry":"fsi","gold_table":"account_health_scorecard","run_order":200,"sql_text":"WITH asat AS (SELECT MAX(to_date(raised_ts)) AS as_at FROM silver.risk_case), ev AS (SELECT v.account_id, COUNT(*) AS risk_cases_12m, SUM(CASE WHEN v.status IN ('OPEN', 'INVESTIGATING') THEN 1 ELSE 0 END) AS open_risk_cases, SUM(CASE WHEN v.case_type = 'ACCOUNT_TAKEOVER' THEN 1 ELSE 0 END) AS urgent_risk_cases_12m, ROUND(SUM(v.case_amount_aud), 2) AS case_amount_aud_12m FROM silver.risk_case v CROSS JOIN asat WHERE v._is_deleted = false AND to_date(v.raised_ts) > add_months(asat.as_at, -12) GROUP BY v.account_id), rd AS (SELECT account_id, MAX(decline_pct) AS max_decline_pct, SUM(CASE WHEN decline_pct > 30 THEN 1 ELSE 0 END) AS breach_readings FROM silver.txn_monitor GROUP BY account_id), s AS (SELECT d.account_id, d.account_number, d.account_name, d.product_type, d.state, d.status, d.risk_grade, COALESCE(ev.risk_cases_12m, 0) AS risk_cases_12m, COALESCE(ev.open_risk_cases, 0) AS open_risk_cases, COALESCE(ev.urgent_risk_cases_12m, 0) AS urgent_risk_cases_12m, COALESCE(ev.case_amount_aud_12m, 0) AS case_amount_aud_12m, rd.max_decline_pct, COALESCE(rd.breach_readings, 0) AS breach_readings, d.balance_aud FROM silver.account d LEFT JOIN ev ON ev.account_id = d.account_id LEFT JOIN rd ON rd.account_id = d.account_id WHERE d._is_current = true) SELECT s.*, ROUND(LEAST(100, COALESCE(s.risk_grade, 1) * 8 + s.open_risk_cases * 6 + s.urgent_risk_cases_12m * 4 + LEAST(s.breach_readings, 20) * 1.5), 1) AS risk_score, CASE WHEN LEAST(100, COALESCE(s.risk_grade, 1) * 8 + s.open_risk_cases * 6 + s.urgent_risk_cases_12m * 4 + LEAST(s.breach_readings, 20) * 1.5) >= 60 THEN 'High' WHEN LEAST(100, COALESCE(s.risk_grade, 1) * 8 + s.open_risk_cases * 6 + s.urgent_risk_cases_12m * 4 + LEAST(s.breach_readings, 20) * 1.5) >= 35 THEN 'Medium' ELSE 'Low' END AS risk_band, ROUND(s.open_risk_cases * 3500 + LEAST(s.breach_readings, 20) * 250 + COALESCE(s.balance_aud, 0) * COALESCE(s.risk_grade, 1) * 0.002, 0) AS expected_loss_aud FROM s","description":"Account risk score - risk cases, urgent work and declined transaction rate breaches, plus expected loss in AUD (weights and dollar rates are illustrative: $3,500 per open case + $250 per suspicious 15-min window + balance x risk grade x 0.2%)","business_owner":"Retail Credit Risk","consumers":"Power BI / Direct Lake, data agent","active":true},{"transform_id":"FS-G06","industry":"fsi","gold_table":"customer_summary","run_order":210,"sql_text":"SELECT state, segment, COUNT(*) AS customers, SUM(CASE WHEN status = 'Active' THEN 1 ELSE 0 END) AS active_customers, ROUND(AVG(year(current_date()) - birth_year), 1) AS avg_age, COUNT(DISTINCT account_id) AS linked_accounts FROM silver.customer WHERE _is_deleted = false GROUP BY state, segment","description":"Customers by state and segment","business_owner":"Customer Experience","consumers":"Power BI / Direct Lake, data agent","active":true}],"onboard":{"object":{"object_id":"FS_PAY_SCAM_REPORTS","industry":"fsi","source_system":"pay","source_object":"scam_reports","source_type":"file","ingest_method":"landed_file","landing_format":"txt","landing_path":"landing/pay/scam_reports","read_options":"{\"header\": true, \"delimiter\": \"|\"}","source_table":null,"load_type":"incremental","watermark_column":null,"primary_keys":"state,obs_date","dedupe_order_by":"_ingested_at DESC","bronze_table":"pay_scam_reports","silver_table":"scam_report","silver_write_mode":"merge_scd1","load_group":"reference","priority":25,"sla_hours":26,"data_owner":"Financial Crime","sensitivity":"Public","active":true,"description":"Daily Scamwatch reports by state (pipe-delimited, dd/MM/yyyy) - onboarded live in the demo","fabric_alternative":"Public or partner feed - could be a REST copy, or a shortcut if the provider shares a folder"},"columns":[["state","state","string",null],["obs_date","obs_date","date","to_date(obs_date, 'dd/MM/yyyy')"],["reports","reports","double",null],["reported_loss_aud","reported_loss_aud","double",null],["phishing_share_pct","phishing_share_pct","double",null],["top_scam_type","top_scam_type","string",null]],"rules":[["FS-EXT-01","reports plausible","technical","range","reports","{\"min\": 0, \"max\": 5000}","warn"],["FS-EXT-02","Known top_scam_type","business","allowed_values","top_scam_type","{\"values\": [\"Phishing\", \"Investment\", \"Romance\", \"Remote access\"]}","warn"]],"gold":{"transform_id":"FS-G07","industry":"fsi","gold_table":"risk_case_vs_scam_report","run_order":220,"sql_text":"WITH ev AS (SELECT d.state, to_date(v.raised_ts) AS obs_date, COUNT(*) AS risk_cases, SUM(CASE WHEN v.case_type = 'ACCOUNT_TAKEOVER' THEN 1 ELSE 0 END) AS urgent_risk_cases FROM silver.risk_case v JOIN silver.account d ON d.account_id = v.account_id AND d._is_current = true WHERE v._is_deleted = false GROUP BY d.state, to_date(v.raised_ts)) SELECT x.state, x.obs_date, x.reports, x.reported_loss_aud, x.phishing_share_pct, x.top_scam_type, COALESCE(ev.risk_cases, 0) AS risk_cases, COALESCE(ev.urgent_risk_cases, 0) AS urgent_risk_cases FROM silver.scam_report x LEFT JOIN ev ON ev.state = x.state AND ev.obs_date = x.obs_date WHERE x._is_deleted = false","description":"Public scam reports against risk cases by state","business_owner":"Financial Crime","consumers":"Power BI / Direct Lake","active":true}}},"manufacturing":{"pack":{"id":"manufacturing","industry":"Manufacturing","org":"Kestrel Components","short":"KCO","prefix":"MF","tagline":"automotive and agricultural components manufacturer","email_domain":"kestrel.example.com.au","region_col":"plant","region_label":"plant","regions":["Geelong","Dandenong","Wodonga","Ballarat","Bendigo","Albury"],"name_words":["Line A","Line B","Line C","Cell 1","Cell 2","Cell 3","Bay 4","Bay 5","Press Shop","Paint Shop"],"sys":{"ops":{"code":"mes","name":"Manufacturing Execution System (on-prem SQL Server)","type":"Database","owner":"Plant IT","fabric":"Mirroring for SQL Server vs Copy job incremental vs pipeline Copy through the gateway"},"tel":{"code":"iot","name":"Machine IoT historian (15-min CSV) and energy price feed","type":"OT file drop","owner":"Manufacturing Engineering","fabric":"Eventstream from IoT Hub / MQTT for real time; files to OneLake for batch"},"crm":{"code":"d365","name":"Dynamics 365 Sales (distributor contacts)","type":"SaaS CRM","owner":"Sales Operations","fabric":"Link to Microsoft Fabric (Dataverse) or Copy job"}},"entity":{"table":"machine","src":"machines","label":"machine","plural":"machines","id":"machine_id","code":"machine_code","code_prefix":"MC","name_col":"machine_name","type_col":"machine_type","types":["CNC_MILL","INJECTION_MOULDER","STAMPING_PRESS","ROBOT_CELL","CONVEYOR"],"status":["RUNNING","IDLE","MAINTENANCE","RETIRED"],"tier_col":"criticality","tier_label":"criticality (1-5, 5 = bottleneck machine)","cap_col":"rated_units_per_hour","cap_label":"rated output (units/hour)","cap_range":[20,1200],"start_col":"commissioned_date","start_years":[1998,2025],"drift_col":"line_supervisor","drift_values":["Day shift","Afternoon shift","Night shift"],"owner":"Manufacturing Engineering","count":240},"event":{"table":"maintenance_order","src":"maintenance_orders","label":"maintenance order","plural":"maintenance orders","id":"mo_id","id_start":300001,"type_col":"mo_type","types":["BREAKDOWN","PREVENTIVE","CALIBRATION","CHANGEOVER"],"type_names":["Breakdown","Preventive","Calibration","Changeover"],"urgent_type":"BREAKDOWN","status":["OPEN","IN_PROGRESS","DONE","CANCELLED"],"open_ts":"reported_ts","close_ts":"finished_ts","open_verb":"reported","close_verb":"finished","amount_col":"parts_cost_aud","amount_range":[0,40000],"qty_col":"downtime_hours","qty_range":[0.25,72],"descriptions":["Spindle bearing noise","Hydraulic leak on ram","Mould temperature drift","Gripper misalignment","Belt slipping","Tool change overdue","Servo overload alarm","Coolant contamination"],"owner":"Maintenance"},"reading":{"table":"machine_telemetry","src":"machine_telemetry","label":"machine telemetry","fact":"machine_telemetry","monitored":24,"monitored_types":["CNC_MILL","INJECTION_MOULDER","STAMPING_PRESS"],"metrics":[{"col":"vibration_mm_s","label":"vibration","unit":"mm/s","base":3.2,"sd":1.0,"min":0,"max":50,"threshold":7.1},{"col":"units_produced","label":"units produced","unit":"units","base":110,"sd":30,"min":0,"max":5000},{"col":"motor_temp_c","label":"motor temperature","unit":"°C","base":58,"sd":7,"min":-10,"max":160}],"sentinel":999.9,"owner":"Manufacturing Engineering"},"party":{"table":"distributor_contact","src":"distributor_contacts","label":"distributor contact","plural":"distributor contacts","id":"contact_id","id_prefix":"DC","sf_type":"contact","segment_col":"channel","segments":["Distributor","OEM","Retail chain","Export"],"status":["Active","Inactive"],"sensitive_col":"driver_licence_no","sensitive_label":"driver licence number","sensitive_treatment":"drop","owner":"Sales Operations","count":300},"extra":{"table":"energy_price","src":"energy_prices","label":"wholesale energy prices","sys":"tel","cols":[{"col":"avg_price_mwh","base":95,"sd":30,"min":-1000,"max":17500},{"col":"peak_price_mwh","base":210,"sd":90,"min":-1000,"max":17500},{"col":"max_temp_c","base":26,"sd":6,"min":-10,"max":50}],"cat_col":"price_band","cat_values":["Low","Normal","High","Spike"],"owner":"Operations Finance","story":"energy price spikes against breakdowns by plant"},"alert":{"stream":"machine_alarm","label":"machine alarm","codes":["OVER_TEMP","E_STOP","JAM"],"storm":"line stoppage"},"ref":{"title":"Maintenance response targets","owner":"Maintenance","rows":[["BREAKDOWN","Unplanned",4,"Maintenance Rapid Response"],["PREVENTIVE","Planned",168,"Maintenance Planning"],["CALIBRATION","Quality",72,"Quality Assurance"],["CHANGEOVER","Planned",8,"Production"]]},"ai":{"persona":"plant operations and maintenance","cust":"discrete and process manufacturers, including food and beverage","q_model":["Which plant had the most breakdown hours this month?","Show parts cost by machine type and month.","Which bottleneck machines (criticality 4 or 5) have open maintenance orders?"],"q_agent":["Which 10 machines have the highest risk score and how many open maintenance orders does each have?","Compare average hours to finish breakdowns by plant.","Which machines exceeded 7.1 mm/s vibration in the last 3 days?","What was the total downtime caused by breakdowns in Geelong?","How many OEM contacts are active per plant?"],"q_onto":["Show injection moulders in Dandenong with an open breakdown and a high risk band.","For the machine with the highest vibration, list its maintenance orders and their parts cost.","Which distributor contacts are linked to machines currently in maintenance?"]},"impact":{"col":"downtime_cost_at_risk_aud","label":"downtime cost at risk","open_cost":5000,"breach_cost":3000,"cap_rate":2,"rule":"$5,000 per open maintenance order + $3,000 per 15-min window above the vibration limit + rated units/hour x criticality x $2"}},"source_system":[{"source_system":"mes","industry":"manufacturing","display_name":"Manufacturing Execution System (on-prem SQL Server)","system_type":"Database","connection_pattern":"On-premises data gateway -> pipeline Copy with watermark query","technical_owner":"Plant IT","fabric_options":"Mirroring for SQL Server vs Copy job incremental vs pipeline Copy through the gateway"},{"source_system":"iot","industry":"manufacturing","display_name":"Machine IoT historian (15-min CSV) and energy price feed","system_type":"OT file drop","connection_pattern":"Files dropped to OneLake (Files/landing) every 15 minutes","technical_owner":"Manufacturing Engineering","fabric_options":"Eventstream from IoT Hub / MQTT for real time; files to OneLake for batch"},{"source_system":"d365","industry":"manufacturing","display_name":"Dynamics 365 Sales (distributor contacts)","system_type":"SaaS CRM","connection_pattern":"REST/OData API -> JSON landed per day","technical_owner":"Sales Operations","fabric_options":"Link to Microsoft Fabric (Dataverse) or Copy job"}],"ingestion_object":[{"object_id":"MF_MES_MACHINES","industry":"manufacturing","source_system":"mes","source_object":"machines","source_type":"database","ingest_method":"copy_db","landing_format":"csv","landing_path":"landing/mes/machines","read_options":"{\"header\": true, \"delimiter\": \",\"}","source_table":"src.machines","load_type":"incremental","watermark_column":"last_modified","primary_keys":"machine_id","dedupe_order_by":"last_modified DESC, _ingested_at DESC","bronze_table":"mes_machines","silver_table":"machine","silver_write_mode":"merge_scd2","load_group":"reference","priority":10,"sla_hours":26,"data_owner":"Manufacturing Engineering","sensitivity":"Internal","active":true,"description":"Machine register. SCD2 keeps status, criticality and rated_units_per_hour history.","fabric_alternative":"Strong mirroring candidate - small, slowly changing, needed in near real time by operations"},{"object_id":"MF_MES_MAINTENANCE_ORDERS","industry":"manufacturing","source_system":"mes","source_object":"maintenance_orders","source_type":"database","ingest_method":"copy_db","landing_format":"csv","landing_path":"landing/mes/maintenance_orders","read_options":"{\"header\": true, \"delimiter\": \",\"}","source_table":"src.maintenance_orders","load_type":"incremental","watermark_column":"last_modified","primary_keys":"mo_id","dedupe_order_by":"last_modified DESC, _ingested_at DESC","bronze_table":"mes_maintenance_orders","silver_table":"maintenance_order","silver_write_mode":"merge_scd1","load_group":"transactional","priority":20,"sla_hours":26,"data_owner":"Maintenance","sensitivity":"Internal","active":true,"description":"Maintenance orders - incremental by last_modified; can arrive before their machine (late-arriving).","fabric_alternative":"Copy job incremental or mirroring; keep the watermark query if the source is on-prem"},{"object_id":"MF_IOT_MACHINE_TELEMETRY","industry":"manufacturing","source_system":"iot","source_object":"machine_telemetry","source_type":"file","ingest_method":"landed_file","landing_format":"csv","landing_path":"landing/iot/machine_telemetry","read_options":"{\"header\": true, \"delimiter\": \",\"}","source_table":null,"load_type":"incremental","watermark_column":null,"primary_keys":"machine_id,reading_ts","dedupe_order_by":"_ingested_at DESC","bronze_table":"iot_machine_telemetry","silver_table":"machine_telemetry","silver_write_mode":"insert_only","load_group":"telemetry","priority":30,"sla_hours":2,"data_owner":"Manufacturing Engineering","sensitivity":"Internal","active":true,"description":"Machine telemetry every 15 minutes, four micro-batch files a day. Insert-only and idempotent.","fabric_alternative":"Eventstream into Eventhouse for real time; keep this batch path for history and reconciliation"},{"object_id":"MF_D365_DISTRIBUTOR_CONTACTS","industry":"manufacturing","source_system":"d365","source_object":"distributor_contacts","source_type":"api","ingest_method":"landed_file","landing_format":"json","landing_path":"landing/d365/distributor_contacts","read_options":"{\"multiline\": true, \"json_root\": \"records\"}","source_table":null,"load_type":"full","watermark_column":null,"primary_keys":"contact_id","dedupe_order_by":"last_modified_ts DESC, _ingested_at DESC","bronze_table":"d365_distributor_contacts","silver_table":"distributor_contact","silver_write_mode":"merge_scd1","load_group":"reference","priority":15,"sla_hours":26,"data_owner":"Sales Operations","sensitivity":"Confidential","active":true,"description":"Distributor contacts - full daily snapshot from the CRM API (contains PII). Missing rows are soft-deleted.","fabric_alternative":"Copy job with the CRM connector; Dataverse-based CRMs can use Link to Fabric instead of copying"}],"column_map":[{"object_id":"MF_MES_MACHINES","column_order":0,"source_column":"machine_id","target_column":"machine_id","data_type":"int","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"MF_MES_MACHINES","column_order":1,"source_column":"machine_code","target_column":"machine_code","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"MF_MES_MACHINES","column_order":2,"source_column":"machine_name","target_column":"machine_name","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"MF_MES_MACHINES","column_order":3,"source_column":"machine_type","target_column":"machine_type","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"MF_MES_MACHINES","column_order":4,"source_column":"plant","target_column":"plant","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"MF_MES_MACHINES","column_order":5,"source_column":"status","target_column":"status","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"MF_MES_MACHINES","column_order":6,"source_column":"criticality","target_column":"criticality","data_type":"int","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"MF_MES_MACHINES","column_order":7,"source_column":"rated_units_per_hour","target_column":"rated_units_per_hour","data_type":"double","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"MF_MES_MACHINES","column_order":8,"source_column":"commissioned_date","target_column":"commissioned_date","data_type":"date","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"MF_MES_MACHINES","column_order":9,"source_column":"last_modified","target_column":"last_modified","data_type":"timestamp","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"MF_MES_MAINTENANCE_ORDERS","column_order":0,"source_column":"mo_id","target_column":"mo_id","data_type":"int","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"MF_MES_MAINTENANCE_ORDERS","column_order":1,"source_column":"machine_id","target_column":"machine_id","data_type":"int","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"MF_MES_MAINTENANCE_ORDERS","column_order":2,"source_column":"mo_type","target_column":"mo_type","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"MF_MES_MAINTENANCE_ORDERS","column_order":3,"source_column":"priority","target_column":"priority","data_type":"int","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"MF_MES_MAINTENANCE_ORDERS","column_order":4,"source_column":"status","target_column":"status","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"MF_MES_MAINTENANCE_ORDERS","column_order":5,"source_column":"reported_ts","target_column":"reported_ts","data_type":"timestamp","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"MF_MES_MAINTENANCE_ORDERS","column_order":6,"source_column":"finished_ts","target_column":"finished_ts","data_type":"timestamp","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"MF_MES_MAINTENANCE_ORDERS","column_order":7,"source_column":"parts_cost_aud","target_column":"parts_cost_aud","data_type":"decimal(12,2)","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"MF_MES_MAINTENANCE_ORDERS","column_order":8,"source_column":"downtime_hours","target_column":"downtime_hours","data_type":"double","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"MF_MES_MAINTENANCE_ORDERS","column_order":9,"source_column":"description","target_column":"description","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"MF_MES_MAINTENANCE_ORDERS","column_order":10,"source_column":"last_modified","target_column":"last_modified","data_type":"timestamp","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"MF_IOT_MACHINE_TELEMETRY","column_order":0,"source_column":"machine_id","target_column":"machine_id","data_type":"int","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"MF_IOT_MACHINE_TELEMETRY","column_order":1,"source_column":"reading_time","target_column":"reading_ts","data_type":"timestamp","transform_expr":"to_timestamp(reading_time, 'dd/MM/yyyy HH:mm')","pii_treatment":"none","sensitivity":"none","description":""},{"object_id":"MF_IOT_MACHINE_TELEMETRY","column_order":2,"source_column":"vibration_mm_s","target_column":"vibration_mm_s","data_type":"double","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"MF_IOT_MACHINE_TELEMETRY","column_order":3,"source_column":"units_produced","target_column":"units_produced","data_type":"double","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"MF_IOT_MACHINE_TELEMETRY","column_order":4,"source_column":"motor_temp_c","target_column":"motor_temp_c","data_type":"double","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"MF_D365_DISTRIBUTOR_CONTACTS","column_order":0,"source_column":"Id","target_column":"contact_id","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"MF_D365_DISTRIBUTOR_CONTACTS","column_order":1,"source_column":"FirstName","target_column":"first_name","data_type":"string","transform_expr":null,"pii_treatment":"drop","sensitivity":"Confidential","description":""},{"object_id":"MF_D365_DISTRIBUTOR_CONTACTS","column_order":2,"source_column":"LastName","target_column":"last_name","data_type":"string","transform_expr":null,"pii_treatment":"drop","sensitivity":"Confidential","description":""},{"object_id":"MF_D365_DISTRIBUTOR_CONTACTS","column_order":3,"source_column":"Email","target_column":"email_masked","data_type":"string","transform_expr":null,"pii_treatment":"mask_email","sensitivity":"Confidential","description":""},{"object_id":"MF_D365_DISTRIBUTOR_CONTACTS","column_order":4,"source_column":"MobilePhone","target_column":"mobile_masked","data_type":"string","transform_expr":null,"pii_treatment":"mask_phone","sensitivity":"Confidential","description":""},{"object_id":"MF_D365_DISTRIBUTOR_CONTACTS","column_order":5,"source_column":"Birthdate","target_column":"birth_year","data_type":"int","transform_expr":null,"pii_treatment":"year_only","sensitivity":"Confidential","description":""},{"object_id":"MF_D365_DISTRIBUTOR_CONTACTS","column_order":6,"source_column":"MailingPostalCode","target_column":"postcode","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"MF_D365_DISTRIBUTOR_CONTACTS","column_order":7,"source_column":"Plant__c","target_column":"plant","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"MF_D365_DISTRIBUTOR_CONTACTS","column_order":8,"source_column":"Channel__c","target_column":"channel","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"MF_D365_DISTRIBUTOR_CONTACTS","column_order":9,"source_column":"MachineId__c","target_column":"machine_id","data_type":"int","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"MF_D365_DISTRIBUTOR_CONTACTS","column_order":10,"source_column":"Status__c","target_column":"status","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"MF_D365_DISTRIBUTOR_CONTACTS","column_order":11,"source_column":"DriverLicenceNo__c","target_column":"driver_licence_no","data_type":"string","transform_expr":null,"pii_treatment":"drop","sensitivity":"Restricted","description":""},{"object_id":"MF_D365_DISTRIBUTOR_CONTACTS","column_order":12,"source_column":"LastModifiedDate","target_column":"last_modified_ts","data_type":"timestamp","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""}],"dq_rule":[{"rule_id":"MF-ENT-01","object_id":"MF_MES_MACHINES","rule_name":"Machine id present","rule_category":"technical","check_type":"not_null","column_name":"machine_id","params":"{}","severity":"fail","fail_threshold_pct":0,"rule_owner":"Data engineering","description":"Primary key must never be null","active":true},{"rule_id":"MF-ENT-02","object_id":"MF_MES_MACHINES","rule_name":"Known machine_type","rule_category":"technical","check_type":"allowed_values","column_name":"machine_type","params":"{\"values\": [\"CNC_MILL\", \"INJECTION_MOULDER\", \"STAMPING_PRESS\", \"ROBOT_CELL\", \"CONVEYOR\"]}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Data engineering","description":"Unknown classes are kept but flagged for the data owner","active":true},{"rule_id":"MF-ENT-03","object_id":"MF_MES_MACHINES","rule_name":"criticality between 1 and 5","rule_category":"technical","check_type":"range","column_name":"criticality","params":"{\"min\": 1, \"max\": 5}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Data engineering","description":"Out-of-range ratings usually mean a keying error in the source","active":true},{"rule_id":"MF-ENT-04","object_id":"MF_MES_MACHINES","rule_name":"Known status","rule_category":"business","check_type":"allowed_values","column_name":"status","params":"{\"values\": [\"RUNNING\", \"IDLE\", \"MAINTENANCE\", \"RETIRED\"]}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Manufacturing Engineering","description":"Status list agreed with Manufacturing Engineering","active":true},{"rule_id":"MF-EVT-01","object_id":"MF_MES_MAINTENANCE_ORDERS","rule_name":"Maintenance order id present","rule_category":"technical","check_type":"not_null","column_name":"mo_id","params":"{}","severity":"fail","fail_threshold_pct":0,"rule_owner":"Data engineering","description":"Primary key must never be null","active":true},{"rule_id":"MF-EVT-02","object_id":"MF_MES_MAINTENANCE_ORDERS","rule_name":"Machine exists","rule_category":"technical","check_type":"referential","column_name":"machine_id","params":"{\"ref_table\": \"silver.machine\", \"ref_column\": \"machine_id\"}","severity":"quarantine","fail_threshold_pct":null,"rule_owner":"Data engineering","description":"Held until the machine arrives (late-arriving reference data), then merged automatically","active":true},{"rule_id":"MF-EVT-03","object_id":"MF_MES_MAINTENANCE_ORDERS","rule_name":"Finished after reported","rule_category":"business","check_type":"expression","column_name":null,"params":"{\"expr\": \"finished_ts IS NULL OR finished_ts >= reported_ts\"}","severity":"quarantine","fail_threshold_pct":null,"rule_owner":"Maintenance","description":"A maintenance order can't be finished before it was reported","active":true},{"rule_id":"MF-EVT-04","object_id":"MF_MES_MAINTENANCE_ORDERS","rule_name":"parts_cost_aud not negative","rule_category":"technical","check_type":"range","column_name":"parts_cost_aud","params":"{\"min\": 0}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Data engineering","description":"Negative values are sign errors in the source extract","active":true},{"rule_id":"MF-EVT-05","object_id":"MF_MES_MAINTENANCE_ORDERS","rule_name":"BREAKDOWN is priority 1","rule_category":"business","check_type":"expression","column_name":null,"params":"{\"expr\": \"mo_type <> 'BREAKDOWN' OR priority = 1\"}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Maintenance","description":"Breakdown maintenance orders must be raised as priority 1","active":true},{"rule_id":"MF-EVT-06","object_id":"MF_MES_MAINTENANCE_ORDERS","rule_name":"reported_ts parses","rule_category":"technical","check_type":"not_null","column_name":"reported_ts","params":"{}","severity":"fail","fail_threshold_pct":5,"rule_owner":"Data engineering","description":"If more than 5% of timestamps fail to parse, the extract format has changed - stop the load","active":true},{"rule_id":"MF-RDG-01","object_id":"MF_IOT_MACHINE_TELEMETRY","rule_name":"Vibration plausible","rule_category":"technical","check_type":"range","column_name":"vibration_mm_s","params":"{\"min\": 0, \"max\": 50}","severity":"quarantine","fail_threshold_pct":null,"rule_owner":"Data engineering","description":"Sensor fault values (e.g. 999.9) are quarantined, never averaged into Gold","active":true},{"rule_id":"MF-RDG-02","object_id":"MF_IOT_MACHINE_TELEMETRY","rule_name":"Machine known","rule_category":"technical","check_type":"referential","column_name":"machine_id","params":"{\"ref_table\": \"silver.machine\", \"ref_column\": \"machine_id\"}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Data engineering","description":"Readings for unknown tags are flagged","active":true},{"rule_id":"MF-RDG-03","object_id":"MF_IOT_MACHINE_TELEMETRY","rule_name":"Reading time parses","rule_category":"technical","check_type":"not_null","column_name":"reading_ts","params":"{}","severity":"fail","fail_threshold_pct":5,"rule_owner":"Data engineering","description":"Timestamp format change stops the load","active":true},{"rule_id":"MF-PTY-01","object_id":"MF_D365_DISTRIBUTOR_CONTACTS","rule_name":"Distributor contact id present","rule_category":"technical","check_type":"not_null","column_name":"contact_id","params":"{}","severity":"fail","fail_threshold_pct":0,"rule_owner":"Data engineering","description":"Primary key must never be null","active":true},{"rule_id":"MF-PTY-02","object_id":"MF_D365_DISTRIBUTOR_CONTACTS","rule_name":"Email looks valid","rule_category":"technical","check_type":"regex","column_name":"email_masked","params":"{\"pattern\": \"^[^@\\\\s]+@[^@\\\\s]+\\\\.[^@\\\\s]+$\"}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Data engineering","description":"Checked on the raw value before masking","active":true},{"rule_id":"MF-PTY-03","object_id":"MF_D365_DISTRIBUTOR_CONTACTS","rule_name":"Australian postcode","rule_category":"technical","check_type":"regex","column_name":"postcode","params":"{\"pattern\": \"^\\\\d{4}$\"}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Data engineering","description":"Four digits","active":true},{"rule_id":"MF-PTY-04","object_id":"MF_D365_DISTRIBUTOR_CONTACTS","rule_name":"Known channel","rule_category":"business","check_type":"allowed_values","column_name":"channel","params":"{\"values\": [\"Distributor\", \"OEM\", \"Retail chain\", \"Export\"]}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Sales Operations","description":"Segments agreed with Sales Operations","active":true}],"gold_transform":[{"transform_id":"MF-G01","industry":"manufacturing","gold_table":"dim_date","run_order":100,"sql_text":"SELECT CAST(date_format(d, 'yyyyMMdd') AS INT) AS date_key, d AS calendar_date, year(d) AS calendar_year, quarter(d) AS calendar_quarter, month(d) AS month_number, date_format(d, 'MMM') AS month_name, date_format(d, 'yyyy-MM') AS year_month, dayofmonth(d) AS day_of_month, date_format(d, 'EEE') AS day_name, CASE WHEN month(d) >= 7 THEN year(d) + 1 ELSE year(d) END AS fy_year, concat('FY', substring(CAST(CASE WHEN month(d) >= 7 THEN year(d) + 1 ELSE year(d) END AS STRING), 3, 2)) AS fy_label, CASE WHEN dayofweek(d) IN (1, 7) THEN true ELSE false END AS is_weekend FROM (SELECT explode(sequence(to_date('2020-01-01'), to_date('2028-12-31'), interval 1 day)) AS d) dates","description":"Calendar with Australian financial year","business_owner":"Data engineering","consumers":"Power BI / Direct Lake, data agent","active":true},{"transform_id":"MF-G02","industry":"manufacturing","gold_table":"dim_machine","run_order":110,"sql_text":"SELECT machine_id, machine_code, machine_name, machine_type, plant, status, criticality, rated_units_per_hour, commissioned_date, CAST(floor(months_between(current_date(), commissioned_date) / 12) AS INT) AS age_years, _valid_from AS version_valid_from FROM silver.machine WHERE _is_current = true","description":"Current view of the machine register (history stays in Silver SCD2)","business_owner":"Manufacturing Engineering","consumers":"Power BI / Direct Lake, data agent","active":true},{"transform_id":"MF-G03","industry":"manufacturing","gold_table":"fact_maintenance_order","run_order":120,"sql_text":"SELECT v.mo_id, v.machine_id, v.mo_type, v.priority, v.status, v.reported_ts, v.finished_ts, to_date(v.reported_ts) AS opened_date, CAST(date_format(v.reported_ts, 'yyyyMMdd') AS INT) AS opened_date_key, ROUND((unix_timestamp(v.finished_ts) - unix_timestamp(v.reported_ts)) / 3600.0, 2) AS hours_to_close, CASE WHEN v.status IN ('OPEN', 'IN_PROGRESS') THEN 1 ELSE 0 END AS is_open, v.parts_cost_aud, v.downtime_hours, d.plant, d.machine_type, d.criticality FROM silver.maintenance_order v LEFT JOIN silver.machine d ON d.machine_id = v.machine_id AND d._is_current = true WHERE v._is_deleted = false","description":"Maintenance orders with duration and machine attributes","business_owner":"Maintenance","consumers":"Power BI / Direct Lake, data agent","active":true},{"transform_id":"MF-G04","industry":"manufacturing","gold_table":"fact_machine_telemetry_hourly","run_order":130,"sql_text":"SELECT machine_id, date_trunc('HOUR', reading_ts) AS reading_hour, to_date(reading_ts) AS reading_date, CAST(date_format(reading_ts, 'yyyyMMdd') AS INT) AS reading_date_key, COUNT(*) AS readings, ROUND(AVG(vibration_mm_s), 2) AS avg_vibration_mm_s, ROUND(MAX(vibration_mm_s), 2) AS max_vibration_mm_s, ROUND(AVG(units_produced), 2) AS avg_units_produced, ROUND(AVG(motor_temp_c), 2) AS avg_motor_temp_c, SUM(CASE WHEN vibration_mm_s > 7.1 THEN 1 ELSE 0 END) AS breach_readings FROM silver.machine_telemetry GROUP BY machine_id, date_trunc('HOUR', reading_ts), to_date(reading_ts), CAST(date_format(reading_ts, 'yyyyMMdd') AS INT)","description":"Hourly machine telemetry with threshold breaches","business_owner":"Manufacturing Engineering","consumers":"Power BI / Direct Lake, data agent","active":true},{"transform_id":"MF-G05","industry":"manufacturing","gold_table":"machine_health_scorecard","run_order":200,"sql_text":"WITH asat AS (SELECT MAX(to_date(reported_ts)) AS as_at FROM silver.maintenance_order), ev AS (SELECT v.machine_id, COUNT(*) AS maintenance_orders_12m, SUM(CASE WHEN v.status IN ('OPEN', 'IN_PROGRESS') THEN 1 ELSE 0 END) AS open_maintenance_orders, SUM(CASE WHEN v.mo_type = 'BREAKDOWN' THEN 1 ELSE 0 END) AS urgent_maintenance_orders_12m, ROUND(SUM(v.parts_cost_aud), 2) AS parts_cost_aud_12m FROM silver.maintenance_order v CROSS JOIN asat WHERE v._is_deleted = false AND to_date(v.reported_ts) > add_months(asat.as_at, -12) GROUP BY v.machine_id), rd AS (SELECT machine_id, MAX(vibration_mm_s) AS max_vibration_mm_s, SUM(CASE WHEN vibration_mm_s > 7.1 THEN 1 ELSE 0 END) AS breach_readings FROM silver.machine_telemetry GROUP BY machine_id), s AS (SELECT d.machine_id, d.machine_code, d.machine_name, d.machine_type, d.plant, d.status, d.criticality, COALESCE(ev.maintenance_orders_12m, 0) AS maintenance_orders_12m, COALESCE(ev.open_maintenance_orders, 0) AS open_maintenance_orders, COALESCE(ev.urgent_maintenance_orders_12m, 0) AS urgent_maintenance_orders_12m, COALESCE(ev.parts_cost_aud_12m, 0) AS parts_cost_aud_12m, rd.max_vibration_mm_s, COALESCE(rd.breach_readings, 0) AS breach_readings, d.rated_units_per_hour FROM silver.machine d LEFT JOIN ev ON ev.machine_id = d.machine_id LEFT JOIN rd ON rd.machine_id = d.machine_id WHERE d._is_current = true) SELECT s.*, ROUND(LEAST(100, COALESCE(s.criticality, 1) * 8 + s.open_maintenance_orders * 6 + s.urgent_maintenance_orders_12m * 4 + LEAST(s.breach_readings, 20) * 1.5), 1) AS risk_score, CASE WHEN LEAST(100, COALESCE(s.criticality, 1) * 8 + s.open_maintenance_orders * 6 + s.urgent_maintenance_orders_12m * 4 + LEAST(s.breach_readings, 20) * 1.5) >= 60 THEN 'High' WHEN LEAST(100, COALESCE(s.criticality, 1) * 8 + s.open_maintenance_orders * 6 + s.urgent_maintenance_orders_12m * 4 + LEAST(s.breach_readings, 20) * 1.5) >= 35 THEN 'Medium' ELSE 'Low' END AS risk_band, ROUND(s.open_maintenance_orders * 5000 + LEAST(s.breach_readings, 20) * 3000 + COALESCE(s.rated_units_per_hour, 0) * COALESCE(s.criticality, 1) * 2, 0) AS downtime_cost_at_risk_aud FROM s","description":"Machine risk score - maintenance orders, urgent work and vibration breaches, plus downtime cost at risk in AUD (weights and dollar rates are illustrative: $5,000 per open maintenance order + $3,000 per 15-min window above the vibration limit + rated units/hour x criticality x $2)","business_owner":"Manufacturing Engineering","consumers":"Power BI / Direct Lake, data agent","active":true},{"transform_id":"MF-G06","industry":"manufacturing","gold_table":"distributor_contact_summary","run_order":210,"sql_text":"SELECT plant, channel, COUNT(*) AS distributor_contacts, SUM(CASE WHEN status = 'Active' THEN 1 ELSE 0 END) AS active_distributor_contacts, ROUND(AVG(year(current_date()) - birth_year), 1) AS avg_age, COUNT(DISTINCT machine_id) AS linked_machines FROM silver.distributor_contact WHERE _is_deleted = false GROUP BY plant, channel","description":"Distributor contacts by plant and channel","business_owner":"Sales Operations","consumers":"Power BI / Direct Lake, data agent","active":true}],"onboard":{"object":{"object_id":"MF_IOT_ENERGY_PRICES","industry":"manufacturing","source_system":"iot","source_object":"energy_prices","source_type":"file","ingest_method":"landed_file","landing_format":"txt","landing_path":"landing/iot/energy_prices","read_options":"{\"header\": true, \"delimiter\": \"|\"}","source_table":null,"load_type":"incremental","watermark_column":null,"primary_keys":"plant,obs_date","dedupe_order_by":"_ingested_at DESC","bronze_table":"iot_energy_prices","silver_table":"energy_price","silver_write_mode":"merge_scd1","load_group":"reference","priority":25,"sla_hours":26,"data_owner":"Operations Finance","sensitivity":"Public","active":true,"description":"Daily wholesale energy prices by plant (pipe-delimited, dd/MM/yyyy) - onboarded live in the demo","fabric_alternative":"Public or partner feed - could be a REST copy, or a shortcut if the provider shares a folder"},"columns":[["plant","plant","string",null],["obs_date","obs_date","date","to_date(obs_date, 'dd/MM/yyyy')"],["avg_price_mwh","avg_price_mwh","double",null],["peak_price_mwh","peak_price_mwh","double",null],["max_temp_c","max_temp_c","double",null],["price_band","price_band","string",null]],"rules":[["MF-EXT-01","avg_price_mwh plausible","technical","range","avg_price_mwh","{\"min\": -1000, \"max\": 17500}","warn"],["MF-EXT-02","Known price_band","business","allowed_values","price_band","{\"values\": [\"Low\", \"Normal\", \"High\", \"Spike\"]}","warn"]],"gold":{"transform_id":"MF-G07","industry":"manufacturing","gold_table":"maintenance_order_vs_energy_price","run_order":220,"sql_text":"WITH ev AS (SELECT d.plant, to_date(v.reported_ts) AS obs_date, COUNT(*) AS maintenance_orders, SUM(CASE WHEN v.mo_type = 'BREAKDOWN' THEN 1 ELSE 0 END) AS urgent_maintenance_orders FROM silver.maintenance_order v JOIN silver.machine d ON d.machine_id = v.machine_id AND d._is_current = true WHERE v._is_deleted = false GROUP BY d.plant, to_date(v.reported_ts)) SELECT x.plant, x.obs_date, x.avg_price_mwh, x.peak_price_mwh, x.max_temp_c, x.price_band, COALESCE(ev.maintenance_orders, 0) AS maintenance_orders, COALESCE(ev.urgent_maintenance_orders, 0) AS urgent_maintenance_orders FROM silver.energy_price x LEFT JOIN ev ON ev.plant = x.plant AND ev.obs_date = x.obs_date WHERE x._is_deleted = false","description":"Energy price spikes against breakdowns by plant","business_owner":"Operations Finance","consumers":"Power BI / Direct Lake","active":true}}},"healthcare":{"pack":{"id":"healthcare","industry":"Healthcare","org":"Coastal Health Network","short":"CHN","prefix":"HC","tagline":"public health service running five hospitals","email_domain":"chn.example.com.au","region_col":"hospital","region_label":"hospital","regions":["Gosford","Wyong","Newcastle","Maitland","Tamworth"],"name_words":["Banksia","Waratah","Wattle","Jacaranda","Grevillea","Bottlebrush","Kookaburra","Lyrebird","Pelican","Heron"],"sys":{"ops":{"code":"pas","name":"Patient Administration System (on-prem SQL Server)","type":"Database","owner":"eHealth","fabric":"Mirroring for SQL Server vs Copy job incremental; check privacy obligations before replicating"},"tel":{"code":"bms","name":"Bed management feed (15-min CSV) and flu surveillance","type":"File drop","owner":"Patient Flow","fabric":"Eventstream for real-time bed status; files to OneLake for batch"},"crm":{"code":"crm","name":"Dynamics 365 patient engagement","type":"SaaS CRM","owner":"Patient Experience","fabric":"Link to Microsoft Fabric (Dataverse) or Copy job"}},"entity":{"table":"ward","src":"wards","label":"ward","plural":"wards","id":"ward_id","code":"ward_code","code_prefix":"WD","name_col":"ward_name","type_col":"ward_type","types":["EMERGENCY","ICU","MEDICAL","SURGICAL","MATERNITY"],"status":["OPEN","PARTIAL","CLOSED","DECOMMISSIONED"],"tier_col":"acuity_level","tier_label":"acuity level (1-5, 5 = highest)","cap_col":"bed_count","cap_label":"beds","cap_range":[8,48],"start_col":"opened_date","start_years":[1975,2025],"drift_col":"nurse_unit_manager","drift_values":["NUM roster A","NUM roster B","NUM roster C"],"owner":"Clinical Operations","count":120},"event":{"table":"admission","src":"admissions","label":"admission","plural":"admissions","id":"admission_id","id_start":800001,"type_col":"admission_type","types":["EMERGENCY","ELECTIVE","TRANSFER","MATERNITY"],"type_names":["Emergency","Elective","Inter-hospital transfer","Maternity"],"urgent_type":"EMERGENCY","status":["ADMITTED","IN_TREATMENT","DISCHARGED","CANCELLED"],"open_ts":"admit_ts","close_ts":"discharge_ts","open_verb":"admitted","close_verb":"discharged","amount_col":"episode_cost_aud","amount_range":[900,90000],"qty_col":"nursing_hours","qty_range":[2,300],"descriptions":["Chest pain observation","Hip replacement","Respiratory infection","Planned caesarean","Fractured wrist","Sepsis management","Day surgery","Cardiac monitoring"],"owner":"Clinical Operations"},"reading":{"table":"ward_occupancy","src":"ward_occupancy","label":"ward occupancy (15-min)","fact":"ward_occupancy","monitored":24,"monitored_types":[],"metrics":[{"col":"occupancy_pct","label":"bed occupancy","unit":"%","base":82,"sd":9,"min":0,"max":120,"threshold":95},{"col":"patients_waiting","label":"patients waiting for a bed","unit":"patients","base":3,"sd":2,"min":0,"max":200},{"col":"nurses_on_shift","label":"nurses on shift","unit":"nurses","base":9,"sd":2,"min":0,"max":200}],"sentinel":999.9,"owner":"Patient Flow"},"party":{"table":"patient","src":"patients","label":"patient","plural":"patients","id":"patient_id","id_prefix":"P","sf_type":"contact","segment_col":"funding_type","segments":["Public","Private insured","DVA","Overseas visitor"],"status":["Active","Deceased"],"sensitive_col":"medicare_no","sensitive_label":"Medicare number","sensitive_treatment":"hash","owner":"Patient Experience","count":300},"extra":{"table":"flu_surveillance","src":"flu_surveillance","label":"influenza surveillance","sys":"tel","cols":[{"col":"ili_rate_per_1000","base":12,"sd":6,"min":0,"max":500},{"col":"lab_notifications","base":25,"sd":12,"min":0,"max":5000},{"col":"ed_presentations","base":180,"sd":40,"min":0,"max":5000}],"cat_col":"alert_level","cat_values":["Low","Moderate","High","Very high"],"owner":"Public Health","story":"flu activity against emergency admissions by hospital"},"alert":{"stream":"bed_alert","label":"bed alert","codes":["ED_OVERCAPACITY","BED_BLOCK","CODE_BLUE"],"storm":"ED surge"},"ref":{"title":"Length-of-stay targets","owner":"Clinical Operations","rows":[["EMERGENCY","Unplanned",96,"Emergency Medicine"],["ELECTIVE","Planned",72,"Surgical Services"],["TRANSFER","Transfer",120,"Patient Flow"],["MATERNITY","Planned",60,"Women's Health"]]},"ai":{"persona":"patient flow and clinical operations","cust":"state health departments, local health districts, private hospital groups, aged care","q_model":["Which hospital had the most emergency admissions this month?","Show average length of stay by admission type.","Which high-acuity wards have more than 10 patients currently admitted?"],"q_agent":["Which 10 wards have the highest risk score and how many patients are currently admitted?","Compare average length of stay for emergency admissions by hospital.","Which wards went above 95% occupancy in the last 3 days?","What did elective admissions cost in Newcastle?","How many DVA patients are active per hospital?"],"q_onto":["Show ICU wards in Gosford with emergency admissions still open and a high risk band.","For the ward with the highest occupancy, list current admissions and their episode cost.","Which patients are linked to wards that are partially closed?"]},"impact":{"col":"access_block_cost_aud","label":"access-block cost at risk","open_cost":1500,"breach_cost":2000,"cap_rate":500,"rule":"$1,500 per open admission + $2,000 per 15-min window above 95% occupancy + beds x acuity x $500"}},"source_system":[{"source_system":"pas","industry":"healthcare","display_name":"Patient Administration System (on-prem SQL Server)","system_type":"Database","connection_pattern":"On-premises data gateway -> pipeline Copy with watermark query","technical_owner":"eHealth","fabric_options":"Mirroring for SQL Server vs Copy job incremental; check privacy obligations before replicating"},{"source_system":"bms","industry":"healthcare","display_name":"Bed management feed (15-min CSV) and flu surveillance","system_type":"File drop","connection_pattern":"Files dropped to OneLake (Files/landing) every 15 minutes","technical_owner":"Patient Flow","fabric_options":"Eventstream for real-time bed status; files to OneLake for batch"},{"source_system":"crm","industry":"healthcare","display_name":"Dynamics 365 patient engagement","system_type":"SaaS CRM","connection_pattern":"REST/OData API -> JSON landed per day","technical_owner":"Patient Experience","fabric_options":"Link to Microsoft Fabric (Dataverse) or Copy job"}],"ingestion_object":[{"object_id":"HC_PAS_WARDS","industry":"healthcare","source_system":"pas","source_object":"wards","source_type":"database","ingest_method":"copy_db","landing_format":"csv","landing_path":"landing/pas/wards","read_options":"{\"header\": true, \"delimiter\": \",\"}","source_table":"src.wards","load_type":"incremental","watermark_column":"last_modified","primary_keys":"ward_id","dedupe_order_by":"last_modified DESC, _ingested_at DESC","bronze_table":"pas_wards","silver_table":"ward","silver_write_mode":"merge_scd2","load_group":"reference","priority":10,"sla_hours":26,"data_owner":"Clinical Operations","sensitivity":"Internal","active":true,"description":"Ward register. SCD2 keeps status, acuity_level and bed_count history.","fabric_alternative":"Strong mirroring candidate - small, slowly changing, needed in near real time by operations"},{"object_id":"HC_PAS_ADMISSIONS","industry":"healthcare","source_system":"pas","source_object":"admissions","source_type":"database","ingest_method":"copy_db","landing_format":"csv","landing_path":"landing/pas/admissions","read_options":"{\"header\": true, \"delimiter\": \",\"}","source_table":"src.admissions","load_type":"incremental","watermark_column":"last_modified","primary_keys":"admission_id","dedupe_order_by":"last_modified DESC, _ingested_at DESC","bronze_table":"pas_admissions","silver_table":"admission","silver_write_mode":"merge_scd1","load_group":"transactional","priority":20,"sla_hours":26,"data_owner":"Clinical Operations","sensitivity":"Internal","active":true,"description":"Admissions - incremental by last_modified; can arrive before their ward (late-arriving).","fabric_alternative":"Copy job incremental or mirroring; keep the watermark query if the source is on-prem"},{"object_id":"HC_BMS_WARD_OCCUPANCY","industry":"healthcare","source_system":"bms","source_object":"ward_occupancy","source_type":"file","ingest_method":"landed_file","landing_format":"csv","landing_path":"landing/bms/ward_occupancy","read_options":"{\"header\": true, \"delimiter\": \",\"}","source_table":null,"load_type":"incremental","watermark_column":null,"primary_keys":"ward_id,reading_ts","dedupe_order_by":"_ingested_at DESC","bronze_table":"bms_ward_occupancy","silver_table":"ward_occupancy","silver_write_mode":"insert_only","load_group":"telemetry","priority":30,"sla_hours":2,"data_owner":"Patient Flow","sensitivity":"Internal","active":true,"description":"Ward occupancy (15-min) every 15 minutes, four micro-batch files a day. Insert-only and idempotent.","fabric_alternative":"Eventstream into Eventhouse for real time; keep this batch path for history and reconciliation"},{"object_id":"HC_CRM_PATIENTS","industry":"healthcare","source_system":"crm","source_object":"patients","source_type":"api","ingest_method":"landed_file","landing_format":"json","landing_path":"landing/crm/patients","read_options":"{\"multiline\": true, \"json_root\": \"records\"}","source_table":null,"load_type":"full","watermark_column":null,"primary_keys":"patient_id","dedupe_order_by":"last_modified_ts DESC, _ingested_at DESC","bronze_table":"crm_patients","silver_table":"patient","silver_write_mode":"merge_scd1","load_group":"reference","priority":15,"sla_hours":26,"data_owner":"Patient Experience","sensitivity":"Confidential","active":true,"description":"Patients - full daily snapshot from the CRM API (contains PII). Missing rows are soft-deleted.","fabric_alternative":"Copy job with the CRM connector; Dataverse-based CRMs can use Link to Fabric instead of copying"}],"column_map":[{"object_id":"HC_PAS_WARDS","column_order":0,"source_column":"ward_id","target_column":"ward_id","data_type":"int","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"HC_PAS_WARDS","column_order":1,"source_column":"ward_code","target_column":"ward_code","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"HC_PAS_WARDS","column_order":2,"source_column":"ward_name","target_column":"ward_name","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"HC_PAS_WARDS","column_order":3,"source_column":"ward_type","target_column":"ward_type","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"HC_PAS_WARDS","column_order":4,"source_column":"hospital","target_column":"hospital","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"HC_PAS_WARDS","column_order":5,"source_column":"status","target_column":"status","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"HC_PAS_WARDS","column_order":6,"source_column":"acuity_level","target_column":"acuity_level","data_type":"int","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"HC_PAS_WARDS","column_order":7,"source_column":"bed_count","target_column":"bed_count","data_type":"double","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"HC_PAS_WARDS","column_order":8,"source_column":"opened_date","target_column":"opened_date","data_type":"date","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"HC_PAS_WARDS","column_order":9,"source_column":"last_modified","target_column":"last_modified","data_type":"timestamp","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"HC_PAS_ADMISSIONS","column_order":0,"source_column":"admission_id","target_column":"admission_id","data_type":"int","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"HC_PAS_ADMISSIONS","column_order":1,"source_column":"ward_id","target_column":"ward_id","data_type":"int","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"HC_PAS_ADMISSIONS","column_order":2,"source_column":"admission_type","target_column":"admission_type","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"HC_PAS_ADMISSIONS","column_order":3,"source_column":"priority","target_column":"priority","data_type":"int","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"HC_PAS_ADMISSIONS","column_order":4,"source_column":"status","target_column":"status","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"HC_PAS_ADMISSIONS","column_order":5,"source_column":"admit_ts","target_column":"admit_ts","data_type":"timestamp","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"HC_PAS_ADMISSIONS","column_order":6,"source_column":"discharge_ts","target_column":"discharge_ts","data_type":"timestamp","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"HC_PAS_ADMISSIONS","column_order":7,"source_column":"episode_cost_aud","target_column":"episode_cost_aud","data_type":"decimal(12,2)","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"HC_PAS_ADMISSIONS","column_order":8,"source_column":"nursing_hours","target_column":"nursing_hours","data_type":"double","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"HC_PAS_ADMISSIONS","column_order":9,"source_column":"description","target_column":"description","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"HC_PAS_ADMISSIONS","column_order":10,"source_column":"last_modified","target_column":"last_modified","data_type":"timestamp","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"HC_BMS_WARD_OCCUPANCY","column_order":0,"source_column":"ward_id","target_column":"ward_id","data_type":"int","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"HC_BMS_WARD_OCCUPANCY","column_order":1,"source_column":"reading_time","target_column":"reading_ts","data_type":"timestamp","transform_expr":"to_timestamp(reading_time, 'dd/MM/yyyy HH:mm')","pii_treatment":"none","sensitivity":"none","description":""},{"object_id":"HC_BMS_WARD_OCCUPANCY","column_order":2,"source_column":"occupancy_pct","target_column":"occupancy_pct","data_type":"double","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"HC_BMS_WARD_OCCUPANCY","column_order":3,"source_column":"patients_waiting","target_column":"patients_waiting","data_type":"double","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"HC_BMS_WARD_OCCUPANCY","column_order":4,"source_column":"nurses_on_shift","target_column":"nurses_on_shift","data_type":"double","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"HC_CRM_PATIENTS","column_order":0,"source_column":"Id","target_column":"patient_id","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"HC_CRM_PATIENTS","column_order":1,"source_column":"FirstName","target_column":"first_name","data_type":"string","transform_expr":null,"pii_treatment":"drop","sensitivity":"Confidential","description":""},{"object_id":"HC_CRM_PATIENTS","column_order":2,"source_column":"LastName","target_column":"last_name","data_type":"string","transform_expr":null,"pii_treatment":"drop","sensitivity":"Confidential","description":""},{"object_id":"HC_CRM_PATIENTS","column_order":3,"source_column":"Email","target_column":"email_masked","data_type":"string","transform_expr":null,"pii_treatment":"mask_email","sensitivity":"Confidential","description":""},{"object_id":"HC_CRM_PATIENTS","column_order":4,"source_column":"MobilePhone","target_column":"mobile_masked","data_type":"string","transform_expr":null,"pii_treatment":"mask_phone","sensitivity":"Confidential","description":""},{"object_id":"HC_CRM_PATIENTS","column_order":5,"source_column":"Birthdate","target_column":"birth_year","data_type":"int","transform_expr":null,"pii_treatment":"year_only","sensitivity":"Confidential","description":""},{"object_id":"HC_CRM_PATIENTS","column_order":6,"source_column":"MailingPostalCode","target_column":"postcode","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"HC_CRM_PATIENTS","column_order":7,"source_column":"Hospital__c","target_column":"hospital","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"HC_CRM_PATIENTS","column_order":8,"source_column":"FundingType__c","target_column":"funding_type","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"HC_CRM_PATIENTS","column_order":9,"source_column":"WardId__c","target_column":"ward_id","data_type":"int","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"HC_CRM_PATIENTS","column_order":10,"source_column":"Status__c","target_column":"status","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"HC_CRM_PATIENTS","column_order":11,"source_column":"MedicareNo__c","target_column":"medicare_no_hash","data_type":"string","transform_expr":null,"pii_treatment":"hash","sensitivity":"Restricted","description":""},{"object_id":"HC_CRM_PATIENTS","column_order":12,"source_column":"LastModifiedDate","target_column":"last_modified_ts","data_type":"timestamp","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""}],"dq_rule":[{"rule_id":"HC-ENT-01","object_id":"HC_PAS_WARDS","rule_name":"Ward id present","rule_category":"technical","check_type":"not_null","column_name":"ward_id","params":"{}","severity":"fail","fail_threshold_pct":0,"rule_owner":"Data engineering","description":"Primary key must never be null","active":true},{"rule_id":"HC-ENT-02","object_id":"HC_PAS_WARDS","rule_name":"Known ward_type","rule_category":"technical","check_type":"allowed_values","column_name":"ward_type","params":"{\"values\": [\"EMERGENCY\", \"ICU\", \"MEDICAL\", \"SURGICAL\", \"MATERNITY\"]}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Data engineering","description":"Unknown classes are kept but flagged for the data owner","active":true},{"rule_id":"HC-ENT-03","object_id":"HC_PAS_WARDS","rule_name":"acuity_level between 1 and 5","rule_category":"technical","check_type":"range","column_name":"acuity_level","params":"{\"min\": 1, \"max\": 5}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Data engineering","description":"Out-of-range ratings usually mean a keying error in the source","active":true},{"rule_id":"HC-ENT-04","object_id":"HC_PAS_WARDS","rule_name":"Known status","rule_category":"business","check_type":"allowed_values","column_name":"status","params":"{\"values\": [\"OPEN\", \"PARTIAL\", \"CLOSED\", \"DECOMMISSIONED\"]}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Clinical Operations","description":"Status list agreed with Clinical Operations","active":true},{"rule_id":"HC-EVT-01","object_id":"HC_PAS_ADMISSIONS","rule_name":"Admission id present","rule_category":"technical","check_type":"not_null","column_name":"admission_id","params":"{}","severity":"fail","fail_threshold_pct":0,"rule_owner":"Data engineering","description":"Primary key must never be null","active":true},{"rule_id":"HC-EVT-02","object_id":"HC_PAS_ADMISSIONS","rule_name":"Ward exists","rule_category":"technical","check_type":"referential","column_name":"ward_id","params":"{\"ref_table\": \"silver.ward\", \"ref_column\": \"ward_id\"}","severity":"quarantine","fail_threshold_pct":null,"rule_owner":"Data engineering","description":"Held until the ward arrives (late-arriving reference data), then merged automatically","active":true},{"rule_id":"HC-EVT-03","object_id":"HC_PAS_ADMISSIONS","rule_name":"Discharged after admitted","rule_category":"business","check_type":"expression","column_name":null,"params":"{\"expr\": \"discharge_ts IS NULL OR discharge_ts >= admit_ts\"}","severity":"quarantine","fail_threshold_pct":null,"rule_owner":"Clinical Operations","description":"A admission can't be discharged before it was admitted","active":true},{"rule_id":"HC-EVT-04","object_id":"HC_PAS_ADMISSIONS","rule_name":"episode_cost_aud not negative","rule_category":"technical","check_type":"range","column_name":"episode_cost_aud","params":"{\"min\": 0}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Data engineering","description":"Negative values are sign errors in the source extract","active":true},{"rule_id":"HC-EVT-05","object_id":"HC_PAS_ADMISSIONS","rule_name":"EMERGENCY is priority 1","rule_category":"business","check_type":"expression","column_name":null,"params":"{\"expr\": \"admission_type <> 'EMERGENCY' OR priority = 1\"}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Clinical Operations","description":"Emergency admissions must be raised as priority 1","active":true},{"rule_id":"HC-EVT-06","object_id":"HC_PAS_ADMISSIONS","rule_name":"admit_ts parses","rule_category":"technical","check_type":"not_null","column_name":"admit_ts","params":"{}","severity":"fail","fail_threshold_pct":5,"rule_owner":"Data engineering","description":"If more than 5% of timestamps fail to parse, the extract format has changed - stop the load","active":true},{"rule_id":"HC-RDG-01","object_id":"HC_BMS_WARD_OCCUPANCY","rule_name":"Bed occupancy plausible","rule_category":"technical","check_type":"range","column_name":"occupancy_pct","params":"{\"min\": 0, \"max\": 120}","severity":"quarantine","fail_threshold_pct":null,"rule_owner":"Data engineering","description":"Sensor fault values (e.g. 999.9) are quarantined, never averaged into Gold","active":true},{"rule_id":"HC-RDG-02","object_id":"HC_BMS_WARD_OCCUPANCY","rule_name":"Ward known","rule_category":"technical","check_type":"referential","column_name":"ward_id","params":"{\"ref_table\": \"silver.ward\", \"ref_column\": \"ward_id\"}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Data engineering","description":"Readings for unknown tags are flagged","active":true},{"rule_id":"HC-RDG-03","object_id":"HC_BMS_WARD_OCCUPANCY","rule_name":"Reading time parses","rule_category":"technical","check_type":"not_null","column_name":"reading_ts","params":"{}","severity":"fail","fail_threshold_pct":5,"rule_owner":"Data engineering","description":"Timestamp format change stops the load","active":true},{"rule_id":"HC-PTY-01","object_id":"HC_CRM_PATIENTS","rule_name":"Patient id present","rule_category":"technical","check_type":"not_null","column_name":"patient_id","params":"{}","severity":"fail","fail_threshold_pct":0,"rule_owner":"Data engineering","description":"Primary key must never be null","active":true},{"rule_id":"HC-PTY-02","object_id":"HC_CRM_PATIENTS","rule_name":"Email looks valid","rule_category":"technical","check_type":"regex","column_name":"email_masked","params":"{\"pattern\": \"^[^@\\\\s]+@[^@\\\\s]+\\\\.[^@\\\\s]+$\"}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Data engineering","description":"Checked on the raw value before masking","active":true},{"rule_id":"HC-PTY-03","object_id":"HC_CRM_PATIENTS","rule_name":"Australian postcode","rule_category":"technical","check_type":"regex","column_name":"postcode","params":"{\"pattern\": \"^\\\\d{4}$\"}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Data engineering","description":"Four digits","active":true},{"rule_id":"HC-PTY-04","object_id":"HC_CRM_PATIENTS","rule_name":"Known funding_type","rule_category":"business","check_type":"allowed_values","column_name":"funding_type","params":"{\"values\": [\"Public\", \"Private insured\", \"DVA\", \"Overseas visitor\"]}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Patient Experience","description":"Segments agreed with Patient Experience","active":true}],"gold_transform":[{"transform_id":"HC-G01","industry":"healthcare","gold_table":"dim_date","run_order":100,"sql_text":"SELECT CAST(date_format(d, 'yyyyMMdd') AS INT) AS date_key, d AS calendar_date, year(d) AS calendar_year, quarter(d) AS calendar_quarter, month(d) AS month_number, date_format(d, 'MMM') AS month_name, date_format(d, 'yyyy-MM') AS year_month, dayofmonth(d) AS day_of_month, date_format(d, 'EEE') AS day_name, CASE WHEN month(d) >= 7 THEN year(d) + 1 ELSE year(d) END AS fy_year, concat('FY', substring(CAST(CASE WHEN month(d) >= 7 THEN year(d) + 1 ELSE year(d) END AS STRING), 3, 2)) AS fy_label, CASE WHEN dayofweek(d) IN (1, 7) THEN true ELSE false END AS is_weekend FROM (SELECT explode(sequence(to_date('2020-01-01'), to_date('2028-12-31'), interval 1 day)) AS d) dates","description":"Calendar with Australian financial year","business_owner":"Data engineering","consumers":"Power BI / Direct Lake, data agent","active":true},{"transform_id":"HC-G02","industry":"healthcare","gold_table":"dim_ward","run_order":110,"sql_text":"SELECT ward_id, ward_code, ward_name, ward_type, hospital, status, acuity_level, bed_count, opened_date, CAST(floor(months_between(current_date(), opened_date) / 12) AS INT) AS age_years, _valid_from AS version_valid_from FROM silver.ward WHERE _is_current = true","description":"Current view of the ward register (history stays in Silver SCD2)","business_owner":"Clinical Operations","consumers":"Power BI / Direct Lake, data agent","active":true},{"transform_id":"HC-G03","industry":"healthcare","gold_table":"fact_admission","run_order":120,"sql_text":"SELECT v.admission_id, v.ward_id, v.admission_type, v.priority, v.status, v.admit_ts, v.discharge_ts, to_date(v.admit_ts) AS opened_date, CAST(date_format(v.admit_ts, 'yyyyMMdd') AS INT) AS opened_date_key, ROUND((unix_timestamp(v.discharge_ts) - unix_timestamp(v.admit_ts)) / 3600.0, 2) AS hours_to_close, CASE WHEN v.status IN ('ADMITTED', 'IN_TREATMENT') THEN 1 ELSE 0 END AS is_open, v.episode_cost_aud, v.nursing_hours, d.hospital, d.ward_type, d.acuity_level FROM silver.admission v LEFT JOIN silver.ward d ON d.ward_id = v.ward_id AND d._is_current = true WHERE v._is_deleted = false","description":"Admissions with duration and ward attributes","business_owner":"Clinical Operations","consumers":"Power BI / Direct Lake, data agent","active":true},{"transform_id":"HC-G04","industry":"healthcare","gold_table":"fact_ward_occupancy_hourly","run_order":130,"sql_text":"SELECT ward_id, date_trunc('HOUR', reading_ts) AS reading_hour, to_date(reading_ts) AS reading_date, CAST(date_format(reading_ts, 'yyyyMMdd') AS INT) AS reading_date_key, COUNT(*) AS readings, ROUND(AVG(occupancy_pct), 2) AS avg_occupancy_pct, ROUND(MAX(occupancy_pct), 2) AS max_occupancy_pct, ROUND(AVG(patients_waiting), 2) AS avg_patients_waiting, ROUND(AVG(nurses_on_shift), 2) AS avg_nurses_on_shift, SUM(CASE WHEN occupancy_pct > 95 THEN 1 ELSE 0 END) AS breach_readings FROM silver.ward_occupancy GROUP BY ward_id, date_trunc('HOUR', reading_ts), to_date(reading_ts), CAST(date_format(reading_ts, 'yyyyMMdd') AS INT)","description":"Hourly ward occupancy (15-min) with threshold breaches","business_owner":"Patient Flow","consumers":"Power BI / Direct Lake, data agent","active":true},{"transform_id":"HC-G05","industry":"healthcare","gold_table":"ward_health_scorecard","run_order":200,"sql_text":"WITH asat AS (SELECT MAX(to_date(admit_ts)) AS as_at FROM silver.admission), ev AS (SELECT v.ward_id, COUNT(*) AS admissions_12m, SUM(CASE WHEN v.status IN ('ADMITTED', 'IN_TREATMENT') THEN 1 ELSE 0 END) AS open_admissions, SUM(CASE WHEN v.admission_type = 'EMERGENCY' THEN 1 ELSE 0 END) AS urgent_admissions_12m, ROUND(SUM(v.episode_cost_aud), 2) AS episode_cost_aud_12m FROM silver.admission v CROSS JOIN asat WHERE v._is_deleted = false AND to_date(v.admit_ts) > add_months(asat.as_at, -12) GROUP BY v.ward_id), rd AS (SELECT ward_id, MAX(occupancy_pct) AS max_occupancy_pct, SUM(CASE WHEN occupancy_pct > 95 THEN 1 ELSE 0 END) AS breach_readings FROM silver.ward_occupancy GROUP BY ward_id), s AS (SELECT d.ward_id, d.ward_code, d.ward_name, d.ward_type, d.hospital, d.status, d.acuity_level, COALESCE(ev.admissions_12m, 0) AS admissions_12m, COALESCE(ev.open_admissions, 0) AS open_admissions, COALESCE(ev.urgent_admissions_12m, 0) AS urgent_admissions_12m, COALESCE(ev.episode_cost_aud_12m, 0) AS episode_cost_aud_12m, rd.max_occupancy_pct, COALESCE(rd.breach_readings, 0) AS breach_readings, d.bed_count FROM silver.ward d LEFT JOIN ev ON ev.ward_id = d.ward_id LEFT JOIN rd ON rd.ward_id = d.ward_id WHERE d._is_current = true) SELECT s.*, ROUND(LEAST(100, COALESCE(s.acuity_level, 1) * 8 + s.open_admissions * 6 + s.urgent_admissions_12m * 4 + LEAST(s.breach_readings, 20) * 1.5), 1) AS risk_score, CASE WHEN LEAST(100, COALESCE(s.acuity_level, 1) * 8 + s.open_admissions * 6 + s.urgent_admissions_12m * 4 + LEAST(s.breach_readings, 20) * 1.5) >= 60 THEN 'High' WHEN LEAST(100, COALESCE(s.acuity_level, 1) * 8 + s.open_admissions * 6 + s.urgent_admissions_12m * 4 + LEAST(s.breach_readings, 20) * 1.5) >= 35 THEN 'Medium' ELSE 'Low' END AS risk_band, ROUND(s.open_admissions * 1500 + LEAST(s.breach_readings, 20) * 2000 + COALESCE(s.bed_count, 0) * COALESCE(s.acuity_level, 1) * 500, 0) AS access_block_cost_aud FROM s","description":"Ward risk score - admissions, urgent work and bed occupancy breaches, plus access-block cost at risk in AUD (weights and dollar rates are illustrative: $1,500 per open admission + $2,000 per 15-min window above 95% occupancy + beds x acuity x $500)","business_owner":"Clinical Operations","consumers":"Power BI / Direct Lake, data agent","active":true},{"transform_id":"HC-G06","industry":"healthcare","gold_table":"patient_summary","run_order":210,"sql_text":"SELECT hospital, funding_type, COUNT(*) AS patients, SUM(CASE WHEN status = 'Active' THEN 1 ELSE 0 END) AS active_patients, ROUND(AVG(year(current_date()) - birth_year), 1) AS avg_age, COUNT(DISTINCT ward_id) AS linked_wards FROM silver.patient WHERE _is_deleted = false GROUP BY hospital, funding_type","description":"Patients by hospital and funding_type","business_owner":"Patient Experience","consumers":"Power BI / Direct Lake, data agent","active":true}],"onboard":{"object":{"object_id":"HC_BMS_FLU_SURVEILLANCE","industry":"healthcare","source_system":"bms","source_object":"flu_surveillance","source_type":"file","ingest_method":"landed_file","landing_format":"txt","landing_path":"landing/bms/flu_surveillance","read_options":"{\"header\": true, \"delimiter\": \"|\"}","source_table":null,"load_type":"incremental","watermark_column":null,"primary_keys":"hospital,obs_date","dedupe_order_by":"_ingested_at DESC","bronze_table":"bms_flu_surveillance","silver_table":"flu_surveillance","silver_write_mode":"merge_scd1","load_group":"reference","priority":25,"sla_hours":26,"data_owner":"Public Health","sensitivity":"Public","active":true,"description":"Daily influenza surveillance by hospital (pipe-delimited, dd/MM/yyyy) - onboarded live in the demo","fabric_alternative":"Public or partner feed - could be a REST copy, or a shortcut if the provider shares a folder"},"columns":[["hospital","hospital","string",null],["obs_date","obs_date","date","to_date(obs_date, 'dd/MM/yyyy')"],["ili_rate_per_1000","ili_rate_per_1000","double",null],["lab_notifications","lab_notifications","double",null],["ed_presentations","ed_presentations","double",null],["alert_level","alert_level","string",null]],"rules":[["HC-EXT-01","ili_rate_per_1000 plausible","technical","range","ili_rate_per_1000","{\"min\": 0, \"max\": 500}","warn"],["HC-EXT-02","Known alert_level","business","allowed_values","alert_level","{\"values\": [\"Low\", \"Moderate\", \"High\", \"Very high\"]}","warn"]],"gold":{"transform_id":"HC-G07","industry":"healthcare","gold_table":"admission_vs_flu_surveillance","run_order":220,"sql_text":"WITH ev AS (SELECT d.hospital, to_date(v.admit_ts) AS obs_date, COUNT(*) AS admissions, SUM(CASE WHEN v.admission_type = 'EMERGENCY' THEN 1 ELSE 0 END) AS urgent_admissions FROM silver.admission v JOIN silver.ward d ON d.ward_id = v.ward_id AND d._is_current = true WHERE v._is_deleted = false GROUP BY d.hospital, to_date(v.admit_ts)) SELECT x.hospital, x.obs_date, x.ili_rate_per_1000, x.lab_notifications, x.ed_presentations, x.alert_level, COALESCE(ev.admissions, 0) AS admissions, COALESCE(ev.urgent_admissions, 0) AS urgent_admissions FROM silver.flu_surveillance x LEFT JOIN ev ON ev.hospital = x.hospital AND ev.obs_date = x.obs_date WHERE x._is_deleted = false","description":"Flu activity against emergency admissions by hospital","business_owner":"Public Health","consumers":"Power BI / Direct Lake","active":true}}},"ites":{"pack":{"id":"ites","industry":"IT & ITeS","org":"Bluegum Digital Services","short":"BDS","prefix":"IT","tagline":"IT services provider running managed services for enterprise clients","email_domain":"bluegum.example.com.au","region_col":"delivery_centre","region_label":"delivery centre","regions":["Sydney","Melbourne","Brisbane","Bengaluru","Manila","Auckland"],"name_words":["Kookaburra Bank","Southern Rail","Coastal Health","Metro Water","Outback Mining","Harbour Insurance","Summit Retail","Banksia Uni","Gumtree Logistics","Redgum Energy"],"sys":{"ops":{"code":"itsm","name":"ITSM and service catalogue database (on-prem SQL Server)","type":"Database","owner":"Service Management","fabric":"Mirroring for SQL Server vs Copy job incremental; ServiceNow connector if SaaS"},"tel":{"code":"mon","name":"Service-level monitoring exports (15-min CSV) and change calendar","type":"File drop","owner":"Service Assurance","fabric":"Eventstream from Azure Monitor / Event Hubs; Azure Monitor logs via KQL"},"crm":{"code":"hub","name":"HubSpot CRM (client contacts)","type":"SaaS CRM","owner":"Client Services","fabric":"Copy job / pipeline with the REST connector"}},"entity":{"table":"client_service","src":"client_services","label":"client service","plural":"client services","id":"service_id","code":"contract_ref","code_prefix":"MSA","name_col":"service_name","type_col":"service_line","types":["MANAGED_INFRA","APP_SUPPORT","CLOUD_OPS","SERVICE_DESK","SECURITY_OPS"],"status":["LIVE","AT_RISK","IN_TRANSITION","EXITED"],"tier_col":"sla_tier","tier_label":"SLA tier (1-5, 5 = platinum)","cap_col":"monthly_fee_aud","cap_label":"monthly contract fee (AUD)","cap_range":[8000,450000],"start_col":"contract_start","start_years":[2015,2025],"drift_col":"delivery_lead","drift_values":["Delivery Lead A","Delivery Lead B","Offshore Centre"],"owner":"Service Delivery","count":240},"event":{"table":"incident","src":"incidents","label":"incident","plural":"incidents","id":"incident_id","id_start":400001,"type_col":"category","types":["OUTAGE","PERFORMANCE","SECURITY","REQUEST"],"type_names":["Outage","Performance","Security","Service request"],"urgent_type":"OUTAGE","status":["NEW","IN_PROGRESS","RESOLVED","CANCELLED"],"open_ts":"opened_ts","close_ts":"resolved_ts","open_verb":"opened","close_verb":"resolved","amount_col":"sla_penalty_aud","amount_range":[0,25000],"qty_col":"effort_hours","qty_range":[0.25,60],"descriptions":["Disk full on data volume","TLS certificate expired","Slow page loads","Suspicious sign-in","Password reset","Failed backup job","Memory leak after deploy","VPN tunnel down"],"owner":"Service Delivery"},"reading":{"table":"service_metric","src":"service_metrics","label":"service performance (15-min)","fact":"service_perf","monitored":24,"monitored_types":["MANAGED_INFRA","APP_SUPPORT","CLOUD_OPS"],"metrics":[{"col":"p95_response_ms","label":"p95 response time","unit":"ms","base":900,"sd":300,"min":0,"max":120000,"threshold":2500},{"col":"availability_pct","label":"availability","unit":"%","base":99.8,"sd":0.15,"min":0,"max":100},{"col":"ticket_backlog","label":"ticket backlog","unit":"tickets","base":25,"sd":10,"min":0,"max":10000}],"sentinel":-1,"owner":"Service Assurance"},"party":{"table":"client_contact","src":"client_contacts","label":"client contact","plural":"client contacts","id":"contact_id","id_prefix":"CC","sf_type":"contact","segment_col":"contact_role","segments":["Executive sponsor","Service owner","IT manager","Procurement"],"status":["Active","Left company"],"sensitive_col":"personal_mobile_2","sensitive_label":"secondary mobile","sensitive_treatment":"drop","owner":"Client Services","count":300},"extra":{"table":"change_window","src":"change_calendar","label":"change calendar","sys":"tel","cols":[{"col":"changes_scheduled","base":12,"sd":5,"min":0,"max":500},{"col":"emergency_changes","base":1,"sd":1,"min":0,"max":100},{"col":"patching_hours","base":3,"sd":2,"min":0,"max":24}],"cat_col":"change_risk","cat_values":["Low","Medium","High","Freeze"],"owner":"Change Management","story":"change volume against incidents by delivery centre"},"alert":{"stream":"sla_alert","label":"SLA alert","codes":["SLA_RESPONSE_BREACH","HOST_DOWN","CERT_EXPIRY"],"storm":"failed deployment"},"ref":{"title":"Incident resolution SLAs","owner":"Service Delivery","rows":[["OUTAGE","P1",4,"Major Incident"],["PERFORMANCE","P2",12,"Service Assurance"],["SECURITY","P1",8,"Security Operations"],["REQUEST","P4",72,"Service Desk"]]},"impact":{"col":"revenue_at_risk_aud","label":"contract revenue at risk","open_cost":2000,"breach_cost":500,"cap_rate":0.01,"rule":"$2,000 SLA penalty exposure per open incident + $500 per slow 15-min window + monthly fee x SLA tier x 1%"},"ai":{"persona":"service delivery, SLA and account management","cust":"IT services providers, managed service providers and internal IT teams","q_model":["Which delivery centre had the most outages this month?","Show SLA penalties by incident category and month.","Which platinum client services (SLA tier 4 or 5) have open incidents?"],"q_agent":["Which 10 client services have the most contract revenue at risk, and how many open incidents does each have?","Compare average hours to resolve outages by delivery centre.","Which client services had p95 response time above 2500 ms in the last 3 days?","How much have we paid in SLA penalties for security incidents?","Which service lines carry the most monthly fee in the High risk band?"],"q_onto":["Show managed infrastructure services delivered from Bengaluru with an open outage and a high risk band.","For the client service with the worst response time, list its incidents and the client contacts behind it.","Which executive sponsors are linked to client services that are at risk?"]}},"source_system":[{"source_system":"itsm","industry":"ites","display_name":"ITSM and service catalogue database (on-prem SQL Server)","system_type":"Database","connection_pattern":"On-premises data gateway -> pipeline Copy with watermark query","technical_owner":"Service Management","fabric_options":"Mirroring for SQL Server vs Copy job incremental; ServiceNow connector if SaaS"},{"source_system":"mon","industry":"ites","display_name":"Service-level monitoring exports (15-min CSV) and change calendar","system_type":"File drop","connection_pattern":"Files dropped to OneLake (Files/landing) every 15 minutes","technical_owner":"Service Assurance","fabric_options":"Eventstream from Azure Monitor / Event Hubs; Azure Monitor logs via KQL"},{"source_system":"hub","industry":"ites","display_name":"HubSpot CRM (client contacts)","system_type":"SaaS CRM","connection_pattern":"REST/OData API -> JSON landed per day","technical_owner":"Client Services","fabric_options":"Copy job / pipeline with the REST connector"}],"ingestion_object":[{"object_id":"IT_ITSM_CLIENT_SERVICES","industry":"ites","source_system":"itsm","source_object":"client_services","source_type":"database","ingest_method":"copy_db","landing_format":"csv","landing_path":"landing/itsm/client_services","read_options":"{\"header\": true, \"delimiter\": \",\"}","source_table":"src.client_services","load_type":"incremental","watermark_column":"last_modified","primary_keys":"service_id","dedupe_order_by":"last_modified DESC, _ingested_at DESC","bronze_table":"itsm_client_services","silver_table":"client_service","silver_write_mode":"merge_scd2","load_group":"reference","priority":10,"sla_hours":26,"data_owner":"Service Delivery","sensitivity":"Internal","active":true,"description":"Client service register. SCD2 keeps status, sla_tier and monthly_fee_aud history.","fabric_alternative":"Strong mirroring candidate - small, slowly changing, needed in near real time by operations"},{"object_id":"IT_ITSM_INCIDENTS","industry":"ites","source_system":"itsm","source_object":"incidents","source_type":"database","ingest_method":"copy_db","landing_format":"csv","landing_path":"landing/itsm/incidents","read_options":"{\"header\": true, \"delimiter\": \",\"}","source_table":"src.incidents","load_type":"incremental","watermark_column":"last_modified","primary_keys":"incident_id","dedupe_order_by":"last_modified DESC, _ingested_at DESC","bronze_table":"itsm_incidents","silver_table":"incident","silver_write_mode":"merge_scd1","load_group":"transactional","priority":20,"sla_hours":26,"data_owner":"Service Delivery","sensitivity":"Internal","active":true,"description":"Incidents - incremental by last_modified; can arrive before their client service (late-arriving).","fabric_alternative":"Copy job incremental or mirroring; keep the watermark query if the source is on-prem"},{"object_id":"IT_MON_SERVICE_METRICS","industry":"ites","source_system":"mon","source_object":"service_metrics","source_type":"file","ingest_method":"landed_file","landing_format":"csv","landing_path":"landing/mon/service_metrics","read_options":"{\"header\": true, \"delimiter\": \",\"}","source_table":null,"load_type":"incremental","watermark_column":null,"primary_keys":"service_id,reading_ts","dedupe_order_by":"_ingested_at DESC","bronze_table":"mon_service_metrics","silver_table":"service_metric","silver_write_mode":"insert_only","load_group":"telemetry","priority":30,"sla_hours":2,"data_owner":"Service Assurance","sensitivity":"Internal","active":true,"description":"Service performance (15-min) every 15 minutes, four micro-batch files a day. Insert-only and idempotent.","fabric_alternative":"Eventstream into Eventhouse for real time; keep this batch path for history and reconciliation"},{"object_id":"IT_HUB_CLIENT_CONTACTS","industry":"ites","source_system":"hub","source_object":"client_contacts","source_type":"api","ingest_method":"landed_file","landing_format":"json","landing_path":"landing/hub/client_contacts","read_options":"{\"multiline\": true, \"json_root\": \"records\"}","source_table":null,"load_type":"full","watermark_column":null,"primary_keys":"contact_id","dedupe_order_by":"last_modified_ts DESC, _ingested_at DESC","bronze_table":"hub_client_contacts","silver_table":"client_contact","silver_write_mode":"merge_scd1","load_group":"reference","priority":15,"sla_hours":26,"data_owner":"Client Services","sensitivity":"Confidential","active":true,"description":"Client contacts - full daily snapshot from the CRM API (contains PII). Missing rows are soft-deleted.","fabric_alternative":"Copy job with the CRM connector; Dataverse-based CRMs can use Link to Fabric instead of copying"}],"column_map":[{"object_id":"IT_ITSM_CLIENT_SERVICES","column_order":0,"source_column":"service_id","target_column":"service_id","data_type":"int","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"IT_ITSM_CLIENT_SERVICES","column_order":1,"source_column":"contract_ref","target_column":"contract_ref","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"IT_ITSM_CLIENT_SERVICES","column_order":2,"source_column":"service_name","target_column":"service_name","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"IT_ITSM_CLIENT_SERVICES","column_order":3,"source_column":"service_line","target_column":"service_line","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"IT_ITSM_CLIENT_SERVICES","column_order":4,"source_column":"delivery_centre","target_column":"delivery_centre","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"IT_ITSM_CLIENT_SERVICES","column_order":5,"source_column":"status","target_column":"status","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"IT_ITSM_CLIENT_SERVICES","column_order":6,"source_column":"sla_tier","target_column":"sla_tier","data_type":"int","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"IT_ITSM_CLIENT_SERVICES","column_order":7,"source_column":"monthly_fee_aud","target_column":"monthly_fee_aud","data_type":"double","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"IT_ITSM_CLIENT_SERVICES","column_order":8,"source_column":"contract_start","target_column":"contract_start","data_type":"date","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"IT_ITSM_CLIENT_SERVICES","column_order":9,"source_column":"last_modified","target_column":"last_modified","data_type":"timestamp","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"IT_ITSM_INCIDENTS","column_order":0,"source_column":"incident_id","target_column":"incident_id","data_type":"int","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"IT_ITSM_INCIDENTS","column_order":1,"source_column":"service_id","target_column":"service_id","data_type":"int","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"IT_ITSM_INCIDENTS","column_order":2,"source_column":"category","target_column":"category","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"IT_ITSM_INCIDENTS","column_order":3,"source_column":"priority","target_column":"priority","data_type":"int","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"IT_ITSM_INCIDENTS","column_order":4,"source_column":"status","target_column":"status","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"IT_ITSM_INCIDENTS","column_order":5,"source_column":"opened_ts","target_column":"opened_ts","data_type":"timestamp","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"IT_ITSM_INCIDENTS","column_order":6,"source_column":"resolved_ts","target_column":"resolved_ts","data_type":"timestamp","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"IT_ITSM_INCIDENTS","column_order":7,"source_column":"sla_penalty_aud","target_column":"sla_penalty_aud","data_type":"decimal(12,2)","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"IT_ITSM_INCIDENTS","column_order":8,"source_column":"effort_hours","target_column":"effort_hours","data_type":"double","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"IT_ITSM_INCIDENTS","column_order":9,"source_column":"description","target_column":"description","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"IT_ITSM_INCIDENTS","column_order":10,"source_column":"last_modified","target_column":"last_modified","data_type":"timestamp","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"IT_MON_SERVICE_METRICS","column_order":0,"source_column":"service_id","target_column":"service_id","data_type":"int","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"IT_MON_SERVICE_METRICS","column_order":1,"source_column":"reading_time","target_column":"reading_ts","data_type":"timestamp","transform_expr":"to_timestamp(reading_time, 'dd/MM/yyyy HH:mm')","pii_treatment":"none","sensitivity":"none","description":""},{"object_id":"IT_MON_SERVICE_METRICS","column_order":2,"source_column":"p95_response_ms","target_column":"p95_response_ms","data_type":"double","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"IT_MON_SERVICE_METRICS","column_order":3,"source_column":"availability_pct","target_column":"availability_pct","data_type":"double","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"IT_MON_SERVICE_METRICS","column_order":4,"source_column":"ticket_backlog","target_column":"ticket_backlog","data_type":"double","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"IT_HUB_CLIENT_CONTACTS","column_order":0,"source_column":"Id","target_column":"contact_id","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"IT_HUB_CLIENT_CONTACTS","column_order":1,"source_column":"FirstName","target_column":"first_name","data_type":"string","transform_expr":null,"pii_treatment":"drop","sensitivity":"Confidential","description":""},{"object_id":"IT_HUB_CLIENT_CONTACTS","column_order":2,"source_column":"LastName","target_column":"last_name","data_type":"string","transform_expr":null,"pii_treatment":"drop","sensitivity":"Confidential","description":""},{"object_id":"IT_HUB_CLIENT_CONTACTS","column_order":3,"source_column":"Email","target_column":"email_masked","data_type":"string","transform_expr":null,"pii_treatment":"mask_email","sensitivity":"Confidential","description":""},{"object_id":"IT_HUB_CLIENT_CONTACTS","column_order":4,"source_column":"MobilePhone","target_column":"mobile_masked","data_type":"string","transform_expr":null,"pii_treatment":"mask_phone","sensitivity":"Confidential","description":""},{"object_id":"IT_HUB_CLIENT_CONTACTS","column_order":5,"source_column":"Birthdate","target_column":"birth_year","data_type":"int","transform_expr":null,"pii_treatment":"year_only","sensitivity":"Confidential","description":""},{"object_id":"IT_HUB_CLIENT_CONTACTS","column_order":6,"source_column":"MailingPostalCode","target_column":"postcode","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"IT_HUB_CLIENT_CONTACTS","column_order":7,"source_column":"DeliveryCentre__c","target_column":"delivery_centre","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"IT_HUB_CLIENT_CONTACTS","column_order":8,"source_column":"ContactRole__c","target_column":"contact_role","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"IT_HUB_CLIENT_CONTACTS","column_order":9,"source_column":"ServiceId__c","target_column":"service_id","data_type":"int","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"IT_HUB_CLIENT_CONTACTS","column_order":10,"source_column":"Status__c","target_column":"status","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"IT_HUB_CLIENT_CONTACTS","column_order":11,"source_column":"PersonalMobile2__c","target_column":"personal_mobile_2","data_type":"string","transform_expr":null,"pii_treatment":"drop","sensitivity":"Restricted","description":""},{"object_id":"IT_HUB_CLIENT_CONTACTS","column_order":12,"source_column":"LastModifiedDate","target_column":"last_modified_ts","data_type":"timestamp","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""}],"dq_rule":[{"rule_id":"IT-ENT-01","object_id":"IT_ITSM_CLIENT_SERVICES","rule_name":"Client service id present","rule_category":"technical","check_type":"not_null","column_name":"service_id","params":"{}","severity":"fail","fail_threshold_pct":0,"rule_owner":"Data engineering","description":"Primary key must never be null","active":true},{"rule_id":"IT-ENT-02","object_id":"IT_ITSM_CLIENT_SERVICES","rule_name":"Known service_line","rule_category":"technical","check_type":"allowed_values","column_name":"service_line","params":"{\"values\": [\"MANAGED_INFRA\", \"APP_SUPPORT\", \"CLOUD_OPS\", \"SERVICE_DESK\", \"SECURITY_OPS\"]}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Data engineering","description":"Unknown classes are kept but flagged for the data owner","active":true},{"rule_id":"IT-ENT-03","object_id":"IT_ITSM_CLIENT_SERVICES","rule_name":"sla_tier between 1 and 5","rule_category":"technical","check_type":"range","column_name":"sla_tier","params":"{\"min\": 1, \"max\": 5}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Data engineering","description":"Out-of-range ratings usually mean a keying error in the source","active":true},{"rule_id":"IT-ENT-04","object_id":"IT_ITSM_CLIENT_SERVICES","rule_name":"Known status","rule_category":"business","check_type":"allowed_values","column_name":"status","params":"{\"values\": [\"LIVE\", \"AT_RISK\", \"IN_TRANSITION\", \"EXITED\"]}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Service Delivery","description":"Status list agreed with Service Delivery","active":true},{"rule_id":"IT-EVT-01","object_id":"IT_ITSM_INCIDENTS","rule_name":"Incident id present","rule_category":"technical","check_type":"not_null","column_name":"incident_id","params":"{}","severity":"fail","fail_threshold_pct":0,"rule_owner":"Data engineering","description":"Primary key must never be null","active":true},{"rule_id":"IT-EVT-02","object_id":"IT_ITSM_INCIDENTS","rule_name":"Client service exists","rule_category":"technical","check_type":"referential","column_name":"service_id","params":"{\"ref_table\": \"silver.client_service\", \"ref_column\": \"service_id\"}","severity":"quarantine","fail_threshold_pct":null,"rule_owner":"Data engineering","description":"Held until the client service arrives (late-arriving reference data), then merged automatically","active":true},{"rule_id":"IT-EVT-03","object_id":"IT_ITSM_INCIDENTS","rule_name":"Resolved after opened","rule_category":"business","check_type":"expression","column_name":null,"params":"{\"expr\": \"resolved_ts IS NULL OR resolved_ts >= opened_ts\"}","severity":"quarantine","fail_threshold_pct":null,"rule_owner":"Service Delivery","description":"A incident can't be resolved before it was opened","active":true},{"rule_id":"IT-EVT-04","object_id":"IT_ITSM_INCIDENTS","rule_name":"sla_penalty_aud not negative","rule_category":"technical","check_type":"range","column_name":"sla_penalty_aud","params":"{\"min\": 0}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Data engineering","description":"Negative values are sign errors in the source extract","active":true},{"rule_id":"IT-EVT-05","object_id":"IT_ITSM_INCIDENTS","rule_name":"OUTAGE is priority 1","rule_category":"business","check_type":"expression","column_name":null,"params":"{\"expr\": \"category <> 'OUTAGE' OR priority = 1\"}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Service Delivery","description":"Outage incidents must be raised as priority 1","active":true},{"rule_id":"IT-EVT-06","object_id":"IT_ITSM_INCIDENTS","rule_name":"opened_ts parses","rule_category":"technical","check_type":"not_null","column_name":"opened_ts","params":"{}","severity":"fail","fail_threshold_pct":5,"rule_owner":"Data engineering","description":"If more than 5% of timestamps fail to parse, the extract format has changed - stop the load","active":true},{"rule_id":"IT-RDG-01","object_id":"IT_MON_SERVICE_METRICS","rule_name":"P95 response time plausible","rule_category":"technical","check_type":"range","column_name":"p95_response_ms","params":"{\"min\": 0, \"max\": 120000}","severity":"quarantine","fail_threshold_pct":null,"rule_owner":"Data engineering","description":"Sensor fault values (e.g. -1) are quarantined, never averaged into Gold","active":true},{"rule_id":"IT-RDG-02","object_id":"IT_MON_SERVICE_METRICS","rule_name":"Client service known","rule_category":"technical","check_type":"referential","column_name":"service_id","params":"{\"ref_table\": \"silver.client_service\", \"ref_column\": \"service_id\"}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Data engineering","description":"Readings for unknown tags are flagged","active":true},{"rule_id":"IT-RDG-03","object_id":"IT_MON_SERVICE_METRICS","rule_name":"Reading time parses","rule_category":"technical","check_type":"not_null","column_name":"reading_ts","params":"{}","severity":"fail","fail_threshold_pct":5,"rule_owner":"Data engineering","description":"Timestamp format change stops the load","active":true},{"rule_id":"IT-PTY-01","object_id":"IT_HUB_CLIENT_CONTACTS","rule_name":"Client contact id present","rule_category":"technical","check_type":"not_null","column_name":"contact_id","params":"{}","severity":"fail","fail_threshold_pct":0,"rule_owner":"Data engineering","description":"Primary key must never be null","active":true},{"rule_id":"IT-PTY-02","object_id":"IT_HUB_CLIENT_CONTACTS","rule_name":"Email looks valid","rule_category":"technical","check_type":"regex","column_name":"email_masked","params":"{\"pattern\": \"^[^@\\\\s]+@[^@\\\\s]+\\\\.[^@\\\\s]+$\"}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Data engineering","description":"Checked on the raw value before masking","active":true},{"rule_id":"IT-PTY-03","object_id":"IT_HUB_CLIENT_CONTACTS","rule_name":"Australian postcode","rule_category":"technical","check_type":"regex","column_name":"postcode","params":"{\"pattern\": \"^\\\\d{4}$\"}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Data engineering","description":"Four digits","active":true},{"rule_id":"IT-PTY-04","object_id":"IT_HUB_CLIENT_CONTACTS","rule_name":"Known contact_role","rule_category":"business","check_type":"allowed_values","column_name":"contact_role","params":"{\"values\": [\"Executive sponsor\", \"Service owner\", \"IT manager\", \"Procurement\"]}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Client Services","description":"Segments agreed with Client Services","active":true}],"gold_transform":[{"transform_id":"IT-G01","industry":"ites","gold_table":"dim_date","run_order":100,"sql_text":"SELECT CAST(date_format(d, 'yyyyMMdd') AS INT) AS date_key, d AS calendar_date, year(d) AS calendar_year, quarter(d) AS calendar_quarter, month(d) AS month_number, date_format(d, 'MMM') AS month_name, date_format(d, 'yyyy-MM') AS year_month, dayofmonth(d) AS day_of_month, date_format(d, 'EEE') AS day_name, CASE WHEN month(d) >= 7 THEN year(d) + 1 ELSE year(d) END AS fy_year, concat('FY', substring(CAST(CASE WHEN month(d) >= 7 THEN year(d) + 1 ELSE year(d) END AS STRING), 3, 2)) AS fy_label, CASE WHEN dayofweek(d) IN (1, 7) THEN true ELSE false END AS is_weekend FROM (SELECT explode(sequence(to_date('2020-01-01'), to_date('2028-12-31'), interval 1 day)) AS d) dates","description":"Calendar with Australian financial year","business_owner":"Data engineering","consumers":"Power BI / Direct Lake, data agent","active":true},{"transform_id":"IT-G02","industry":"ites","gold_table":"dim_client_service","run_order":110,"sql_text":"SELECT service_id, contract_ref, service_name, service_line, delivery_centre, status, sla_tier, monthly_fee_aud, contract_start, CAST(floor(months_between(current_date(), contract_start) / 12) AS INT) AS age_years, _valid_from AS version_valid_from FROM silver.client_service WHERE _is_current = true","description":"Current view of the client service register (history stays in Silver SCD2)","business_owner":"Service Delivery","consumers":"Power BI / Direct Lake, data agent","active":true},{"transform_id":"IT-G03","industry":"ites","gold_table":"fact_incident","run_order":120,"sql_text":"SELECT v.incident_id, v.service_id, v.category, v.priority, v.status, v.opened_ts, v.resolved_ts, to_date(v.opened_ts) AS opened_date, CAST(date_format(v.opened_ts, 'yyyyMMdd') AS INT) AS opened_date_key, ROUND((unix_timestamp(v.resolved_ts) - unix_timestamp(v.opened_ts)) / 3600.0, 2) AS hours_to_close, CASE WHEN v.status IN ('NEW', 'IN_PROGRESS') THEN 1 ELSE 0 END AS is_open, v.sla_penalty_aud, v.effort_hours, d.delivery_centre, d.service_line, d.sla_tier FROM silver.incident v LEFT JOIN silver.client_service d ON d.service_id = v.service_id AND d._is_current = true WHERE v._is_deleted = false","description":"Incidents with duration and client service attributes","business_owner":"Service Delivery","consumers":"Power BI / Direct Lake, data agent","active":true},{"transform_id":"IT-G04","industry":"ites","gold_table":"fact_service_perf_hourly","run_order":130,"sql_text":"SELECT service_id, date_trunc('HOUR', reading_ts) AS reading_hour, to_date(reading_ts) AS reading_date, CAST(date_format(reading_ts, 'yyyyMMdd') AS INT) AS reading_date_key, COUNT(*) AS readings, ROUND(AVG(p95_response_ms), 2) AS avg_p95_response_ms, ROUND(MAX(p95_response_ms), 2) AS max_p95_response_ms, ROUND(AVG(availability_pct), 2) AS avg_availability_pct, ROUND(AVG(ticket_backlog), 2) AS avg_ticket_backlog, SUM(CASE WHEN p95_response_ms > 2500 THEN 1 ELSE 0 END) AS breach_readings FROM silver.service_metric GROUP BY service_id, date_trunc('HOUR', reading_ts), to_date(reading_ts), CAST(date_format(reading_ts, 'yyyyMMdd') AS INT)","description":"Hourly service performance (15-min) with threshold breaches","business_owner":"Service Assurance","consumers":"Power BI / Direct Lake, data agent","active":true},{"transform_id":"IT-G05","industry":"ites","gold_table":"client_service_health_scorecard","run_order":200,"sql_text":"WITH asat AS (SELECT MAX(to_date(opened_ts)) AS as_at FROM silver.incident), ev AS (SELECT v.service_id, COUNT(*) AS incidents_12m, SUM(CASE WHEN v.status IN ('NEW', 'IN_PROGRESS') THEN 1 ELSE 0 END) AS open_incidents, SUM(CASE WHEN v.category = 'OUTAGE' THEN 1 ELSE 0 END) AS urgent_incidents_12m, ROUND(SUM(v.sla_penalty_aud), 2) AS sla_penalty_aud_12m FROM silver.incident v CROSS JOIN asat WHERE v._is_deleted = false AND to_date(v.opened_ts) > add_months(asat.as_at, -12) GROUP BY v.service_id), rd AS (SELECT service_id, MAX(p95_response_ms) AS max_p95_response_ms, SUM(CASE WHEN p95_response_ms > 2500 THEN 1 ELSE 0 END) AS breach_readings FROM silver.service_metric GROUP BY service_id), s AS (SELECT d.service_id, d.contract_ref, d.service_name, d.service_line, d.delivery_centre, d.status, d.sla_tier, COALESCE(ev.incidents_12m, 0) AS incidents_12m, COALESCE(ev.open_incidents, 0) AS open_incidents, COALESCE(ev.urgent_incidents_12m, 0) AS urgent_incidents_12m, COALESCE(ev.sla_penalty_aud_12m, 0) AS sla_penalty_aud_12m, rd.max_p95_response_ms, COALESCE(rd.breach_readings, 0) AS breach_readings, d.monthly_fee_aud FROM silver.client_service d LEFT JOIN ev ON ev.service_id = d.service_id LEFT JOIN rd ON rd.service_id = d.service_id WHERE d._is_current = true) SELECT s.*, ROUND(LEAST(100, COALESCE(s.sla_tier, 1) * 8 + s.open_incidents * 6 + s.urgent_incidents_12m * 4 + LEAST(s.breach_readings, 20) * 1.5), 1) AS risk_score, CASE WHEN LEAST(100, COALESCE(s.sla_tier, 1) * 8 + s.open_incidents * 6 + s.urgent_incidents_12m * 4 + LEAST(s.breach_readings, 20) * 1.5) >= 60 THEN 'High' WHEN LEAST(100, COALESCE(s.sla_tier, 1) * 8 + s.open_incidents * 6 + s.urgent_incidents_12m * 4 + LEAST(s.breach_readings, 20) * 1.5) >= 35 THEN 'Medium' ELSE 'Low' END AS risk_band, ROUND(s.open_incidents * 2000 + LEAST(s.breach_readings, 20) * 500 + COALESCE(s.monthly_fee_aud, 0) * COALESCE(s.sla_tier, 1) * 0.01, 0) AS revenue_at_risk_aud FROM s","description":"Client service risk score - incidents, urgent work and p95 response time breaches, plus contract revenue at risk in AUD (weights and dollar rates are illustrative: $2,000 SLA penalty exposure per open incident + $500 per slow 15-min window + monthly fee x SLA tier x 1%)","business_owner":"Service Delivery","consumers":"Power BI / Direct Lake, data agent","active":true},{"transform_id":"IT-G06","industry":"ites","gold_table":"client_contact_summary","run_order":210,"sql_text":"SELECT delivery_centre, contact_role, COUNT(*) AS client_contacts, SUM(CASE WHEN status = 'Active' THEN 1 ELSE 0 END) AS active_client_contacts, ROUND(AVG(year(current_date()) - birth_year), 1) AS avg_age, COUNT(DISTINCT service_id) AS linked_client_services FROM silver.client_contact WHERE _is_deleted = false GROUP BY delivery_centre, contact_role","description":"Client contacts by delivery centre and contact_role","business_owner":"Client Services","consumers":"Power BI / Direct Lake, data agent","active":true}],"onboard":{"object":{"object_id":"IT_MON_CHANGE_CALENDAR","industry":"ites","source_system":"mon","source_object":"change_calendar","source_type":"file","ingest_method":"landed_file","landing_format":"txt","landing_path":"landing/mon/change_calendar","read_options":"{\"header\": true, \"delimiter\": \"|\"}","source_table":null,"load_type":"incremental","watermark_column":null,"primary_keys":"delivery_centre,obs_date","dedupe_order_by":"_ingested_at DESC","bronze_table":"mon_change_calendar","silver_table":"change_window","silver_write_mode":"merge_scd1","load_group":"reference","priority":25,"sla_hours":26,"data_owner":"Change Management","sensitivity":"Public","active":true,"description":"Daily change calendar by delivery centre (pipe-delimited, dd/MM/yyyy) - onboarded live in the demo","fabric_alternative":"Public or partner feed - could be a REST copy, or a shortcut if the provider shares a folder"},"columns":[["delivery_centre","delivery_centre","string",null],["obs_date","obs_date","date","to_date(obs_date, 'dd/MM/yyyy')"],["changes_scheduled","changes_scheduled","double",null],["emergency_changes","emergency_changes","double",null],["patching_hours","patching_hours","double",null],["change_risk","change_risk","string",null]],"rules":[["IT-EXT-01","changes_scheduled plausible","technical","range","changes_scheduled","{\"min\": 0, \"max\": 500}","warn"],["IT-EXT-02","Known change_risk","business","allowed_values","change_risk","{\"values\": [\"Low\", \"Medium\", \"High\", \"Freeze\"]}","warn"]],"gold":{"transform_id":"IT-G07","industry":"ites","gold_table":"incident_vs_change_window","run_order":220,"sql_text":"WITH ev AS (SELECT d.delivery_centre, to_date(v.opened_ts) AS obs_date, COUNT(*) AS incidents, SUM(CASE WHEN v.category = 'OUTAGE' THEN 1 ELSE 0 END) AS urgent_incidents FROM silver.incident v JOIN silver.client_service d ON d.service_id = v.service_id AND d._is_current = true WHERE v._is_deleted = false GROUP BY d.delivery_centre, to_date(v.opened_ts)) SELECT x.delivery_centre, x.obs_date, x.changes_scheduled, x.emergency_changes, x.patching_hours, x.change_risk, COALESCE(ev.incidents, 0) AS incidents, COALESCE(ev.urgent_incidents, 0) AS urgent_incidents FROM silver.change_window x LEFT JOIN ev ON ev.delivery_centre = x.delivery_centre AND ev.obs_date = x.obs_date WHERE x._is_deleted = false","description":"Change volume against incidents by delivery centre","business_owner":"Change Management","consumers":"Power BI / Direct Lake","active":true}}},"government":{"pack":{"id":"government","industry":"Government","org":"Riverbend Regional Authority","short":"RRA","prefix":"GV","tagline":"government authority delivering local infrastructure and citizen services (fits a council or a state agency)","email_domain":"riverbend.example.gov.au","region_col":"district","region_label":"district","regions":["North District","South District","East District","West District","Central District"],"name_words":["Murray St","River Rd","Banksia Park","Station St","Ridge Rd","Lakeside","Market Sq","Hill St","Creek Rd","Library Lane"],"sys":{"ops":{"code":"ams","name":"Asset and works management system (on-prem SQL Server)","type":"Database","owner":"Infrastructure Services","fabric":"Mirroring for SQL Server vs Copy job incremental vs pipeline Copy"},"tel":{"code":"iot","name":"Smart infrastructure sensor network (15-min CSV) and BoM rainfall","type":"IoT file drop","owner":"Smart Infrastructure","fabric":"Eventstream from IoT Hub; files to OneLake for batch"},"crm":{"code":"crm","name":"Citizen request and case portal (Salesforce Public Sector)","type":"SaaS CRM","owner":"Citizen Services","fabric":"Copy job with the Salesforce connector; Dynamics 365 / Dataverse portals can use Link to Fabric"}},"entity":{"table":"public_asset","src":"public_assets","label":"public asset","plural":"public assets","id":"asset_id","code":"asset_ref","code_prefix":"PA","name_col":"asset_name","type_col":"asset_type","types":["ROAD_SEGMENT","BRIDGE","PUBLIC_BUILDING","STORMWATER_PIT","STREETLIGHT"],"status":["IN_SERVICE","UNDER_REPAIR","CLOSED","DISPOSED"],"tier_col":"criticality","tier_label":"criticality (1-5, 5 = most critical)","cap_col":"replacement_value_kaud","cap_label":"replacement value (AUD thousands)","cap_range":[5,25000],"start_col":"constructed_date","start_years":[1950,2025],"drift_col":"maintenance_contractor","drift_values":["Contractor A","Contractor B","In-house crew"],"owner":"Infrastructure Services","count":240},"event":{"table":"citizen_case","src":"citizen_cases","label":"citizen case","plural":"citizen cases","id":"case_id","id_start":600001,"type_col":"case_type","types":["ROAD_REPAIR","FLOOD_DRAINAGE","PUBLIC_SAFETY","PERMIT"],"type_names":["Road repair","Flooding and drainage","Public safety hazard","Permit or licence application"],"urgent_type":"FLOOD_DRAINAGE","status":["LODGED","ASSIGNED","COMPLETED","WITHDRAWN"],"open_ts":"lodged_ts","close_ts":"completed_ts","open_verb":"lodged","close_verb":"completed","amount_col":"cost_aud","amount_range":[80,30000],"qty_col":"crew_hours","qty_range":[0.5,40],"descriptions":["Pothole reported near school","Blocked stormwater drain","Streetlight out on main road","Fallen branch on footpath","Water over road","Event permit for park","Damaged handrail on bridge","Footpath trip hazard"],"owner":"Citizen Services"},"reading":{"table":"sensor_reading","src":"sensor_readings","label":"smart infrastructure sensor readings","fact":"sensor","monitored":24,"monitored_types":["STORMWATER_PIT","BRIDGE","ROAD_SEGMENT"],"metrics":[{"col":"water_level_cm","label":"water level","unit":"cm","base":35,"sd":15,"min":0,"max":500,"threshold":150},{"col":"traffic_count","label":"vehicle count","unit":"vehicles","base":140,"sd":60,"min":0,"max":10000},{"col":"pm25_ugm3","label":"PM2.5","unit":"µg/m³","base":8,"sd":4,"min":0,"max":1000}],"sentinel":999.9,"owner":"Smart Infrastructure"},"party":{"table":"citizen","src":"citizens","label":"citizen","plural":"citizens","id":"citizen_id","id_prefix":"CZ","sf_type":"contact","segment_col":"citizen_type","segments":["Resident","Business","Concession holder","Visitor"],"status":["Active","Moved away"],"sensitive_col":"concession_card_no","sensitive_label":"concession card number","sensitive_treatment":"hash","owner":"Citizen Services","count":300},"extra":{"table":"rainfall_daily","src":"rainfall","label":"BoM rainfall","sys":"tel","cols":[{"col":"rainfall_mm","base":4,"sd":8,"min":0,"max":500},{"col":"max_gust_kmh","base":40,"sd":15,"min":0,"max":200},{"col":"river_height_m","base":1.8,"sd":0.6,"min":0,"max":20}],"cat_col":"warning_level","cat_values":["None","Advice","Watch and act","Emergency"],"owner":"Emergency Management","story":"rainfall and river height against flooding cases by district"},"alert":{"stream":"sensor_alert","label":"sensor alert","codes":["FLOOD_LEVEL","LIGHT_FAULT","TAMPER"],"storm":"storm front"},"ref":{"title":"Citizen case service standards","owner":"Citizen Services","rows":[["ROAD_REPAIR","Roads",72,"Roads Maintenance"],["FLOOD_DRAINAGE","Emergency",4,"Stormwater"],["PUBLIC_SAFETY","Safety",24,"Field Response"],["PERMIT","Approvals",480,"Permits and Licensing"]]},"impact":{"col":"cost_at_risk_aud","label":"cost at risk","open_cost":2500,"breach_cost":1200,"cap_rate":0.5,"rule":"$2,500 per open case + $1,200 per 15-min window above the water-level threshold + replacement value (k) x criticality x $0.50"},"ai":{"persona":"public sector operations and citizen services","cust":"state agencies, local councils, transport, water and emergency services","q_model":["Which district lodged the most citizen cases this month?","Show citizen case cost by case type and month.","Which critical assets have open flooding cases?"],"q_agent":["Which 10 public assets have the highest cost at risk, and how many open cases does each have?","Compare average hours to complete flooding cases by district against the 4-hour standard.","Which stormwater pits went above 150 cm water level in the last 3 days?","How many permit applications are still open and how long have they been waiting?","How many concession holders lodged cases in each district?"],"q_onto":["Show bridges in East District with an open case and a high risk band.","For the stormwater pit with the highest water level, list its citizen cases and their cost.","Which citizens are linked to assets that are closed?"]}},"source_system":[{"source_system":"ams","industry":"government","display_name":"Asset and works management system (on-prem SQL Server)","system_type":"Database","connection_pattern":"On-premises data gateway -> pipeline Copy with watermark query","technical_owner":"Infrastructure Services","fabric_options":"Mirroring for SQL Server vs Copy job incremental vs pipeline Copy"},{"source_system":"iot","industry":"government","display_name":"Smart infrastructure sensor network (15-min CSV) and BoM rainfall","system_type":"IoT file drop","connection_pattern":"Files dropped to OneLake (Files/landing) every 15 minutes","technical_owner":"Smart Infrastructure","fabric_options":"Eventstream from IoT Hub; files to OneLake for batch"},{"source_system":"crm","industry":"government","display_name":"Citizen request and case portal (Salesforce Public Sector)","system_type":"SaaS CRM","connection_pattern":"REST/OData API -> JSON landed per day","technical_owner":"Citizen Services","fabric_options":"Copy job with the Salesforce connector; Dynamics 365 / Dataverse portals can use Link to Fabric"}],"ingestion_object":[{"object_id":"GV_AMS_PUBLIC_ASSETS","industry":"government","source_system":"ams","source_object":"public_assets","source_type":"database","ingest_method":"copy_db","landing_format":"csv","landing_path":"landing/ams/public_assets","read_options":"{\"header\": true, \"delimiter\": \",\"}","source_table":"src.public_assets","load_type":"incremental","watermark_column":"last_modified","primary_keys":"asset_id","dedupe_order_by":"last_modified DESC, _ingested_at DESC","bronze_table":"ams_public_assets","silver_table":"public_asset","silver_write_mode":"merge_scd2","load_group":"reference","priority":10,"sla_hours":26,"data_owner":"Infrastructure Services","sensitivity":"Internal","active":true,"description":"Public asset register. SCD2 keeps status, criticality and replacement_value_kaud history.","fabric_alternative":"Strong mirroring candidate - small, slowly changing, needed in near real time by operations"},{"object_id":"GV_AMS_CITIZEN_CASES","industry":"government","source_system":"ams","source_object":"citizen_cases","source_type":"database","ingest_method":"copy_db","landing_format":"csv","landing_path":"landing/ams/citizen_cases","read_options":"{\"header\": true, \"delimiter\": \",\"}","source_table":"src.citizen_cases","load_type":"incremental","watermark_column":"last_modified","primary_keys":"case_id","dedupe_order_by":"last_modified DESC, _ingested_at DESC","bronze_table":"ams_citizen_cases","silver_table":"citizen_case","silver_write_mode":"merge_scd1","load_group":"transactional","priority":20,"sla_hours":26,"data_owner":"Citizen Services","sensitivity":"Internal","active":true,"description":"Citizen cases - incremental by last_modified; can arrive before their public asset (late-arriving).","fabric_alternative":"Copy job incremental or mirroring; keep the watermark query if the source is on-prem"},{"object_id":"GV_IOT_SENSOR_READINGS","industry":"government","source_system":"iot","source_object":"sensor_readings","source_type":"file","ingest_method":"landed_file","landing_format":"csv","landing_path":"landing/iot/sensor_readings","read_options":"{\"header\": true, \"delimiter\": \",\"}","source_table":null,"load_type":"incremental","watermark_column":null,"primary_keys":"asset_id,reading_ts","dedupe_order_by":"_ingested_at DESC","bronze_table":"iot_sensor_readings","silver_table":"sensor_reading","silver_write_mode":"insert_only","load_group":"telemetry","priority":30,"sla_hours":2,"data_owner":"Smart Infrastructure","sensitivity":"Internal","active":true,"description":"Smart infrastructure sensor readings every 15 minutes, four micro-batch files a day. Insert-only and idempotent.","fabric_alternative":"Eventstream into Eventhouse for real time; keep this batch path for history and reconciliation"},{"object_id":"GV_CRM_CITIZENS","industry":"government","source_system":"crm","source_object":"citizens","source_type":"api","ingest_method":"landed_file","landing_format":"json","landing_path":"landing/crm/citizens","read_options":"{\"multiline\": true, \"json_root\": \"records\"}","source_table":null,"load_type":"full","watermark_column":null,"primary_keys":"citizen_id","dedupe_order_by":"last_modified_ts DESC, _ingested_at DESC","bronze_table":"crm_citizens","silver_table":"citizen","silver_write_mode":"merge_scd1","load_group":"reference","priority":15,"sla_hours":26,"data_owner":"Citizen Services","sensitivity":"Confidential","active":true,"description":"Citizens - full daily snapshot from the CRM API (contains PII). Missing rows are soft-deleted.","fabric_alternative":"Copy job with the CRM connector; Dataverse-based CRMs can use Link to Fabric instead of copying"}],"column_map":[{"object_id":"GV_AMS_PUBLIC_ASSETS","column_order":0,"source_column":"asset_id","target_column":"asset_id","data_type":"int","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"GV_AMS_PUBLIC_ASSETS","column_order":1,"source_column":"asset_ref","target_column":"asset_ref","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"GV_AMS_PUBLIC_ASSETS","column_order":2,"source_column":"asset_name","target_column":"asset_name","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"GV_AMS_PUBLIC_ASSETS","column_order":3,"source_column":"asset_type","target_column":"asset_type","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"GV_AMS_PUBLIC_ASSETS","column_order":4,"source_column":"district","target_column":"district","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"GV_AMS_PUBLIC_ASSETS","column_order":5,"source_column":"status","target_column":"status","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"GV_AMS_PUBLIC_ASSETS","column_order":6,"source_column":"criticality","target_column":"criticality","data_type":"int","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"GV_AMS_PUBLIC_ASSETS","column_order":7,"source_column":"replacement_value_kaud","target_column":"replacement_value_kaud","data_type":"double","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"GV_AMS_PUBLIC_ASSETS","column_order":8,"source_column":"constructed_date","target_column":"constructed_date","data_type":"date","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"GV_AMS_PUBLIC_ASSETS","column_order":9,"source_column":"last_modified","target_column":"last_modified","data_type":"timestamp","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"GV_AMS_CITIZEN_CASES","column_order":0,"source_column":"case_id","target_column":"case_id","data_type":"int","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"GV_AMS_CITIZEN_CASES","column_order":1,"source_column":"asset_id","target_column":"asset_id","data_type":"int","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"GV_AMS_CITIZEN_CASES","column_order":2,"source_column":"case_type","target_column":"case_type","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"GV_AMS_CITIZEN_CASES","column_order":3,"source_column":"priority","target_column":"priority","data_type":"int","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"GV_AMS_CITIZEN_CASES","column_order":4,"source_column":"status","target_column":"status","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"GV_AMS_CITIZEN_CASES","column_order":5,"source_column":"lodged_ts","target_column":"lodged_ts","data_type":"timestamp","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"GV_AMS_CITIZEN_CASES","column_order":6,"source_column":"completed_ts","target_column":"completed_ts","data_type":"timestamp","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"GV_AMS_CITIZEN_CASES","column_order":7,"source_column":"cost_aud","target_column":"cost_aud","data_type":"decimal(12,2)","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"GV_AMS_CITIZEN_CASES","column_order":8,"source_column":"crew_hours","target_column":"crew_hours","data_type":"double","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"GV_AMS_CITIZEN_CASES","column_order":9,"source_column":"description","target_column":"description","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"GV_AMS_CITIZEN_CASES","column_order":10,"source_column":"last_modified","target_column":"last_modified","data_type":"timestamp","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"GV_IOT_SENSOR_READINGS","column_order":0,"source_column":"asset_id","target_column":"asset_id","data_type":"int","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"GV_IOT_SENSOR_READINGS","column_order":1,"source_column":"reading_time","target_column":"reading_ts","data_type":"timestamp","transform_expr":"to_timestamp(reading_time, 'dd/MM/yyyy HH:mm')","pii_treatment":"none","sensitivity":"none","description":""},{"object_id":"GV_IOT_SENSOR_READINGS","column_order":2,"source_column":"water_level_cm","target_column":"water_level_cm","data_type":"double","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"GV_IOT_SENSOR_READINGS","column_order":3,"source_column":"traffic_count","target_column":"traffic_count","data_type":"double","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"GV_IOT_SENSOR_READINGS","column_order":4,"source_column":"pm25_ugm3","target_column":"pm25_ugm3","data_type":"double","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"GV_CRM_CITIZENS","column_order":0,"source_column":"Id","target_column":"citizen_id","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"GV_CRM_CITIZENS","column_order":1,"source_column":"FirstName","target_column":"first_name","data_type":"string","transform_expr":null,"pii_treatment":"drop","sensitivity":"Confidential","description":""},{"object_id":"GV_CRM_CITIZENS","column_order":2,"source_column":"LastName","target_column":"last_name","data_type":"string","transform_expr":null,"pii_treatment":"drop","sensitivity":"Confidential","description":""},{"object_id":"GV_CRM_CITIZENS","column_order":3,"source_column":"Email","target_column":"email_masked","data_type":"string","transform_expr":null,"pii_treatment":"mask_email","sensitivity":"Confidential","description":""},{"object_id":"GV_CRM_CITIZENS","column_order":4,"source_column":"MobilePhone","target_column":"mobile_masked","data_type":"string","transform_expr":null,"pii_treatment":"mask_phone","sensitivity":"Confidential","description":""},{"object_id":"GV_CRM_CITIZENS","column_order":5,"source_column":"Birthdate","target_column":"birth_year","data_type":"int","transform_expr":null,"pii_treatment":"year_only","sensitivity":"Confidential","description":""},{"object_id":"GV_CRM_CITIZENS","column_order":6,"source_column":"MailingPostalCode","target_column":"postcode","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"GV_CRM_CITIZENS","column_order":7,"source_column":"District__c","target_column":"district","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"GV_CRM_CITIZENS","column_order":8,"source_column":"CitizenType__c","target_column":"citizen_type","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"GV_CRM_CITIZENS","column_order":9,"source_column":"AssetId__c","target_column":"asset_id","data_type":"int","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"GV_CRM_CITIZENS","column_order":10,"source_column":"Status__c","target_column":"status","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"GV_CRM_CITIZENS","column_order":11,"source_column":"ConcessionCardNo__c","target_column":"concession_card_no_hash","data_type":"string","transform_expr":null,"pii_treatment":"hash","sensitivity":"Restricted","description":""},{"object_id":"GV_CRM_CITIZENS","column_order":12,"source_column":"LastModifiedDate","target_column":"last_modified_ts","data_type":"timestamp","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""}],"dq_rule":[{"rule_id":"GV-ENT-01","object_id":"GV_AMS_PUBLIC_ASSETS","rule_name":"Public asset id present","rule_category":"technical","check_type":"not_null","column_name":"asset_id","params":"{}","severity":"fail","fail_threshold_pct":0,"rule_owner":"Data engineering","description":"Primary key must never be null","active":true},{"rule_id":"GV-ENT-02","object_id":"GV_AMS_PUBLIC_ASSETS","rule_name":"Known asset_type","rule_category":"technical","check_type":"allowed_values","column_name":"asset_type","params":"{\"values\": [\"ROAD_SEGMENT\", \"BRIDGE\", \"PUBLIC_BUILDING\", \"STORMWATER_PIT\", \"STREETLIGHT\"]}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Data engineering","description":"Unknown classes are kept but flagged for the data owner","active":true},{"rule_id":"GV-ENT-03","object_id":"GV_AMS_PUBLIC_ASSETS","rule_name":"criticality between 1 and 5","rule_category":"technical","check_type":"range","column_name":"criticality","params":"{\"min\": 1, \"max\": 5}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Data engineering","description":"Out-of-range ratings usually mean a keying error in the source","active":true},{"rule_id":"GV-ENT-04","object_id":"GV_AMS_PUBLIC_ASSETS","rule_name":"Known status","rule_category":"business","check_type":"allowed_values","column_name":"status","params":"{\"values\": [\"IN_SERVICE\", \"UNDER_REPAIR\", \"CLOSED\", \"DISPOSED\"]}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Infrastructure Services","description":"Status list agreed with Infrastructure Services","active":true},{"rule_id":"GV-EVT-01","object_id":"GV_AMS_CITIZEN_CASES","rule_name":"Citizen case id present","rule_category":"technical","check_type":"not_null","column_name":"case_id","params":"{}","severity":"fail","fail_threshold_pct":0,"rule_owner":"Data engineering","description":"Primary key must never be null","active":true},{"rule_id":"GV-EVT-02","object_id":"GV_AMS_CITIZEN_CASES","rule_name":"Public asset exists","rule_category":"technical","check_type":"referential","column_name":"asset_id","params":"{\"ref_table\": \"silver.public_asset\", \"ref_column\": \"asset_id\"}","severity":"quarantine","fail_threshold_pct":null,"rule_owner":"Data engineering","description":"Held until the public asset arrives (late-arriving reference data), then merged automatically","active":true},{"rule_id":"GV-EVT-03","object_id":"GV_AMS_CITIZEN_CASES","rule_name":"Completed after lodged","rule_category":"business","check_type":"expression","column_name":null,"params":"{\"expr\": \"completed_ts IS NULL OR completed_ts >= lodged_ts\"}","severity":"quarantine","fail_threshold_pct":null,"rule_owner":"Citizen Services","description":"A citizen case can't be completed before it was lodged","active":true},{"rule_id":"GV-EVT-04","object_id":"GV_AMS_CITIZEN_CASES","rule_name":"cost_aud not negative","rule_category":"technical","check_type":"range","column_name":"cost_aud","params":"{\"min\": 0}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Data engineering","description":"Negative values are sign errors in the source extract","active":true},{"rule_id":"GV-EVT-05","object_id":"GV_AMS_CITIZEN_CASES","rule_name":"FLOOD_DRAINAGE is priority 1","rule_category":"business","check_type":"expression","column_name":null,"params":"{\"expr\": \"case_type <> 'FLOOD_DRAINAGE' OR priority = 1\"}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Citizen Services","description":"Flooding and drainage citizen cases must be raised as priority 1","active":true},{"rule_id":"GV-EVT-06","object_id":"GV_AMS_CITIZEN_CASES","rule_name":"lodged_ts parses","rule_category":"technical","check_type":"not_null","column_name":"lodged_ts","params":"{}","severity":"fail","fail_threshold_pct":5,"rule_owner":"Data engineering","description":"If more than 5% of timestamps fail to parse, the extract format has changed - stop the load","active":true},{"rule_id":"GV-RDG-01","object_id":"GV_IOT_SENSOR_READINGS","rule_name":"Water level plausible","rule_category":"technical","check_type":"range","column_name":"water_level_cm","params":"{\"min\": 0, \"max\": 500}","severity":"quarantine","fail_threshold_pct":null,"rule_owner":"Data engineering","description":"Sensor fault values (e.g. 999.9) are quarantined, never averaged into Gold","active":true},{"rule_id":"GV-RDG-02","object_id":"GV_IOT_SENSOR_READINGS","rule_name":"Public asset known","rule_category":"technical","check_type":"referential","column_name":"asset_id","params":"{\"ref_table\": \"silver.public_asset\", \"ref_column\": \"asset_id\"}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Data engineering","description":"Readings for unknown tags are flagged","active":true},{"rule_id":"GV-RDG-03","object_id":"GV_IOT_SENSOR_READINGS","rule_name":"Reading time parses","rule_category":"technical","check_type":"not_null","column_name":"reading_ts","params":"{}","severity":"fail","fail_threshold_pct":5,"rule_owner":"Data engineering","description":"Timestamp format change stops the load","active":true},{"rule_id":"GV-PTY-01","object_id":"GV_CRM_CITIZENS","rule_name":"Citizen id present","rule_category":"technical","check_type":"not_null","column_name":"citizen_id","params":"{}","severity":"fail","fail_threshold_pct":0,"rule_owner":"Data engineering","description":"Primary key must never be null","active":true},{"rule_id":"GV-PTY-02","object_id":"GV_CRM_CITIZENS","rule_name":"Email looks valid","rule_category":"technical","check_type":"regex","column_name":"email_masked","params":"{\"pattern\": \"^[^@\\\\s]+@[^@\\\\s]+\\\\.[^@\\\\s]+$\"}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Data engineering","description":"Checked on the raw value before masking","active":true},{"rule_id":"GV-PTY-03","object_id":"GV_CRM_CITIZENS","rule_name":"Australian postcode","rule_category":"technical","check_type":"regex","column_name":"postcode","params":"{\"pattern\": \"^\\\\d{4}$\"}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Data engineering","description":"Four digits","active":true},{"rule_id":"GV-PTY-04","object_id":"GV_CRM_CITIZENS","rule_name":"Known citizen_type","rule_category":"business","check_type":"allowed_values","column_name":"citizen_type","params":"{\"values\": [\"Resident\", \"Business\", \"Concession holder\", \"Visitor\"]}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Citizen Services","description":"Segments agreed with Citizen Services","active":true}],"gold_transform":[{"transform_id":"GV-G01","industry":"government","gold_table":"dim_date","run_order":100,"sql_text":"SELECT CAST(date_format(d, 'yyyyMMdd') AS INT) AS date_key, d AS calendar_date, year(d) AS calendar_year, quarter(d) AS calendar_quarter, month(d) AS month_number, date_format(d, 'MMM') AS month_name, date_format(d, 'yyyy-MM') AS year_month, dayofmonth(d) AS day_of_month, date_format(d, 'EEE') AS day_name, CASE WHEN month(d) >= 7 THEN year(d) + 1 ELSE year(d) END AS fy_year, concat('FY', substring(CAST(CASE WHEN month(d) >= 7 THEN year(d) + 1 ELSE year(d) END AS STRING), 3, 2)) AS fy_label, CASE WHEN dayofweek(d) IN (1, 7) THEN true ELSE false END AS is_weekend FROM (SELECT explode(sequence(to_date('2020-01-01'), to_date('2028-12-31'), interval 1 day)) AS d) dates","description":"Calendar with Australian financial year","business_owner":"Data engineering","consumers":"Power BI / Direct Lake, data agent","active":true},{"transform_id":"GV-G02","industry":"government","gold_table":"dim_public_asset","run_order":110,"sql_text":"SELECT asset_id, asset_ref, asset_name, asset_type, district, status, criticality, replacement_value_kaud, constructed_date, CAST(floor(months_between(current_date(), constructed_date) / 12) AS INT) AS age_years, _valid_from AS version_valid_from FROM silver.public_asset WHERE _is_current = true","description":"Current view of the public asset register (history stays in Silver SCD2)","business_owner":"Infrastructure Services","consumers":"Power BI / Direct Lake, data agent","active":true},{"transform_id":"GV-G03","industry":"government","gold_table":"fact_citizen_case","run_order":120,"sql_text":"SELECT v.case_id, v.asset_id, v.case_type, v.priority, v.status, v.lodged_ts, v.completed_ts, to_date(v.lodged_ts) AS opened_date, CAST(date_format(v.lodged_ts, 'yyyyMMdd') AS INT) AS opened_date_key, ROUND((unix_timestamp(v.completed_ts) - unix_timestamp(v.lodged_ts)) / 3600.0, 2) AS hours_to_close, CASE WHEN v.status IN ('LODGED', 'ASSIGNED') THEN 1 ELSE 0 END AS is_open, v.cost_aud, v.crew_hours, d.district, d.asset_type, d.criticality FROM silver.citizen_case v LEFT JOIN silver.public_asset d ON d.asset_id = v.asset_id AND d._is_current = true WHERE v._is_deleted = false","description":"Citizen cases with duration and public asset attributes","business_owner":"Citizen Services","consumers":"Power BI / Direct Lake, data agent","active":true},{"transform_id":"GV-G04","industry":"government","gold_table":"fact_sensor_hourly","run_order":130,"sql_text":"SELECT asset_id, date_trunc('HOUR', reading_ts) AS reading_hour, to_date(reading_ts) AS reading_date, CAST(date_format(reading_ts, 'yyyyMMdd') AS INT) AS reading_date_key, COUNT(*) AS readings, ROUND(AVG(water_level_cm), 2) AS avg_water_level_cm, ROUND(MAX(water_level_cm), 2) AS max_water_level_cm, ROUND(AVG(traffic_count), 2) AS avg_traffic_count, ROUND(AVG(pm25_ugm3), 2) AS avg_pm25_ugm3, SUM(CASE WHEN water_level_cm > 150 THEN 1 ELSE 0 END) AS breach_readings FROM silver.sensor_reading GROUP BY asset_id, date_trunc('HOUR', reading_ts), to_date(reading_ts), CAST(date_format(reading_ts, 'yyyyMMdd') AS INT)","description":"Hourly smart infrastructure sensor readings with threshold breaches","business_owner":"Smart Infrastructure","consumers":"Power BI / Direct Lake, data agent","active":true},{"transform_id":"GV-G05","industry":"government","gold_table":"public_asset_health_scorecard","run_order":200,"sql_text":"WITH asat AS (SELECT MAX(to_date(lodged_ts)) AS as_at FROM silver.citizen_case), ev AS (SELECT v.asset_id, COUNT(*) AS citizen_cases_12m, SUM(CASE WHEN v.status IN ('LODGED', 'ASSIGNED') THEN 1 ELSE 0 END) AS open_citizen_cases, SUM(CASE WHEN v.case_type = 'FLOOD_DRAINAGE' THEN 1 ELSE 0 END) AS urgent_citizen_cases_12m, ROUND(SUM(v.cost_aud), 2) AS cost_aud_12m FROM silver.citizen_case v CROSS JOIN asat WHERE v._is_deleted = false AND to_date(v.lodged_ts) > add_months(asat.as_at, -12) GROUP BY v.asset_id), rd AS (SELECT asset_id, MAX(water_level_cm) AS max_water_level_cm, SUM(CASE WHEN water_level_cm > 150 THEN 1 ELSE 0 END) AS breach_readings FROM silver.sensor_reading GROUP BY asset_id), s AS (SELECT d.asset_id, d.asset_ref, d.asset_name, d.asset_type, d.district, d.status, d.criticality, COALESCE(ev.citizen_cases_12m, 0) AS citizen_cases_12m, COALESCE(ev.open_citizen_cases, 0) AS open_citizen_cases, COALESCE(ev.urgent_citizen_cases_12m, 0) AS urgent_citizen_cases_12m, COALESCE(ev.cost_aud_12m, 0) AS cost_aud_12m, rd.max_water_level_cm, COALESCE(rd.breach_readings, 0) AS breach_readings, d.replacement_value_kaud FROM silver.public_asset d LEFT JOIN ev ON ev.asset_id = d.asset_id LEFT JOIN rd ON rd.asset_id = d.asset_id WHERE d._is_current = true) SELECT s.*, ROUND(LEAST(100, COALESCE(s.criticality, 1) * 8 + s.open_citizen_cases * 6 + s.urgent_citizen_cases_12m * 4 + LEAST(s.breach_readings, 20) * 1.5), 1) AS risk_score, CASE WHEN LEAST(100, COALESCE(s.criticality, 1) * 8 + s.open_citizen_cases * 6 + s.urgent_citizen_cases_12m * 4 + LEAST(s.breach_readings, 20) * 1.5) >= 60 THEN 'High' WHEN LEAST(100, COALESCE(s.criticality, 1) * 8 + s.open_citizen_cases * 6 + s.urgent_citizen_cases_12m * 4 + LEAST(s.breach_readings, 20) * 1.5) >= 35 THEN 'Medium' ELSE 'Low' END AS risk_band, ROUND(s.open_citizen_cases * 2500 + LEAST(s.breach_readings, 20) * 1200 + COALESCE(s.replacement_value_kaud, 0) * COALESCE(s.criticality, 1) * 0.5, 0) AS cost_at_risk_aud FROM s","description":"Public asset risk score - citizen cases, urgent work and water level breaches, plus cost at risk in AUD (weights and dollar rates are illustrative: $2,500 per open case + $1,200 per 15-min window above the water-level threshold + replacement value (k) x criticality x $0.50)","business_owner":"Infrastructure Services","consumers":"Power BI / Direct Lake, data agent","active":true},{"transform_id":"GV-G06","industry":"government","gold_table":"citizen_summary","run_order":210,"sql_text":"SELECT district, citizen_type, COUNT(*) AS citizens, SUM(CASE WHEN status = 'Active' THEN 1 ELSE 0 END) AS active_citizens, ROUND(AVG(year(current_date()) - birth_year), 1) AS avg_age, COUNT(DISTINCT asset_id) AS linked_public_assets FROM silver.citizen WHERE _is_deleted = false GROUP BY district, citizen_type","description":"Citizens by district and citizen_type","business_owner":"Citizen Services","consumers":"Power BI / Direct Lake, data agent","active":true}],"onboard":{"object":{"object_id":"GV_IOT_RAINFALL","industry":"government","source_system":"iot","source_object":"rainfall","source_type":"file","ingest_method":"landed_file","landing_format":"txt","landing_path":"landing/iot/rainfall","read_options":"{\"header\": true, \"delimiter\": \"|\"}","source_table":null,"load_type":"incremental","watermark_column":null,"primary_keys":"district,obs_date","dedupe_order_by":"_ingested_at DESC","bronze_table":"iot_rainfall","silver_table":"rainfall_daily","silver_write_mode":"merge_scd1","load_group":"reference","priority":25,"sla_hours":26,"data_owner":"Emergency Management","sensitivity":"Public","active":true,"description":"Daily BoM rainfall by district (pipe-delimited, dd/MM/yyyy) - onboarded live in the demo","fabric_alternative":"Public or partner feed - could be a REST copy, or a shortcut if the provider shares a folder"},"columns":[["district","district","string",null],["obs_date","obs_date","date","to_date(obs_date, 'dd/MM/yyyy')"],["rainfall_mm","rainfall_mm","double",null],["max_gust_kmh","max_gust_kmh","double",null],["river_height_m","river_height_m","double",null],["warning_level","warning_level","string",null]],"rules":[["GV-EXT-01","rainfall_mm plausible","technical","range","rainfall_mm","{\"min\": 0, \"max\": 500}","warn"],["GV-EXT-02","Known warning_level","business","allowed_values","warning_level","{\"values\": [\"None\", \"Advice\", \"Watch and act\", \"Emergency\"]}","warn"]],"gold":{"transform_id":"GV-G07","industry":"government","gold_table":"citizen_case_vs_rainfall_daily","run_order":220,"sql_text":"WITH ev AS (SELECT d.district, to_date(v.lodged_ts) AS obs_date, COUNT(*) AS citizen_cases, SUM(CASE WHEN v.case_type = 'FLOOD_DRAINAGE' THEN 1 ELSE 0 END) AS urgent_citizen_cases FROM silver.citizen_case v JOIN silver.public_asset d ON d.asset_id = v.asset_id AND d._is_current = true WHERE v._is_deleted = false GROUP BY d.district, to_date(v.lodged_ts)) SELECT x.district, x.obs_date, x.rainfall_mm, x.max_gust_kmh, x.river_height_m, x.warning_level, COALESCE(ev.citizen_cases, 0) AS citizen_cases, COALESCE(ev.urgent_citizen_cases, 0) AS urgent_citizen_cases FROM silver.rainfall_daily x LEFT JOIN ev ON ev.district = x.district AND ev.obs_date = x.obs_date WHERE x._is_deleted = false","description":"Rainfall and river height against flooding cases by district","business_owner":"Emergency Management","consumers":"Power BI / Direct Lake","active":true}}},"retail":{"pack":{"id":"retail","industry":"Retail","org":"Wattle & Co","short":"WCO","prefix":"RT","tagline":"national supermarket and general merchandise retailer","email_domain":"wattle.example.com.au","region_col":"state","region_label":"state","regions":["NSW","VIC","QLD","WA","SA","TAS","ACT"],"name_words":["Bondi","Chadstone","Chermside","Joondalup","Marion","Hobart","Belconnen","Penrith","Werribee","Southport"],"sys":{"ops":{"code":"ret","name":"Retail back office, inventory and replenishment (on-prem SQL Server)","type":"Database","owner":"Retail Systems","fabric":"Mirroring for SQL Server vs Copy job incremental"},"tel":{"code":"pos","name":"Store POS and stock-on-hand feed (15-min CSV) and promotions calendar","type":"Store file drop","owner":"Store Operations","fabric":"Eventstream for real-time POS / IoT; files to OneLake for batch"},"crm":{"code":"loy","name":"Loyalty platform (Salesforce Marketing Cloud)","type":"SaaS","owner":"Loyalty & Marketing","fabric":"Copy job with the Salesforce connector"}},"entity":{"table":"store","src":"stores","label":"store","plural":"stores","id":"store_id","code":"store_code","code_prefix":"ST","name_col":"store_name","type_col":"store_format","types":["SUPERMARKET","EXPRESS","MEGASTORE","DARK_STORE","PHARMACY"],"status":["TRADING","REFURBISHING","TEMP_CLOSED","CLOSED"],"tier_col":"store_tier","tier_label":"store tier (1-5, 5 = flagship)","cap_col":"weekly_sales_kaud","cap_label":"average weekly sales (AUD thousands)","cap_range":[150,2500],"start_col":"open_date","start_years":[1980,2025],"drift_col":"area_manager","drift_values":["Metro North","Metro South","Regional"],"owner":"Store Operations","count":240},"event":{"table":"stock_issue","src":"stock_issues","label":"stock issue","plural":"stock issues","id":"issue_id","id_start":200001,"type_col":"issue_type","types":["OUT_OF_STOCK","SHORT_DELIVERY","COLD_CHAIN","PRICE_ERROR"],"type_names":["Out of stock","Short delivery","Cold-chain breach","Price or label error"],"urgent_type":"OUT_OF_STOCK","status":["RAISED","ACTIONING","RESOLVED","CANCELLED"],"open_ts":"raised_ts","close_ts":"resolved_ts","open_verb":"raised","close_verb":"resolved","amount_col":"lost_sales_aud","amount_range":[0,15000],"qty_col":"units_affected","qty_range":[1,600],"descriptions":["Promo line sold out by 10am","Supplier short-shipped milk","Freezer door left open overnight","Shelf label price mismatch","DC pick error on bread","Gap on shelf at peak","Late chilled delivery","Online substitution spike"],"owner":"Supply Chain"},"reading":{"table":"store_trading","src":"store_trading","label":"store trading and availability (15-min)","fact":"store_trading","monitored":24,"monitored_types":["SUPERMARKET","MEGASTORE","EXPRESS"],"metrics":[{"col":"out_of_stock_pct","label":"out-of-stock lines","unit":"%","base":3,"sd":1.2,"min":0,"max":100,"threshold":8},{"col":"sales_aud","label":"sales","unit":"AUD","base":4200,"sd":1500,"min":0,"max":500000},{"col":"footfall","label":"footfall","unit":"people","base":180,"sd":70,"min":0,"max":20000}],"sentinel":999.9,"owner":"Store Operations"},"party":{"table":"loyalty_member","src":"loyalty_members","label":"loyalty member","plural":"loyalty members","id":"member_id","id_prefix":"LM","sf_type":"contact","segment_col":"tier","segments":["Gold","Silver","Bronze","Staff"],"status":["Active","Lapsed"],"sensitive_col":"card_token","sensitive_label":"payment card token","sensitive_treatment":"hash","owner":"Loyalty & Marketing","count":300},"extra":{"table":"promotion_day","src":"promotions","label":"promotions calendar","sys":"tel","cols":[{"col":"promo_lines","base":120,"sd":40,"min":0,"max":5000},{"col":"avg_discount_pct","base":22,"sd":8,"min":0,"max":90},{"col":"catalogue_reach_k","base":300,"sd":90,"min":0,"max":10000}],"cat_col":"promo_type","cat_values":["Weekly specials","Half price","Seasonal","Clearance"],"owner":"Merchandise","story":"promotions against out-of-stocks by state"},"alert":{"stream":"store_alert","label":"store alert","codes":["OUT_OF_STOCK_SPIKE","FREEZER_FAULT","TILL_OFFLINE"],"storm":"half-price promo rush"},"ref":{"title":"Stock issue response targets","owner":"Supply Chain","rows":[["OUT_OF_STOCK","Availability",4,"Store Replenishment"],["SHORT_DELIVERY","Supply",24,"DC Operations"],["COLD_CHAIN","Food safety",2,"Store Operations"],["PRICE_ERROR","Compliance",8,"Pricing"]]},"impact":{"col":"lost_sales_at_risk_aud","label":"lost sales at risk","open_cost":1800,"breach_cost":900,"cap_rate":1,"rule":"$1,800 per open stock issue + $900 per 15-min window above 8% out-of-stock + weekly sales (k) x store tier x $1"},"ai":{"persona":"store operations, availability and supply chain","cust":"supermarkets, general merchandise and specialty retailers","q_model":["Which state lost the most sales to stock issues this month?","Show lost sales by issue type and month.","Which flagship stores (tier 4 or 5) have open out-of-stock issues?"],"q_agent":["Which 10 stores have the most lost sales at risk, and how many stock issues are still open?","Compare average hours to resolve out-of-stock issues by state against the 4-hour target.","Which stores had more than 8% of lines out of stock in the last 3 days?","On half-price promotion days, do we see more out-of-stocks?","How many Gold loyalty members are active in each state?"],"q_onto":["Show supermarkets in NSW with an open out-of-stock issue and a high risk band.","For the store with the worst availability, list its open stock issues and the lost sales.","Which loyalty members shop at stores with an open cold-chain breach?"]}},"source_system":[{"source_system":"ret","industry":"retail","display_name":"Retail back office, inventory and replenishment (on-prem SQL Server)","system_type":"Database","connection_pattern":"On-premises data gateway -> pipeline Copy with watermark query","technical_owner":"Retail Systems","fabric_options":"Mirroring for SQL Server vs Copy job incremental"},{"source_system":"pos","industry":"retail","display_name":"Store POS and stock-on-hand feed (15-min CSV) and promotions calendar","system_type":"Store file drop","connection_pattern":"Files dropped to OneLake (Files/landing) every 15 minutes","technical_owner":"Store Operations","fabric_options":"Eventstream for real-time POS / IoT; files to OneLake for batch"},{"source_system":"loy","industry":"retail","display_name":"Loyalty platform (Salesforce Marketing Cloud)","system_type":"SaaS","connection_pattern":"REST/OData API -> JSON landed per day","technical_owner":"Loyalty & Marketing","fabric_options":"Copy job with the Salesforce connector"}],"ingestion_object":[{"object_id":"RT_RET_STORES","industry":"retail","source_system":"ret","source_object":"stores","source_type":"database","ingest_method":"copy_db","landing_format":"csv","landing_path":"landing/ret/stores","read_options":"{\"header\": true, \"delimiter\": \",\"}","source_table":"src.stores","load_type":"incremental","watermark_column":"last_modified","primary_keys":"store_id","dedupe_order_by":"last_modified DESC, _ingested_at DESC","bronze_table":"ret_stores","silver_table":"store","silver_write_mode":"merge_scd2","load_group":"reference","priority":10,"sla_hours":26,"data_owner":"Store Operations","sensitivity":"Internal","active":true,"description":"Store register. SCD2 keeps status, store_tier and weekly_sales_kaud history.","fabric_alternative":"Strong mirroring candidate - small, slowly changing, needed in near real time by operations"},{"object_id":"RT_RET_STOCK_ISSUES","industry":"retail","source_system":"ret","source_object":"stock_issues","source_type":"database","ingest_method":"copy_db","landing_format":"csv","landing_path":"landing/ret/stock_issues","read_options":"{\"header\": true, \"delimiter\": \",\"}","source_table":"src.stock_issues","load_type":"incremental","watermark_column":"last_modified","primary_keys":"issue_id","dedupe_order_by":"last_modified DESC, _ingested_at DESC","bronze_table":"ret_stock_issues","silver_table":"stock_issue","silver_write_mode":"merge_scd1","load_group":"transactional","priority":20,"sla_hours":26,"data_owner":"Supply Chain","sensitivity":"Internal","active":true,"description":"Stock issues - incremental by last_modified; can arrive before their store (late-arriving).","fabric_alternative":"Copy job incremental or mirroring; keep the watermark query if the source is on-prem"},{"object_id":"RT_POS_STORE_TRADING","industry":"retail","source_system":"pos","source_object":"store_trading","source_type":"file","ingest_method":"landed_file","landing_format":"csv","landing_path":"landing/pos/store_trading","read_options":"{\"header\": true, \"delimiter\": \",\"}","source_table":null,"load_type":"incremental","watermark_column":null,"primary_keys":"store_id,reading_ts","dedupe_order_by":"_ingested_at DESC","bronze_table":"pos_store_trading","silver_table":"store_trading","silver_write_mode":"insert_only","load_group":"telemetry","priority":30,"sla_hours":2,"data_owner":"Store Operations","sensitivity":"Internal","active":true,"description":"Store trading and availability (15-min) every 15 minutes, four micro-batch files a day. Insert-only and idempotent.","fabric_alternative":"Eventstream into Eventhouse for real time; keep this batch path for history and reconciliation"},{"object_id":"RT_LOY_LOYALTY_MEMBERS","industry":"retail","source_system":"loy","source_object":"loyalty_members","source_type":"api","ingest_method":"landed_file","landing_format":"json","landing_path":"landing/loy/loyalty_members","read_options":"{\"multiline\": true, \"json_root\": \"records\"}","source_table":null,"load_type":"full","watermark_column":null,"primary_keys":"member_id","dedupe_order_by":"last_modified_ts DESC, _ingested_at DESC","bronze_table":"loy_loyalty_members","silver_table":"loyalty_member","silver_write_mode":"merge_scd1","load_group":"reference","priority":15,"sla_hours":26,"data_owner":"Loyalty & Marketing","sensitivity":"Confidential","active":true,"description":"Loyalty members - full daily snapshot from the CRM API (contains PII). Missing rows are soft-deleted.","fabric_alternative":"Copy job with the CRM connector; Dataverse-based CRMs can use Link to Fabric instead of copying"}],"column_map":[{"object_id":"RT_RET_STORES","column_order":0,"source_column":"store_id","target_column":"store_id","data_type":"int","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"RT_RET_STORES","column_order":1,"source_column":"store_code","target_column":"store_code","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"RT_RET_STORES","column_order":2,"source_column":"store_name","target_column":"store_name","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"RT_RET_STORES","column_order":3,"source_column":"store_format","target_column":"store_format","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"RT_RET_STORES","column_order":4,"source_column":"state","target_column":"state","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"RT_RET_STORES","column_order":5,"source_column":"status","target_column":"status","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"RT_RET_STORES","column_order":6,"source_column":"store_tier","target_column":"store_tier","data_type":"int","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"RT_RET_STORES","column_order":7,"source_column":"weekly_sales_kaud","target_column":"weekly_sales_kaud","data_type":"double","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"RT_RET_STORES","column_order":8,"source_column":"open_date","target_column":"open_date","data_type":"date","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"RT_RET_STORES","column_order":9,"source_column":"last_modified","target_column":"last_modified","data_type":"timestamp","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"RT_RET_STOCK_ISSUES","column_order":0,"source_column":"issue_id","target_column":"issue_id","data_type":"int","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"RT_RET_STOCK_ISSUES","column_order":1,"source_column":"store_id","target_column":"store_id","data_type":"int","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"RT_RET_STOCK_ISSUES","column_order":2,"source_column":"issue_type","target_column":"issue_type","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"RT_RET_STOCK_ISSUES","column_order":3,"source_column":"priority","target_column":"priority","data_type":"int","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"RT_RET_STOCK_ISSUES","column_order":4,"source_column":"status","target_column":"status","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"RT_RET_STOCK_ISSUES","column_order":5,"source_column":"raised_ts","target_column":"raised_ts","data_type":"timestamp","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"RT_RET_STOCK_ISSUES","column_order":6,"source_column":"resolved_ts","target_column":"resolved_ts","data_type":"timestamp","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"RT_RET_STOCK_ISSUES","column_order":7,"source_column":"lost_sales_aud","target_column":"lost_sales_aud","data_type":"decimal(12,2)","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"RT_RET_STOCK_ISSUES","column_order":8,"source_column":"units_affected","target_column":"units_affected","data_type":"double","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"RT_RET_STOCK_ISSUES","column_order":9,"source_column":"description","target_column":"description","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"RT_RET_STOCK_ISSUES","column_order":10,"source_column":"last_modified","target_column":"last_modified","data_type":"timestamp","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"RT_POS_STORE_TRADING","column_order":0,"source_column":"store_id","target_column":"store_id","data_type":"int","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"RT_POS_STORE_TRADING","column_order":1,"source_column":"reading_time","target_column":"reading_ts","data_type":"timestamp","transform_expr":"to_timestamp(reading_time, 'dd/MM/yyyy HH:mm')","pii_treatment":"none","sensitivity":"none","description":""},{"object_id":"RT_POS_STORE_TRADING","column_order":2,"source_column":"out_of_stock_pct","target_column":"out_of_stock_pct","data_type":"double","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"RT_POS_STORE_TRADING","column_order":3,"source_column":"sales_aud","target_column":"sales_aud","data_type":"double","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"RT_POS_STORE_TRADING","column_order":4,"source_column":"footfall","target_column":"footfall","data_type":"double","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"RT_LOY_LOYALTY_MEMBERS","column_order":0,"source_column":"Id","target_column":"member_id","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"RT_LOY_LOYALTY_MEMBERS","column_order":1,"source_column":"FirstName","target_column":"first_name","data_type":"string","transform_expr":null,"pii_treatment":"drop","sensitivity":"Confidential","description":""},{"object_id":"RT_LOY_LOYALTY_MEMBERS","column_order":2,"source_column":"LastName","target_column":"last_name","data_type":"string","transform_expr":null,"pii_treatment":"drop","sensitivity":"Confidential","description":""},{"object_id":"RT_LOY_LOYALTY_MEMBERS","column_order":3,"source_column":"Email","target_column":"email_masked","data_type":"string","transform_expr":null,"pii_treatment":"mask_email","sensitivity":"Confidential","description":""},{"object_id":"RT_LOY_LOYALTY_MEMBERS","column_order":4,"source_column":"MobilePhone","target_column":"mobile_masked","data_type":"string","transform_expr":null,"pii_treatment":"mask_phone","sensitivity":"Confidential","description":""},{"object_id":"RT_LOY_LOYALTY_MEMBERS","column_order":5,"source_column":"Birthdate","target_column":"birth_year","data_type":"int","transform_expr":null,"pii_treatment":"year_only","sensitivity":"Confidential","description":""},{"object_id":"RT_LOY_LOYALTY_MEMBERS","column_order":6,"source_column":"MailingPostalCode","target_column":"postcode","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"RT_LOY_LOYALTY_MEMBERS","column_order":7,"source_column":"State__c","target_column":"state","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"RT_LOY_LOYALTY_MEMBERS","column_order":8,"source_column":"Tier__c","target_column":"tier","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"RT_LOY_LOYALTY_MEMBERS","column_order":9,"source_column":"StoreId__c","target_column":"store_id","data_type":"int","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"RT_LOY_LOYALTY_MEMBERS","column_order":10,"source_column":"Status__c","target_column":"status","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"RT_LOY_LOYALTY_MEMBERS","column_order":11,"source_column":"CardToken__c","target_column":"card_token_hash","data_type":"string","transform_expr":null,"pii_treatment":"hash","sensitivity":"Restricted","description":""},{"object_id":"RT_LOY_LOYALTY_MEMBERS","column_order":12,"source_column":"LastModifiedDate","target_column":"last_modified_ts","data_type":"timestamp","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""}],"dq_rule":[{"rule_id":"RT-ENT-01","object_id":"RT_RET_STORES","rule_name":"Store id present","rule_category":"technical","check_type":"not_null","column_name":"store_id","params":"{}","severity":"fail","fail_threshold_pct":0,"rule_owner":"Data engineering","description":"Primary key must never be null","active":true},{"rule_id":"RT-ENT-02","object_id":"RT_RET_STORES","rule_name":"Known store_format","rule_category":"technical","check_type":"allowed_values","column_name":"store_format","params":"{\"values\": [\"SUPERMARKET\", \"EXPRESS\", \"MEGASTORE\", \"DARK_STORE\", \"PHARMACY\"]}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Data engineering","description":"Unknown classes are kept but flagged for the data owner","active":true},{"rule_id":"RT-ENT-03","object_id":"RT_RET_STORES","rule_name":"store_tier between 1 and 5","rule_category":"technical","check_type":"range","column_name":"store_tier","params":"{\"min\": 1, \"max\": 5}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Data engineering","description":"Out-of-range ratings usually mean a keying error in the source","active":true},{"rule_id":"RT-ENT-04","object_id":"RT_RET_STORES","rule_name":"Known status","rule_category":"business","check_type":"allowed_values","column_name":"status","params":"{\"values\": [\"TRADING\", \"REFURBISHING\", \"TEMP_CLOSED\", \"CLOSED\"]}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Store Operations","description":"Status list agreed with Store Operations","active":true},{"rule_id":"RT-EVT-01","object_id":"RT_RET_STOCK_ISSUES","rule_name":"Stock issue id present","rule_category":"technical","check_type":"not_null","column_name":"issue_id","params":"{}","severity":"fail","fail_threshold_pct":0,"rule_owner":"Data engineering","description":"Primary key must never be null","active":true},{"rule_id":"RT-EVT-02","object_id":"RT_RET_STOCK_ISSUES","rule_name":"Store exists","rule_category":"technical","check_type":"referential","column_name":"store_id","params":"{\"ref_table\": \"silver.store\", \"ref_column\": \"store_id\"}","severity":"quarantine","fail_threshold_pct":null,"rule_owner":"Data engineering","description":"Held until the store arrives (late-arriving reference data), then merged automatically","active":true},{"rule_id":"RT-EVT-03","object_id":"RT_RET_STOCK_ISSUES","rule_name":"Resolved after raised","rule_category":"business","check_type":"expression","column_name":null,"params":"{\"expr\": \"resolved_ts IS NULL OR resolved_ts >= raised_ts\"}","severity":"quarantine","fail_threshold_pct":null,"rule_owner":"Supply Chain","description":"A stock issue can't be resolved before it was raised","active":true},{"rule_id":"RT-EVT-04","object_id":"RT_RET_STOCK_ISSUES","rule_name":"lost_sales_aud not negative","rule_category":"technical","check_type":"range","column_name":"lost_sales_aud","params":"{\"min\": 0}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Data engineering","description":"Negative values are sign errors in the source extract","active":true},{"rule_id":"RT-EVT-05","object_id":"RT_RET_STOCK_ISSUES","rule_name":"OUT_OF_STOCK is priority 1","rule_category":"business","check_type":"expression","column_name":null,"params":"{\"expr\": \"issue_type <> 'OUT_OF_STOCK' OR priority = 1\"}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Supply Chain","description":"Out of stock stock issues must be raised as priority 1","active":true},{"rule_id":"RT-EVT-06","object_id":"RT_RET_STOCK_ISSUES","rule_name":"raised_ts parses","rule_category":"technical","check_type":"not_null","column_name":"raised_ts","params":"{}","severity":"fail","fail_threshold_pct":5,"rule_owner":"Data engineering","description":"If more than 5% of timestamps fail to parse, the extract format has changed - stop the load","active":true},{"rule_id":"RT-RDG-01","object_id":"RT_POS_STORE_TRADING","rule_name":"Out-of-stock lines plausible","rule_category":"technical","check_type":"range","column_name":"out_of_stock_pct","params":"{\"min\": 0, \"max\": 100}","severity":"quarantine","fail_threshold_pct":null,"rule_owner":"Data engineering","description":"Sensor fault values (e.g. 999.9) are quarantined, never averaged into Gold","active":true},{"rule_id":"RT-RDG-02","object_id":"RT_POS_STORE_TRADING","rule_name":"Store known","rule_category":"technical","check_type":"referential","column_name":"store_id","params":"{\"ref_table\": \"silver.store\", \"ref_column\": \"store_id\"}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Data engineering","description":"Readings for unknown tags are flagged","active":true},{"rule_id":"RT-RDG-03","object_id":"RT_POS_STORE_TRADING","rule_name":"Reading time parses","rule_category":"technical","check_type":"not_null","column_name":"reading_ts","params":"{}","severity":"fail","fail_threshold_pct":5,"rule_owner":"Data engineering","description":"Timestamp format change stops the load","active":true},{"rule_id":"RT-PTY-01","object_id":"RT_LOY_LOYALTY_MEMBERS","rule_name":"Loyalty member id present","rule_category":"technical","check_type":"not_null","column_name":"member_id","params":"{}","severity":"fail","fail_threshold_pct":0,"rule_owner":"Data engineering","description":"Primary key must never be null","active":true},{"rule_id":"RT-PTY-02","object_id":"RT_LOY_LOYALTY_MEMBERS","rule_name":"Email looks valid","rule_category":"technical","check_type":"regex","column_name":"email_masked","params":"{\"pattern\": \"^[^@\\\\s]+@[^@\\\\s]+\\\\.[^@\\\\s]+$\"}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Data engineering","description":"Checked on the raw value before masking","active":true},{"rule_id":"RT-PTY-03","object_id":"RT_LOY_LOYALTY_MEMBERS","rule_name":"Australian postcode","rule_category":"technical","check_type":"regex","column_name":"postcode","params":"{\"pattern\": \"^\\\\d{4}$\"}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Data engineering","description":"Four digits","active":true},{"rule_id":"RT-PTY-04","object_id":"RT_LOY_LOYALTY_MEMBERS","rule_name":"Known tier","rule_category":"business","check_type":"allowed_values","column_name":"tier","params":"{\"values\": [\"Gold\", \"Silver\", \"Bronze\", \"Staff\"]}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Loyalty & Marketing","description":"Segments agreed with Loyalty & Marketing","active":true}],"gold_transform":[{"transform_id":"RT-G01","industry":"retail","gold_table":"dim_date","run_order":100,"sql_text":"SELECT CAST(date_format(d, 'yyyyMMdd') AS INT) AS date_key, d AS calendar_date, year(d) AS calendar_year, quarter(d) AS calendar_quarter, month(d) AS month_number, date_format(d, 'MMM') AS month_name, date_format(d, 'yyyy-MM') AS year_month, dayofmonth(d) AS day_of_month, date_format(d, 'EEE') AS day_name, CASE WHEN month(d) >= 7 THEN year(d) + 1 ELSE year(d) END AS fy_year, concat('FY', substring(CAST(CASE WHEN month(d) >= 7 THEN year(d) + 1 ELSE year(d) END AS STRING), 3, 2)) AS fy_label, CASE WHEN dayofweek(d) IN (1, 7) THEN true ELSE false END AS is_weekend FROM (SELECT explode(sequence(to_date('2020-01-01'), to_date('2028-12-31'), interval 1 day)) AS d) dates","description":"Calendar with Australian financial year","business_owner":"Data engineering","consumers":"Power BI / Direct Lake, data agent","active":true},{"transform_id":"RT-G02","industry":"retail","gold_table":"dim_store","run_order":110,"sql_text":"SELECT store_id, store_code, store_name, store_format, state, status, store_tier, weekly_sales_kaud, open_date, CAST(floor(months_between(current_date(), open_date) / 12) AS INT) AS age_years, _valid_from AS version_valid_from FROM silver.store WHERE _is_current = true","description":"Current view of the store register (history stays in Silver SCD2)","business_owner":"Store Operations","consumers":"Power BI / Direct Lake, data agent","active":true},{"transform_id":"RT-G03","industry":"retail","gold_table":"fact_stock_issue","run_order":120,"sql_text":"SELECT v.issue_id, v.store_id, v.issue_type, v.priority, v.status, v.raised_ts, v.resolved_ts, to_date(v.raised_ts) AS opened_date, CAST(date_format(v.raised_ts, 'yyyyMMdd') AS INT) AS opened_date_key, ROUND((unix_timestamp(v.resolved_ts) - unix_timestamp(v.raised_ts)) / 3600.0, 2) AS hours_to_close, CASE WHEN v.status IN ('RAISED', 'ACTIONING') THEN 1 ELSE 0 END AS is_open, v.lost_sales_aud, v.units_affected, d.state, d.store_format, d.store_tier FROM silver.stock_issue v LEFT JOIN silver.store d ON d.store_id = v.store_id AND d._is_current = true WHERE v._is_deleted = false","description":"Stock issues with duration and store attributes","business_owner":"Supply Chain","consumers":"Power BI / Direct Lake, data agent","active":true},{"transform_id":"RT-G04","industry":"retail","gold_table":"fact_store_trading_hourly","run_order":130,"sql_text":"SELECT store_id, date_trunc('HOUR', reading_ts) AS reading_hour, to_date(reading_ts) AS reading_date, CAST(date_format(reading_ts, 'yyyyMMdd') AS INT) AS reading_date_key, COUNT(*) AS readings, ROUND(AVG(out_of_stock_pct), 2) AS avg_out_of_stock_pct, ROUND(MAX(out_of_stock_pct), 2) AS max_out_of_stock_pct, ROUND(AVG(sales_aud), 2) AS avg_sales_aud, ROUND(AVG(footfall), 2) AS avg_footfall, SUM(CASE WHEN out_of_stock_pct > 8 THEN 1 ELSE 0 END) AS breach_readings FROM silver.store_trading GROUP BY store_id, date_trunc('HOUR', reading_ts), to_date(reading_ts), CAST(date_format(reading_ts, 'yyyyMMdd') AS INT)","description":"Hourly store trading and availability (15-min) with threshold breaches","business_owner":"Store Operations","consumers":"Power BI / Direct Lake, data agent","active":true},{"transform_id":"RT-G05","industry":"retail","gold_table":"store_health_scorecard","run_order":200,"sql_text":"WITH asat AS (SELECT MAX(to_date(raised_ts)) AS as_at FROM silver.stock_issue), ev AS (SELECT v.store_id, COUNT(*) AS stock_issues_12m, SUM(CASE WHEN v.status IN ('RAISED', 'ACTIONING') THEN 1 ELSE 0 END) AS open_stock_issues, SUM(CASE WHEN v.issue_type = 'OUT_OF_STOCK' THEN 1 ELSE 0 END) AS urgent_stock_issues_12m, ROUND(SUM(v.lost_sales_aud), 2) AS lost_sales_aud_12m FROM silver.stock_issue v CROSS JOIN asat WHERE v._is_deleted = false AND to_date(v.raised_ts) > add_months(asat.as_at, -12) GROUP BY v.store_id), rd AS (SELECT store_id, MAX(out_of_stock_pct) AS max_out_of_stock_pct, SUM(CASE WHEN out_of_stock_pct > 8 THEN 1 ELSE 0 END) AS breach_readings FROM silver.store_trading GROUP BY store_id), s AS (SELECT d.store_id, d.store_code, d.store_name, d.store_format, d.state, d.status, d.store_tier, COALESCE(ev.stock_issues_12m, 0) AS stock_issues_12m, COALESCE(ev.open_stock_issues, 0) AS open_stock_issues, COALESCE(ev.urgent_stock_issues_12m, 0) AS urgent_stock_issues_12m, COALESCE(ev.lost_sales_aud_12m, 0) AS lost_sales_aud_12m, rd.max_out_of_stock_pct, COALESCE(rd.breach_readings, 0) AS breach_readings, d.weekly_sales_kaud FROM silver.store d LEFT JOIN ev ON ev.store_id = d.store_id LEFT JOIN rd ON rd.store_id = d.store_id WHERE d._is_current = true) SELECT s.*, ROUND(LEAST(100, COALESCE(s.store_tier, 1) * 8 + s.open_stock_issues * 6 + s.urgent_stock_issues_12m * 4 + LEAST(s.breach_readings, 20) * 1.5), 1) AS risk_score, CASE WHEN LEAST(100, COALESCE(s.store_tier, 1) * 8 + s.open_stock_issues * 6 + s.urgent_stock_issues_12m * 4 + LEAST(s.breach_readings, 20) * 1.5) >= 60 THEN 'High' WHEN LEAST(100, COALESCE(s.store_tier, 1) * 8 + s.open_stock_issues * 6 + s.urgent_stock_issues_12m * 4 + LEAST(s.breach_readings, 20) * 1.5) >= 35 THEN 'Medium' ELSE 'Low' END AS risk_band, ROUND(s.open_stock_issues * 1800 + LEAST(s.breach_readings, 20) * 900 + COALESCE(s.weekly_sales_kaud, 0) * COALESCE(s.store_tier, 1) * 1, 0) AS lost_sales_at_risk_aud FROM s","description":"Store risk score - stock issues, urgent work and out-of-stock lines breaches, plus lost sales at risk in AUD (weights and dollar rates are illustrative: $1,800 per open stock issue + $900 per 15-min window above 8% out-of-stock + weekly sales (k) x store tier x $1)","business_owner":"Store Operations","consumers":"Power BI / Direct Lake, data agent","active":true},{"transform_id":"RT-G06","industry":"retail","gold_table":"loyalty_member_summary","run_order":210,"sql_text":"SELECT state, tier, COUNT(*) AS loyalty_members, SUM(CASE WHEN status = 'Active' THEN 1 ELSE 0 END) AS active_loyalty_members, ROUND(AVG(year(current_date()) - birth_year), 1) AS avg_age, COUNT(DISTINCT store_id) AS linked_stores FROM silver.loyalty_member WHERE _is_deleted = false GROUP BY state, tier","description":"Loyalty members by state and tier","business_owner":"Loyalty & Marketing","consumers":"Power BI / Direct Lake, data agent","active":true}],"onboard":{"object":{"object_id":"RT_POS_PROMOTIONS","industry":"retail","source_system":"pos","source_object":"promotions","source_type":"file","ingest_method":"landed_file","landing_format":"txt","landing_path":"landing/pos/promotions","read_options":"{\"header\": true, \"delimiter\": \"|\"}","source_table":null,"load_type":"incremental","watermark_column":null,"primary_keys":"state,obs_date","dedupe_order_by":"_ingested_at DESC","bronze_table":"pos_promotions","silver_table":"promotion_day","silver_write_mode":"merge_scd1","load_group":"reference","priority":25,"sla_hours":26,"data_owner":"Merchandise","sensitivity":"Public","active":true,"description":"Daily promotions calendar by state (pipe-delimited, dd/MM/yyyy) - onboarded live in the demo","fabric_alternative":"Public or partner feed - could be a REST copy, or a shortcut if the provider shares a folder"},"columns":[["state","state","string",null],["obs_date","obs_date","date","to_date(obs_date, 'dd/MM/yyyy')"],["promo_lines","promo_lines","double",null],["avg_discount_pct","avg_discount_pct","double",null],["catalogue_reach_k","catalogue_reach_k","double",null],["promo_type","promo_type","string",null]],"rules":[["RT-EXT-01","promo_lines plausible","technical","range","promo_lines","{\"min\": 0, \"max\": 5000}","warn"],["RT-EXT-02","Known promo_type","business","allowed_values","promo_type","{\"values\": [\"Weekly specials\", \"Half price\", \"Seasonal\", \"Clearance\"]}","warn"]],"gold":{"transform_id":"RT-G07","industry":"retail","gold_table":"stock_issue_vs_promotion_day","run_order":220,"sql_text":"WITH ev AS (SELECT d.state, to_date(v.raised_ts) AS obs_date, COUNT(*) AS stock_issues, SUM(CASE WHEN v.issue_type = 'OUT_OF_STOCK' THEN 1 ELSE 0 END) AS urgent_stock_issues FROM silver.stock_issue v JOIN silver.store d ON d.store_id = v.store_id AND d._is_current = true WHERE v._is_deleted = false GROUP BY d.state, to_date(v.raised_ts)) SELECT x.state, x.obs_date, x.promo_lines, x.avg_discount_pct, x.catalogue_reach_k, x.promo_type, COALESCE(ev.stock_issues, 0) AS stock_issues, COALESCE(ev.urgent_stock_issues, 0) AS urgent_stock_issues FROM silver.promotion_day x LEFT JOIN ev ON ev.state = x.state AND ev.obs_date = x.obs_date WHERE x._is_deleted = false","description":"Promotions against out-of-stocks by state","business_owner":"Merchandise","consumers":"Power BI / Direct Lake","active":true}}},"digital":{"pack":{"id":"digital","industry":"Digital natives","org":"Lyrebird Apps","short":"LYB","prefix":"DN","tagline":"B2B SaaS scale-up selling a collaboration platform","email_domain":"lyrebird.example.com","region_col":"cloud_region","region_label":"cloud region","regions":["AU East","AU Southeast","SE Asia","US West","EU West"],"name_words":["Acme","Nimbus","Quokka","Harbour","Summit","Pixel","Ember","Orbit","Tandem","Wombat"],"sys":{"ops":{"code":"app","name":"Product database (Azure SQL / PostgreSQL)","type":"Database","owner":"Platform Engineering","fabric":"Mirroring for Azure SQL Database or Azure Database for PostgreSQL (no ETL) vs Copy job"},"tel":{"code":"tel","name":"Product telemetry exports (15-min CSV) and marketing campaign feed","type":"Event export","owner":"Data Platform","fabric":"Eventstream from Event Hubs / Kafka; OneLake shortcut to S3/ADLS for exports"},"crm":{"code":"hub","name":"HubSpot CRM (app users)","type":"SaaS CRM","owner":"Growth","fabric":"Copy job / pipeline with the REST connector"}},"entity":{"table":"tenant","src":"tenants","label":"tenant","plural":"tenants","id":"tenant_id","code":"tenant_slug","code_prefix":"TN","name_col":"tenant_name","type_col":"plan","types":["FREE","STARTER","PRO","BUSINESS","ENTERPRISE"],"status":["ACTIVE","TRIAL","PAST_DUE","CHURNED"],"tier_col":"account_tier","tier_label":"account tier (1-5, 5 = strategic)","cap_col":"seats","cap_label":"seats","cap_range":[1,5000],"start_col":"signup_date","start_years":[2018,2025],"drift_col":"success_manager","drift_values":["CSM team A","CSM team B","Self-serve"],"owner":"Customer Success","count":240},"event":{"table":"support_ticket","src":"support_tickets","label":"support ticket","plural":"support tickets","id":"ticket_id","id_start":100001,"type_col":"ticket_type","types":["BUG","OUTAGE","BILLING","HOW_TO"],"type_names":["Bug","Outage","Billing","How-to"],"urgent_type":"OUTAGE","status":["OPEN","PENDING","SOLVED","CLOSED_NO_ACTION"],"open_ts":"opened_ts","close_ts":"solved_ts","open_verb":"opened","close_verb":"solved","amount_col":"credit_aud","amount_range":[0,5000],"qty_col":"agent_hours","qty_range":[0.1,20],"descriptions":["Cannot export to PDF","Login loop on SSO","Invoice shows wrong seats","Webhook delivery failing","Slow search results","How to set up SCIM","Mobile app crashes on upload","API rate limit questions"],"owner":"Customer Support"},"reading":{"table":"usage_metric","src":"usage_metrics","label":"product usage metrics","fact":"usage","monitored":24,"monitored_types":["PRO","BUSINESS","ENTERPRISE"],"metrics":[{"col":"p95_latency_ms","label":"p95 API latency","unit":"ms","base":320,"sd":90,"min":0,"max":60000,"threshold":800},{"col":"active_users","label":"active users","unit":"users","base":140,"sd":60,"min":0,"max":100000},{"col":"error_rate_pct","label":"error rate","unit":"%","base":0.4,"sd":0.25,"min":0,"max":100}],"sentinel":-1,"owner":"Data Platform"},"party":{"table":"app_user","src":"app_users","label":"app user","plural":"app users","id":"user_id","id_prefix":"U","sf_type":"contact","segment_col":"user_role","segments":["Admin","Member","Guest","Billing"],"status":["Active","Deactivated"],"sensitive_col":"ip_address","sensitive_label":"last sign-in IP address","sensitive_treatment":"hash","owner":"Growth","count":300},"extra":{"table":"campaign_day","src":"campaigns","label":"marketing campaigns","sys":"tel","cols":[{"col":"spend_aud","base":4000,"sd":1500,"min":0,"max":1000000},{"col":"signups","base":60,"sd":25,"min":0,"max":100000},{"col":"click_through_pct","base":2.4,"sd":0.8,"min":0,"max":100}],"cat_col":"channel","cat_values":["Paid search","Social","Partner","Email"],"owner":"Growth","story":"campaign spend and sign-ups against support load by cloud region"},"alert":{"stream":"platform_alert","label":"platform alert","codes":["API_5XX_SPIKE","LOGIN_FAILURES","PAYMENT_FAILED"],"storm":"bad release"},"ref":{"title":"Support SLAs by ticket type","owner":"Customer Support","rows":[["BUG","Product",48,"Tier 2 Support"],["OUTAGE","Critical",2,"SRE On-call"],["BILLING","Commercial",24,"Billing Ops"],["HOW_TO","Self-help",72,"Tier 1 Support"]]},"ai":{"persona":"product, growth and customer success","cust":"SaaS scale-ups and fast-growing start-ups","q_model":["Which cloud region raised the most support tickets this month?","Show credits issued by ticket type and month.","Which strategic tenants (tier 4 or 5) have open outage tickets?"],"q_agent":["Which 10 tenants have the highest risk score and how many open tickets does each have?","Compare average hours to solve outage tickets by cloud region.","Which tenants saw p95 latency above 800 ms in the last 3 days?","How much credit have we issued to Enterprise tenants?","How many admins are active across tenants in AU East?"],"q_onto":["Show Enterprise tenants in AU East with an open outage ticket and a high risk band.","For the tenant with the worst latency, list its support tickets and the users affected.","Which app users belong to tenants that are past due?"]},"impact":{"col":"arr_at_risk_aud","label":"ARR at risk","open_cost":1500,"breach_cost":400,"cap_rate":12,"rule":"$1,500 per open ticket + $400 per slow 15-min window + seats x account tier x $12"}},"source_system":[{"source_system":"app","industry":"digital","display_name":"Product database (Azure SQL / PostgreSQL)","system_type":"Database","connection_pattern":"On-premises data gateway -> pipeline Copy with watermark query","technical_owner":"Platform Engineering","fabric_options":"Mirroring for Azure SQL Database or Azure Database for PostgreSQL (no ETL) vs Copy job"},{"source_system":"tel","industry":"digital","display_name":"Product telemetry exports (15-min CSV) and marketing campaign feed","system_type":"Event export","connection_pattern":"Files dropped to OneLake (Files/landing) every 15 minutes","technical_owner":"Data Platform","fabric_options":"Eventstream from Event Hubs / Kafka; OneLake shortcut to S3/ADLS for exports"},{"source_system":"hub","industry":"digital","display_name":"HubSpot CRM (app users)","system_type":"SaaS CRM","connection_pattern":"REST/OData API -> JSON landed per day","technical_owner":"Growth","fabric_options":"Copy job / pipeline with the REST connector"}],"ingestion_object":[{"object_id":"DN_APP_TENANTS","industry":"digital","source_system":"app","source_object":"tenants","source_type":"database","ingest_method":"copy_db","landing_format":"csv","landing_path":"landing/app/tenants","read_options":"{\"header\": true, \"delimiter\": \",\"}","source_table":"src.tenants","load_type":"incremental","watermark_column":"last_modified","primary_keys":"tenant_id","dedupe_order_by":"last_modified DESC, _ingested_at DESC","bronze_table":"app_tenants","silver_table":"tenant","silver_write_mode":"merge_scd2","load_group":"reference","priority":10,"sla_hours":26,"data_owner":"Customer Success","sensitivity":"Internal","active":true,"description":"Tenant register. SCD2 keeps status, account_tier and seats history.","fabric_alternative":"Strong mirroring candidate - small, slowly changing, needed in near real time by operations"},{"object_id":"DN_APP_SUPPORT_TICKETS","industry":"digital","source_system":"app","source_object":"support_tickets","source_type":"database","ingest_method":"copy_db","landing_format":"csv","landing_path":"landing/app/support_tickets","read_options":"{\"header\": true, \"delimiter\": \",\"}","source_table":"src.support_tickets","load_type":"incremental","watermark_column":"last_modified","primary_keys":"ticket_id","dedupe_order_by":"last_modified DESC, _ingested_at DESC","bronze_table":"app_support_tickets","silver_table":"support_ticket","silver_write_mode":"merge_scd1","load_group":"transactional","priority":20,"sla_hours":26,"data_owner":"Customer Support","sensitivity":"Internal","active":true,"description":"Support tickets - incremental by last_modified; can arrive before their tenant (late-arriving).","fabric_alternative":"Copy job incremental or mirroring; keep the watermark query if the source is on-prem"},{"object_id":"DN_TEL_USAGE_METRICS","industry":"digital","source_system":"tel","source_object":"usage_metrics","source_type":"file","ingest_method":"landed_file","landing_format":"csv","landing_path":"landing/tel/usage_metrics","read_options":"{\"header\": true, \"delimiter\": \",\"}","source_table":null,"load_type":"incremental","watermark_column":null,"primary_keys":"tenant_id,reading_ts","dedupe_order_by":"_ingested_at DESC","bronze_table":"tel_usage_metrics","silver_table":"usage_metric","silver_write_mode":"insert_only","load_group":"telemetry","priority":30,"sla_hours":2,"data_owner":"Data Platform","sensitivity":"Internal","active":true,"description":"Product usage metrics every 15 minutes, four micro-batch files a day. Insert-only and idempotent.","fabric_alternative":"Eventstream into Eventhouse for real time; keep this batch path for history and reconciliation"},{"object_id":"DN_HUB_APP_USERS","industry":"digital","source_system":"hub","source_object":"app_users","source_type":"api","ingest_method":"landed_file","landing_format":"json","landing_path":"landing/hub/app_users","read_options":"{\"multiline\": true, \"json_root\": \"records\"}","source_table":null,"load_type":"full","watermark_column":null,"primary_keys":"user_id","dedupe_order_by":"last_modified_ts DESC, _ingested_at DESC","bronze_table":"hub_app_users","silver_table":"app_user","silver_write_mode":"merge_scd1","load_group":"reference","priority":15,"sla_hours":26,"data_owner":"Growth","sensitivity":"Confidential","active":true,"description":"App users - full daily snapshot from the CRM API (contains PII). Missing rows are soft-deleted.","fabric_alternative":"Copy job with the CRM connector; Dataverse-based CRMs can use Link to Fabric instead of copying"}],"column_map":[{"object_id":"DN_APP_TENANTS","column_order":0,"source_column":"tenant_id","target_column":"tenant_id","data_type":"int","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"DN_APP_TENANTS","column_order":1,"source_column":"tenant_slug","target_column":"tenant_slug","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"DN_APP_TENANTS","column_order":2,"source_column":"tenant_name","target_column":"tenant_name","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"DN_APP_TENANTS","column_order":3,"source_column":"plan","target_column":"plan","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"DN_APP_TENANTS","column_order":4,"source_column":"cloud_region","target_column":"cloud_region","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"DN_APP_TENANTS","column_order":5,"source_column":"status","target_column":"status","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"DN_APP_TENANTS","column_order":6,"source_column":"account_tier","target_column":"account_tier","data_type":"int","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"DN_APP_TENANTS","column_order":7,"source_column":"seats","target_column":"seats","data_type":"double","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"DN_APP_TENANTS","column_order":8,"source_column":"signup_date","target_column":"signup_date","data_type":"date","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"DN_APP_TENANTS","column_order":9,"source_column":"last_modified","target_column":"last_modified","data_type":"timestamp","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"DN_APP_SUPPORT_TICKETS","column_order":0,"source_column":"ticket_id","target_column":"ticket_id","data_type":"int","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"DN_APP_SUPPORT_TICKETS","column_order":1,"source_column":"tenant_id","target_column":"tenant_id","data_type":"int","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"DN_APP_SUPPORT_TICKETS","column_order":2,"source_column":"ticket_type","target_column":"ticket_type","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"DN_APP_SUPPORT_TICKETS","column_order":3,"source_column":"priority","target_column":"priority","data_type":"int","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"DN_APP_SUPPORT_TICKETS","column_order":4,"source_column":"status","target_column":"status","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"DN_APP_SUPPORT_TICKETS","column_order":5,"source_column":"opened_ts","target_column":"opened_ts","data_type":"timestamp","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"DN_APP_SUPPORT_TICKETS","column_order":6,"source_column":"solved_ts","target_column":"solved_ts","data_type":"timestamp","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"DN_APP_SUPPORT_TICKETS","column_order":7,"source_column":"credit_aud","target_column":"credit_aud","data_type":"decimal(12,2)","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"DN_APP_SUPPORT_TICKETS","column_order":8,"source_column":"agent_hours","target_column":"agent_hours","data_type":"double","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"DN_APP_SUPPORT_TICKETS","column_order":9,"source_column":"description","target_column":"description","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"DN_APP_SUPPORT_TICKETS","column_order":10,"source_column":"last_modified","target_column":"last_modified","data_type":"timestamp","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"DN_TEL_USAGE_METRICS","column_order":0,"source_column":"tenant_id","target_column":"tenant_id","data_type":"int","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"DN_TEL_USAGE_METRICS","column_order":1,"source_column":"reading_time","target_column":"reading_ts","data_type":"timestamp","transform_expr":"to_timestamp(reading_time, 'dd/MM/yyyy HH:mm')","pii_treatment":"none","sensitivity":"none","description":""},{"object_id":"DN_TEL_USAGE_METRICS","column_order":2,"source_column":"p95_latency_ms","target_column":"p95_latency_ms","data_type":"double","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"DN_TEL_USAGE_METRICS","column_order":3,"source_column":"active_users","target_column":"active_users","data_type":"double","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"DN_TEL_USAGE_METRICS","column_order":4,"source_column":"error_rate_pct","target_column":"error_rate_pct","data_type":"double","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"DN_HUB_APP_USERS","column_order":0,"source_column":"Id","target_column":"user_id","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"DN_HUB_APP_USERS","column_order":1,"source_column":"FirstName","target_column":"first_name","data_type":"string","transform_expr":null,"pii_treatment":"drop","sensitivity":"Confidential","description":""},{"object_id":"DN_HUB_APP_USERS","column_order":2,"source_column":"LastName","target_column":"last_name","data_type":"string","transform_expr":null,"pii_treatment":"drop","sensitivity":"Confidential","description":""},{"object_id":"DN_HUB_APP_USERS","column_order":3,"source_column":"Email","target_column":"email_masked","data_type":"string","transform_expr":null,"pii_treatment":"mask_email","sensitivity":"Confidential","description":""},{"object_id":"DN_HUB_APP_USERS","column_order":4,"source_column":"MobilePhone","target_column":"mobile_masked","data_type":"string","transform_expr":null,"pii_treatment":"mask_phone","sensitivity":"Confidential","description":""},{"object_id":"DN_HUB_APP_USERS","column_order":5,"source_column":"Birthdate","target_column":"birth_year","data_type":"int","transform_expr":null,"pii_treatment":"year_only","sensitivity":"Confidential","description":""},{"object_id":"DN_HUB_APP_USERS","column_order":6,"source_column":"MailingPostalCode","target_column":"postcode","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"DN_HUB_APP_USERS","column_order":7,"source_column":"CloudRegion__c","target_column":"cloud_region","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"DN_HUB_APP_USERS","column_order":8,"source_column":"UserRole__c","target_column":"user_role","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"DN_HUB_APP_USERS","column_order":9,"source_column":"TenantId__c","target_column":"tenant_id","data_type":"int","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"DN_HUB_APP_USERS","column_order":10,"source_column":"Status__c","target_column":"status","data_type":"string","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""},{"object_id":"DN_HUB_APP_USERS","column_order":11,"source_column":"IpAddress__c","target_column":"ip_address_hash","data_type":"string","transform_expr":null,"pii_treatment":"hash","sensitivity":"Restricted","description":""},{"object_id":"DN_HUB_APP_USERS","column_order":12,"source_column":"LastModifiedDate","target_column":"last_modified_ts","data_type":"timestamp","transform_expr":null,"pii_treatment":"none","sensitivity":"Internal","description":""}],"dq_rule":[{"rule_id":"DN-ENT-01","object_id":"DN_APP_TENANTS","rule_name":"Tenant id present","rule_category":"technical","check_type":"not_null","column_name":"tenant_id","params":"{}","severity":"fail","fail_threshold_pct":0,"rule_owner":"Data engineering","description":"Primary key must never be null","active":true},{"rule_id":"DN-ENT-02","object_id":"DN_APP_TENANTS","rule_name":"Known plan","rule_category":"technical","check_type":"allowed_values","column_name":"plan","params":"{\"values\": [\"FREE\", \"STARTER\", \"PRO\", \"BUSINESS\", \"ENTERPRISE\"]}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Data engineering","description":"Unknown classes are kept but flagged for the data owner","active":true},{"rule_id":"DN-ENT-03","object_id":"DN_APP_TENANTS","rule_name":"account_tier between 1 and 5","rule_category":"technical","check_type":"range","column_name":"account_tier","params":"{\"min\": 1, \"max\": 5}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Data engineering","description":"Out-of-range ratings usually mean a keying error in the source","active":true},{"rule_id":"DN-ENT-04","object_id":"DN_APP_TENANTS","rule_name":"Known status","rule_category":"business","check_type":"allowed_values","column_name":"status","params":"{\"values\": [\"ACTIVE\", \"TRIAL\", \"PAST_DUE\", \"CHURNED\"]}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Customer Success","description":"Status list agreed with Customer Success","active":true},{"rule_id":"DN-EVT-01","object_id":"DN_APP_SUPPORT_TICKETS","rule_name":"Support ticket id present","rule_category":"technical","check_type":"not_null","column_name":"ticket_id","params":"{}","severity":"fail","fail_threshold_pct":0,"rule_owner":"Data engineering","description":"Primary key must never be null","active":true},{"rule_id":"DN-EVT-02","object_id":"DN_APP_SUPPORT_TICKETS","rule_name":"Tenant exists","rule_category":"technical","check_type":"referential","column_name":"tenant_id","params":"{\"ref_table\": \"silver.tenant\", \"ref_column\": \"tenant_id\"}","severity":"quarantine","fail_threshold_pct":null,"rule_owner":"Data engineering","description":"Held until the tenant arrives (late-arriving reference data), then merged automatically","active":true},{"rule_id":"DN-EVT-03","object_id":"DN_APP_SUPPORT_TICKETS","rule_name":"Solved after opened","rule_category":"business","check_type":"expression","column_name":null,"params":"{\"expr\": \"solved_ts IS NULL OR solved_ts >= opened_ts\"}","severity":"quarantine","fail_threshold_pct":null,"rule_owner":"Customer Support","description":"A support ticket can't be solved before it was opened","active":true},{"rule_id":"DN-EVT-04","object_id":"DN_APP_SUPPORT_TICKETS","rule_name":"credit_aud not negative","rule_category":"technical","check_type":"range","column_name":"credit_aud","params":"{\"min\": 0}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Data engineering","description":"Negative values are sign errors in the source extract","active":true},{"rule_id":"DN-EVT-05","object_id":"DN_APP_SUPPORT_TICKETS","rule_name":"OUTAGE is priority 1","rule_category":"business","check_type":"expression","column_name":null,"params":"{\"expr\": \"ticket_type <> 'OUTAGE' OR priority = 1\"}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Customer Support","description":"Outage support tickets must be raised as priority 1","active":true},{"rule_id":"DN-EVT-06","object_id":"DN_APP_SUPPORT_TICKETS","rule_name":"opened_ts parses","rule_category":"technical","check_type":"not_null","column_name":"opened_ts","params":"{}","severity":"fail","fail_threshold_pct":5,"rule_owner":"Data engineering","description":"If more than 5% of timestamps fail to parse, the extract format has changed - stop the load","active":true},{"rule_id":"DN-RDG-01","object_id":"DN_TEL_USAGE_METRICS","rule_name":"P95 api latency plausible","rule_category":"technical","check_type":"range","column_name":"p95_latency_ms","params":"{\"min\": 0, \"max\": 60000}","severity":"quarantine","fail_threshold_pct":null,"rule_owner":"Data engineering","description":"Sensor fault values (e.g. -1) are quarantined, never averaged into Gold","active":true},{"rule_id":"DN-RDG-02","object_id":"DN_TEL_USAGE_METRICS","rule_name":"Tenant known","rule_category":"technical","check_type":"referential","column_name":"tenant_id","params":"{\"ref_table\": \"silver.tenant\", \"ref_column\": \"tenant_id\"}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Data engineering","description":"Readings for unknown tags are flagged","active":true},{"rule_id":"DN-RDG-03","object_id":"DN_TEL_USAGE_METRICS","rule_name":"Reading time parses","rule_category":"technical","check_type":"not_null","column_name":"reading_ts","params":"{}","severity":"fail","fail_threshold_pct":5,"rule_owner":"Data engineering","description":"Timestamp format change stops the load","active":true},{"rule_id":"DN-PTY-01","object_id":"DN_HUB_APP_USERS","rule_name":"App user id present","rule_category":"technical","check_type":"not_null","column_name":"user_id","params":"{}","severity":"fail","fail_threshold_pct":0,"rule_owner":"Data engineering","description":"Primary key must never be null","active":true},{"rule_id":"DN-PTY-02","object_id":"DN_HUB_APP_USERS","rule_name":"Email looks valid","rule_category":"technical","check_type":"regex","column_name":"email_masked","params":"{\"pattern\": \"^[^@\\\\s]+@[^@\\\\s]+\\\\.[^@\\\\s]+$\"}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Data engineering","description":"Checked on the raw value before masking","active":true},{"rule_id":"DN-PTY-03","object_id":"DN_HUB_APP_USERS","rule_name":"Australian postcode","rule_category":"technical","check_type":"regex","column_name":"postcode","params":"{\"pattern\": \"^\\\\d{4}$\"}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Data engineering","description":"Four digits","active":true},{"rule_id":"DN-PTY-04","object_id":"DN_HUB_APP_USERS","rule_name":"Known user_role","rule_category":"business","check_type":"allowed_values","column_name":"user_role","params":"{\"values\": [\"Admin\", \"Member\", \"Guest\", \"Billing\"]}","severity":"warn","fail_threshold_pct":null,"rule_owner":"Growth","description":"Segments agreed with Growth","active":true}],"gold_transform":[{"transform_id":"DN-G01","industry":"digital","gold_table":"dim_date","run_order":100,"sql_text":"SELECT CAST(date_format(d, 'yyyyMMdd') AS INT) AS date_key, d AS calendar_date, year(d) AS calendar_year, quarter(d) AS calendar_quarter, month(d) AS month_number, date_format(d, 'MMM') AS month_name, date_format(d, 'yyyy-MM') AS year_month, dayofmonth(d) AS day_of_month, date_format(d, 'EEE') AS day_name, CASE WHEN month(d) >= 7 THEN year(d) + 1 ELSE year(d) END AS fy_year, concat('FY', substring(CAST(CASE WHEN month(d) >= 7 THEN year(d) + 1 ELSE year(d) END AS STRING), 3, 2)) AS fy_label, CASE WHEN dayofweek(d) IN (1, 7) THEN true ELSE false END AS is_weekend FROM (SELECT explode(sequence(to_date('2020-01-01'), to_date('2028-12-31'), interval 1 day)) AS d) dates","description":"Calendar with Australian financial year","business_owner":"Data engineering","consumers":"Power BI / Direct Lake, data agent","active":true},{"transform_id":"DN-G02","industry":"digital","gold_table":"dim_tenant","run_order":110,"sql_text":"SELECT tenant_id, tenant_slug, tenant_name, plan, cloud_region, status, account_tier, seats, signup_date, CAST(floor(months_between(current_date(), signup_date) / 12) AS INT) AS age_years, _valid_from AS version_valid_from FROM silver.tenant WHERE _is_current = true","description":"Current view of the tenant register (history stays in Silver SCD2)","business_owner":"Customer Success","consumers":"Power BI / Direct Lake, data agent","active":true},{"transform_id":"DN-G03","industry":"digital","gold_table":"fact_support_ticket","run_order":120,"sql_text":"SELECT v.ticket_id, v.tenant_id, v.ticket_type, v.priority, v.status, v.opened_ts, v.solved_ts, to_date(v.opened_ts) AS opened_date, CAST(date_format(v.opened_ts, 'yyyyMMdd') AS INT) AS opened_date_key, ROUND((unix_timestamp(v.solved_ts) - unix_timestamp(v.opened_ts)) / 3600.0, 2) AS hours_to_close, CASE WHEN v.status IN ('OPEN', 'PENDING') THEN 1 ELSE 0 END AS is_open, v.credit_aud, v.agent_hours, d.cloud_region, d.plan, d.account_tier FROM silver.support_ticket v LEFT JOIN silver.tenant d ON d.tenant_id = v.tenant_id AND d._is_current = true WHERE v._is_deleted = false","description":"Support tickets with duration and tenant attributes","business_owner":"Customer Support","consumers":"Power BI / Direct Lake, data agent","active":true},{"transform_id":"DN-G04","industry":"digital","gold_table":"fact_usage_hourly","run_order":130,"sql_text":"SELECT tenant_id, date_trunc('HOUR', reading_ts) AS reading_hour, to_date(reading_ts) AS reading_date, CAST(date_format(reading_ts, 'yyyyMMdd') AS INT) AS reading_date_key, COUNT(*) AS readings, ROUND(AVG(p95_latency_ms), 2) AS avg_p95_latency_ms, ROUND(MAX(p95_latency_ms), 2) AS max_p95_latency_ms, ROUND(AVG(active_users), 2) AS avg_active_users, ROUND(AVG(error_rate_pct), 2) AS avg_error_rate_pct, SUM(CASE WHEN p95_latency_ms > 800 THEN 1 ELSE 0 END) AS breach_readings FROM silver.usage_metric GROUP BY tenant_id, date_trunc('HOUR', reading_ts), to_date(reading_ts), CAST(date_format(reading_ts, 'yyyyMMdd') AS INT)","description":"Hourly product usage metrics with threshold breaches","business_owner":"Data Platform","consumers":"Power BI / Direct Lake, data agent","active":true},{"transform_id":"DN-G05","industry":"digital","gold_table":"tenant_health_scorecard","run_order":200,"sql_text":"WITH asat AS (SELECT MAX(to_date(opened_ts)) AS as_at FROM silver.support_ticket), ev AS (SELECT v.tenant_id, COUNT(*) AS support_tickets_12m, SUM(CASE WHEN v.status IN ('OPEN', 'PENDING') THEN 1 ELSE 0 END) AS open_support_tickets, SUM(CASE WHEN v.ticket_type = 'OUTAGE' THEN 1 ELSE 0 END) AS urgent_support_tickets_12m, ROUND(SUM(v.credit_aud), 2) AS credit_aud_12m FROM silver.support_ticket v CROSS JOIN asat WHERE v._is_deleted = false AND to_date(v.opened_ts) > add_months(asat.as_at, -12) GROUP BY v.tenant_id), rd AS (SELECT tenant_id, MAX(p95_latency_ms) AS max_p95_latency_ms, SUM(CASE WHEN p95_latency_ms > 800 THEN 1 ELSE 0 END) AS breach_readings FROM silver.usage_metric GROUP BY tenant_id), s AS (SELECT d.tenant_id, d.tenant_slug, d.tenant_name, d.plan, d.cloud_region, d.status, d.account_tier, COALESCE(ev.support_tickets_12m, 0) AS support_tickets_12m, COALESCE(ev.open_support_tickets, 0) AS open_support_tickets, COALESCE(ev.urgent_support_tickets_12m, 0) AS urgent_support_tickets_12m, COALESCE(ev.credit_aud_12m, 0) AS credit_aud_12m, rd.max_p95_latency_ms, COALESCE(rd.breach_readings, 0) AS breach_readings, d.seats FROM silver.tenant d LEFT JOIN ev ON ev.tenant_id = d.tenant_id LEFT JOIN rd ON rd.tenant_id = d.tenant_id WHERE d._is_current = true) SELECT s.*, ROUND(LEAST(100, COALESCE(s.account_tier, 1) * 8 + s.open_support_tickets * 6 + s.urgent_support_tickets_12m * 4 + LEAST(s.breach_readings, 20) * 1.5), 1) AS risk_score, CASE WHEN LEAST(100, COALESCE(s.account_tier, 1) * 8 + s.open_support_tickets * 6 + s.urgent_support_tickets_12m * 4 + LEAST(s.breach_readings, 20) * 1.5) >= 60 THEN 'High' WHEN LEAST(100, COALESCE(s.account_tier, 1) * 8 + s.open_support_tickets * 6 + s.urgent_support_tickets_12m * 4 + LEAST(s.breach_readings, 20) * 1.5) >= 35 THEN 'Medium' ELSE 'Low' END AS risk_band, ROUND(s.open_support_tickets * 1500 + LEAST(s.breach_readings, 20) * 400 + COALESCE(s.seats, 0) * COALESCE(s.account_tier, 1) * 12, 0) AS arr_at_risk_aud FROM s","description":"Tenant risk score - support tickets, urgent work and p95 API latency breaches, plus ARR at risk in AUD (weights and dollar rates are illustrative: $1,500 per open ticket + $400 per slow 15-min window + seats x account tier x $12)","business_owner":"Customer Success","consumers":"Power BI / Direct Lake, data agent","active":true},{"transform_id":"DN-G06","industry":"digital","gold_table":"app_user_summary","run_order":210,"sql_text":"SELECT cloud_region, user_role, COUNT(*) AS app_users, SUM(CASE WHEN status = 'Active' THEN 1 ELSE 0 END) AS active_app_users, ROUND(AVG(year(current_date()) - birth_year), 1) AS avg_age, COUNT(DISTINCT tenant_id) AS linked_tenants FROM silver.app_user WHERE _is_deleted = false GROUP BY cloud_region, user_role","description":"App users by cloud region and user_role","business_owner":"Growth","consumers":"Power BI / Direct Lake, data agent","active":true}],"onboard":{"object":{"object_id":"DN_TEL_CAMPAIGNS","industry":"digital","source_system":"tel","source_object":"campaigns","source_type":"file","ingest_method":"landed_file","landing_format":"txt","landing_path":"landing/tel/campaigns","read_options":"{\"header\": true, \"delimiter\": \"|\"}","source_table":null,"load_type":"incremental","watermark_column":null,"primary_keys":"cloud_region,obs_date","dedupe_order_by":"_ingested_at DESC","bronze_table":"tel_campaigns","silver_table":"campaign_day","silver_write_mode":"merge_scd1","load_group":"reference","priority":25,"sla_hours":26,"data_owner":"Growth","sensitivity":"Public","active":true,"description":"Daily marketing campaigns by cloud region (pipe-delimited, dd/MM/yyyy) - onboarded live in the demo","fabric_alternative":"Public or partner feed - could be a REST copy, or a shortcut if the provider shares a folder"},"columns":[["cloud_region","cloud_region","string",null],["obs_date","obs_date","date","to_date(obs_date, 'dd/MM/yyyy')"],["spend_aud","spend_aud","double",null],["signups","signups","double",null],["click_through_pct","click_through_pct","double",null],["channel","channel","string",null]],"rules":[["DN-EXT-01","spend_aud plausible","technical","range","spend_aud","{\"min\": 0, \"max\": 1000000}","warn"],["DN-EXT-02","Known channel","business","allowed_values","channel","{\"values\": [\"Paid search\", \"Social\", \"Partner\", \"Email\"]}","warn"]],"gold":{"transform_id":"DN-G07","industry":"digital","gold_table":"support_ticket_vs_campaign_day","run_order":220,"sql_text":"WITH ev AS (SELECT d.cloud_region, to_date(v.opened_ts) AS obs_date, COUNT(*) AS support_tickets, SUM(CASE WHEN v.ticket_type = 'OUTAGE' THEN 1 ELSE 0 END) AS urgent_support_tickets FROM silver.support_ticket v JOIN silver.tenant d ON d.tenant_id = v.tenant_id AND d._is_current = true WHERE v._is_deleted = false GROUP BY d.cloud_region, to_date(v.opened_ts)) SELECT x.cloud_region, x.obs_date, x.spend_aud, x.signups, x.click_through_pct, x.channel, COALESCE(ev.support_tickets, 0) AS support_tickets, COALESCE(ev.urgent_support_tickets, 0) AS urgent_support_tickets FROM silver.campaign_day x LEFT JOIN ev ON ev.cloud_region = x.cloud_region AND ev.obs_date = x.obs_date WHERE x._is_deleted = false","description":"Campaign spend and sign-ups against support load by cloud region","business_owner":"Growth","consumers":"Power BI / Direct Lake","active":true}}}}''')

In [ ]:
def load_pack(pack):
    path = f"/lakehouse/default/Files/config/{pack}.json"
    if os.path.exists(path):
        print(f"loading {pack} from Files/config/{pack}.json")
        return json.load(open(path))
    assert pack in EMBEDDED_CONFIG, f"unknown pack '{pack}' - choose one of {sorted(EMBEDDED_CONFIG)}"
    print(f"loading {pack} from the embedded copy")
    return EMBEDDED_CONFIG[pack]


In [ ]:
S = lambda *names: StructType([StructField(n, t, True) for n, t in names])
str_ = StringType(); int_ = IntegerType(); bool_ = BooleanType(); dbl = DoubleType()

SCHEMA = {
    "source_system": S(("source_system", str_), ("industry", str_), ("display_name", str_), ("system_type", str_),
                       ("connection_pattern", str_), ("technical_owner", str_), ("fabric_options", str_)),
    "ingestion_object": S(("object_id", str_), ("industry", str_), ("source_system", str_), ("source_object", str_),
                          ("source_type", str_), ("ingest_method", str_), ("landing_format", str_), ("landing_path", str_),
                          ("read_options", str_), ("source_table", str_), ("load_type", str_), ("watermark_column", str_),
                          ("primary_keys", str_), ("dedupe_order_by", str_), ("bronze_table", str_), ("silver_table", str_),
                          ("silver_write_mode", str_), ("load_group", str_), ("priority", int_), ("sla_hours", int_),
                          ("data_owner", str_), ("sensitivity", str_), ("active", bool_), ("description", str_),
                          ("fabric_alternative", str_)),
    "column_map": S(("object_id", str_), ("column_order", int_), ("source_column", str_), ("target_column", str_),
                    ("data_type", str_), ("transform_expr", str_), ("pii_treatment", str_), ("sensitivity", str_),
                    ("description", str_)),
    "dq_rule": S(("rule_id", str_), ("object_id", str_), ("rule_name", str_), ("rule_category", str_),
                 ("check_type", str_), ("column_name", str_), ("params", str_), ("severity", str_),
                 ("fail_threshold_pct", dbl), ("rule_owner", str_), ("description", str_), ("active", bool_)),
    "gold_transform": S(("transform_id", str_), ("industry", str_), ("gold_table", str_), ("run_order", int_),
                        ("sql_text", str_), ("description", str_), ("business_owner", str_), ("consumers", str_),
                        ("active", bool_)),
}


def to_rows(items, schema, pack):
    rows = []
    for i, it in enumerate(items):
        it = dict(it)
        it.setdefault("industry", pack)
        it.setdefault("column_order", i)
        row = []
        for f in schema.fields:
            v = it.get(f.name)
            if v is not None and isinstance(f.dataType, IntegerType):
                v = int(v)
            if v is not None and isinstance(f.dataType, DoubleType):
                v = float(v)
            row.append(v)
        rows.append(tuple(row))
    return rows


cfg = load_pack(industry)
for table, schema in SCHEMA.items():
    rows = to_rows(cfg[table], schema, industry)
    spark.createDataFrame(rows, schema).write.mode("overwrite").option("overwriteSchema", "true") \
        .saveAsTable(f"meta.{table}")
    print(f"meta.{table}: {len(rows)} rows")

settings = [("industry", industry), ("pii_salt", pii_salt), ("pack_spec", json.dumps(cfg["pack"])),
            ("onboard_spec", json.dumps(cfg["onboard"]))]
spark.createDataFrame(settings, "key string, value string").write.mode("overwrite").saveAsTable("meta.setting")
print(f"Loaded pack '{industry}' - {cfg['pack']['org']} ({cfg['pack']['industry']})")


In [ ]:
# Operational tables - created empty once, never overwritten by a config reload
OPS = {
    "watermark": "object_id string, layer string, watermark_value string, run_id string, updated_at timestamp",
    "run_log": ("run_id string, parent_run_id string, object_id string, layer string, status string, "
                "started_at timestamp, ended_at timestamp, duration_sec double, rows_read bigint, rows_written bigint, "
                "rows_quarantined bigint, rows_deduplicated bigint, message string"),
    "dq_result": ("run_id string, object_id string, rule_id string, rule_name string, rule_category string, "
                  "check_type string, severity string, rows_evaluated bigint, rows_failed bigint, failed_pct double, "
                  "outcome string, evaluated_at timestamp"),
}
for t, ddl in OPS.items():
    spark.sql(f"CREATE TABLE IF NOT EXISTS meta.{t} ({ddl}) USING DELTA")
print("operational tables ready")

In [ ]:
display(spark.sql("""SELECT object_id, source_system, ingest_method, landing_format, load_type, silver_write_mode,
                     load_group, priority, data_owner, sensitivity FROM meta.ingestion_object ORDER BY priority, object_id"""))

In [ ]:
display(spark.sql("""SELECT object_id, target_column, pii_treatment, sensitivity FROM meta.column_map
                     WHERE pii_treatment <> 'none' ORDER BY object_id"""))